In [1]:
import pandas as pd
import json
import zipfile
from pathlib import Path

print("DocuMind metadata extraction started")

DocuMind metadata extraction started


In [2]:
docile_zip = r"C:\Projects\DocuMind\data\docile\annotated-trainval.zip"

invoice_metadata = []

with zipfile.ZipFile(docile_zip, "r") as z:

    train_meta = json.loads(z.read("train.json"))
    val_meta = json.loads(z.read("val.json"))

    all_ids = train_meta + val_meta

    for doc_id in all_ids:

        annotation = json.loads(
            z.read(f"annotations/{doc_id}.json")
        )

        document_type = annotation["metadata"]["document_type"]

        if document_type == "tax_invoice":
            metadata = annotation["metadata"]

            invoice_metadata.append({
                "document_id": doc_id,
                "document_type": document_type,
                "vendor_name": metadata.get("vendor_name"),
                "vendor_address": metadata.get("vendor_address"),
                "customer_billing_name": metadata.get("customer_billing_name"),
                "customer_billing_address": metadata.get("customer_billing_address"),
                "invoice_date": metadata.get("date_issue"),
                "amount_total_gross": metadata.get("amount_total_gross"),
                "amount_due": metadata.get("amount_due"),
                "payment_terms": metadata.get("payment_terms")
            })

invoice_metadata_df = pd.DataFrame(invoice_metadata)

print(invoice_metadata_df.shape)
print(invoice_metadata_df.head())

(3850, 10)
                document_id document_type vendor_name vendor_address  \
0  00134dd365a24343b35b78c6   tax_invoice        None           None   
1  00136a27c7774c1e8dc6b2f2   tax_invoice        None           None   
2  002e3cf97973428f905671b3   tax_invoice        None           None   
3  002f9b82b74f4258b3b072d0   tax_invoice        None           None   
4  003568b1286f4dab953fc2d5   tax_invoice        None           None   

  customer_billing_name customer_billing_address invoice_date  \
0                  None                     None         None   
1                  None                     None         None   
2                  None                     None         None   
3                  None                     None         None   
4                  None                     None         None   

  amount_total_gross amount_due payment_terms  
0               None       None          None  
1               None       None          None  
2               None 

In [3]:
with zipfile.ZipFile(docile_zip, "r") as z:
    sample_id = all_ids[0]

    sample_annotation = json.loads(
        z.read(f"annotations/{sample_id}.json")
    )

print("Top-level keys:")
print(sample_annotation.keys())

print("\nMetadata:")
print(sample_annotation.get("metadata"))

print("\nField extractions:")
print(sample_annotation.get("field_extractions"))

Top-level keys:
dict_keys(['field_extractions', 'line_item_extractions', 'line_item_headers', 'metadata'])

Metadata:
{'cluster_id': 222, 'currency': 'usd', 'document_type': 'tax_invoice', 'language': 'eng', 'original_filename': 'rtyj0081', 'page_count': 1, 'page_sizes_at_200dpi': [[1692, 2245]], 'page_to_table_grid': {'0': {'bbox': [0.06532258064516129, 0.46200607902735563, 0.9008064516129032, 0.7854103343465045], 'columns': [{'column_type': 'line_item_quantity', 'left': 0.09112903225806451, 'right': 0.16129032258064516}, {'column_type': 'line_item_units_of_measure', 'left': 0.19032258064516128, 'right': 0.21693548387096775}, {'column_type': 'line_item_code', 'left': 0.24596774193548387, 'right': 0.31693548387096776}, {'column_type': 'line_item_unit_price_gross', 'left': 0.6362903225806451, 'right': 0.7387096774193549}, {'column_type': 'line_item_amount_gross', 'left': 0.7911290322580645, 'right': 0.9008064516129032}], 'missing_columns': False, 'missing_second_table_on_page': False, '

In [4]:
invoice_records = []

fields = [
    "document_id",
    "date_issue",
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "amount_total_gross",
    "amount_due",
    "payment_terms"
]

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_id in all_ids:

        annotation = json.loads(
            z.read(f"annotations/{doc_id}.json")
        )

        if annotation["metadata"]["document_type"] != "tax_invoice":
            continue

        record = {
            "document_id": doc_id
        }

        # Extract fields from field_extractions
        for field in fields[1:]:
            values = []

            for item in annotation["field_extractions"]:
                if item["fieldtype"] == field:
                    values.append(item["text"])

            record[field] = values[0] if values else None

        invoice_records.append(record)

invoice_gt_df = pd.DataFrame(invoice_records)

print(invoice_gt_df.shape)
print(invoice_gt_df.head())

(3850, 9)
                document_id  date_issue  \
0  00134dd365a24343b35b78c6    03/25/99   
1  00136a27c7774c1e8dc6b2f2  10/31/2020   
2  002e3cf97973428f905671b3    12/09/99   
3  002f9b82b74f4258b3b072d0    09/27/20   
4  003568b1286f4dab953fc2d5    05/08/01   

                                vendor_name  \
0  BIO RELIANCE Testing & Development, Inc.   
1                       KMOZ 92.3 The Moose   
2                    UMI PUBLICATIONS, INC.   
3                                      WZZM   
4             Walt Klein & Associates, Inc.   

                                      vendor_address  \
0  BIORELIANCE- Testing & Development, Inc.\n1492...   
1  KMOZ 92.3 The Moose\n1360 E. Sherwood Drive\nG...   
2  UMI PUBLICATIONS, INC.\nP. O. BOX 30036 (28230...   
3    WZZM\n645 3 Mile Road NW\nGrand Rapids MI 49544   
4  Walt Klein & Associates, Inc.\n725 Highland Oa...   

                               customer_billing_name  \
0                          LORILLARD RESEARCH CENTER   

In [5]:
print("Missing values by field:")
print(invoice_gt_df.isnull().sum())

Missing values by field:
document_id                    0
date_issue                   197
vendor_name                  478
vendor_address               363
customer_billing_name        114
customer_billing_address     420
amount_total_gross           201
amount_due                   176
payment_terms               1949
dtype: int64


In [6]:
field_summary = pd.DataFrame({
    "field": invoice_gt_df.columns[1:],
    "missing": [
        invoice_gt_df[field].isna().sum()
        for field in invoice_gt_df.columns[1:]
    ]
})

field_summary["available"] = (
    len(invoice_gt_df) - field_summary["missing"]
)

field_summary["coverage_percent"] = (
    field_summary["available"] / len(invoice_gt_df) * 100
)

print(field_summary.sort_values(
    "coverage_percent",
    ascending=False
))

                      field  missing  available  coverage_percent
3     customer_billing_name      114       3736         97.038961
6                amount_due      176       3674         95.428571
0                date_issue      197       3653         94.883117
5        amount_total_gross      201       3649         94.779221
2            vendor_address      363       3487         90.571429
4  customer_billing_address      420       3430         89.090909
1               vendor_name      478       3372         87.584416
7             payment_terms     1949       1901         49.376623


In [7]:
extraction_records = []

target_fields = [
    "date_issue",
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "amount_total_gross",
    "amount_due"
]

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_id in all_ids:

        annotation = json.loads(
            z.read(f"annotations/{doc_id}.json")
        )

        if annotation["metadata"]["document_type"] != "tax_invoice":
            continue

        for item in annotation["field_extractions"]:

            field = item["fieldtype"]

            if field in target_fields:

                extraction_records.append({
                    "document_id": doc_id,
                    "field": field,
                    "value": item["text"]
                })

invoice_extraction_df = pd.DataFrame(extraction_records)

print(invoice_extraction_df.shape)
print(invoice_extraction_df.head())

(30823, 3)
                document_id           field  \
0  00134dd365a24343b35b78c6      date_issue   
1  00134dd365a24343b35b78c6     vendor_name   
2  00134dd365a24343b35b78c6  vendor_address   
3  00134dd365a24343b35b78c6     vendor_name   
4  00134dd365a24343b35b78c6     vendor_name   

                                               value  
0                                           03/25/99  
1           BIO RELIANCE Testing & Development, Inc.  
2  BIORELIANCE- Testing & Development, Inc.\n1492...  
3            BioReliance Testing & Development, Inc.  
4            BioReliance Testing & Development, Inc.  


In [8]:
print(
    invoice_extraction_df["field"]
    .value_counts()
)

field
vendor_address              5285
vendor_name                 5140
date_issue                  4431
amount_due                  4270
customer_billing_name       4081
amount_total_gross          3910
customer_billing_address    3706
Name: count, dtype: int64


In [9]:
field_occurrence_stats = (
    invoice_extraction_df
    .groupby(["document_id", "field"])
    .size()
    .reset_index(name="occurrences")
)

print(
    field_occurrence_stats
    .groupby("field")["occurrences"]
    .agg(
        documents="count",
        total_occurrences="sum",
        max_occurrences="max"
    )
)

                          documents  total_occurrences  max_occurrences
field                                                                  
amount_due                     3674               4270                4
amount_total_gross             3649               3910                4
customer_billing_address       3430               3706                3
customer_billing_name          3736               4081                3
date_issue                     3653               4431                4
vendor_address                 3487               5285                9
vendor_name                    3372               5140                9


In [10]:
extraction_records = []

target_fields = [
    "date_issue",
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "amount_total_gross",
    "amount_due"
]

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_id in all_ids:

        annotation = json.loads(
            z.read(f"annotations/{doc_id}.json")
        )

        if annotation["metadata"]["document_type"] != "tax_invoice":
            continue

        for item in annotation["field_extractions"]:

            if item["fieldtype"] in target_fields:

                extraction_records.append({
                    "document_id": doc_id,
                    "field": item["fieldtype"],
                    "value": item["text"],
                    "page": item["page"],
                    "bbox": item["bbox"]
                })

invoice_extraction_df = pd.DataFrame(extraction_records)

print(invoice_extraction_df.shape)
print(invoice_extraction_df.head())

(30823, 5)
                document_id           field  \
0  00134dd365a24343b35b78c6      date_issue   
1  00134dd365a24343b35b78c6     vendor_name   
2  00134dd365a24343b35b78c6  vendor_address   
3  00134dd365a24343b35b78c6     vendor_name   
4  00134dd365a24343b35b78c6     vendor_name   

                                               value  page  \
0                                           03/25/99     0   
1           BIO RELIANCE Testing & Development, Inc.     0   
2  BIORELIANCE- Testing & Development, Inc.\n1492...     0   
3            BioReliance Testing & Development, Inc.     0   
4            BioReliance Testing & Development, Inc.     0   

                                                bbox  
0  [0.7830815892017836, 0.36320179960633614, 0.87...  
1  [0.0726911179849757, 0.1801480926047427, 0.463...  
2  [0.07385267708679748, 0.17891134682472695, 0.4...  
3  [0.6949711431593889, 0.17973300483376403, 0.91...  
4  [0.6933190722960939, 0.26316564680047666, 0.91...  


In [11]:
unique_value_stats = (
    invoice_extraction_df
    .assign(
        normalized_value=lambda x:
        x["value"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
    )
    .groupby(["document_id", "field"])["normalized_value"]
    .nunique()
    .reset_index(name="unique_values")
)

print(
    unique_value_stats
    .groupby("field")["unique_values"]
    .agg(
        average_unique_values="mean",
        max_unique_values="max"
    )
)

                          average_unique_values  max_unique_values
field                                                             
amount_due                             1.019597                  2
amount_total_gross                     1.012058                  3
customer_billing_address               1.013120                  2
customer_billing_name                  1.005086                  2
date_issue                             1.008486                  3
vendor_address                         1.349011                  6
vendor_name                            1.173784                  7


In [12]:
vendor_multi = (
    unique_value_stats[
        (unique_value_stats["field"] == "vendor_name") &
        (unique_value_stats["unique_values"] > 1)
    ]
    .head(5)
)

for doc_id in vendor_multi["document_id"]:
    print("\n" + "=" * 80)
    print("Document:", doc_id)

    print(
        invoice_extraction_df[
            (invoice_extraction_df["document_id"] == doc_id) &
            (invoice_extraction_df["field"] == "vendor_name")
        ][["value", "page", "bbox"]]
        .to_string(index=False)
    )


Document: 00134dd365a24343b35b78c6
                                   value  page                                                                               bbox
BIO RELIANCE Testing & Development, Inc.     0  [0.0726911179849757, 0.1801480926047427, 0.46368122229810255, 0.2038080955505269]
 BioReliance Testing & Development, Inc.     0 [0.6949711431593889, 0.17973300483376403, 0.9191020902797307, 0.19052528687920947]
 BioReliance Testing & Development, Inc.     0  [0.6933190722960939, 0.26316564680047666, 0.9174500194164358, 0.2743730166169007]

Document: 002f9b82b74f4258b3b072d0
     value  page                                                                                 bbox
      WZZM     0 [0.2022152892237658, 0.030411775439450155, 0.24076995526253828, 0.04196825010644121]
WZZM\nWZZM     0  [0.4610823326269524, 0.20436713095310505, 0.49974025003808803, 0.23016014870320664]

Document: 0079863fa739477290959e5d
                       value  page                                

In [13]:
canonical_df = (
    invoice_extraction_df
    .sort_values(["document_id", "field", "page"])
    .groupby(["document_id", "field"], as_index=False)
    .first()
)

print(canonical_df.shape)
print(canonical_df.head(15))

(25001, 5)
                 document_id                     field  \
0   00134dd365a24343b35b78c6                amount_due   
1   00134dd365a24343b35b78c6        amount_total_gross   
2   00134dd365a24343b35b78c6  customer_billing_address   
3   00134dd365a24343b35b78c6     customer_billing_name   
4   00134dd365a24343b35b78c6                date_issue   
5   00134dd365a24343b35b78c6            vendor_address   
6   00134dd365a24343b35b78c6               vendor_name   
7   00136a27c7774c1e8dc6b2f2                amount_due   
8   00136a27c7774c1e8dc6b2f2        amount_total_gross   
9   00136a27c7774c1e8dc6b2f2  customer_billing_address   
10  00136a27c7774c1e8dc6b2f2     customer_billing_name   
11  00136a27c7774c1e8dc6b2f2                date_issue   
12  00136a27c7774c1e8dc6b2f2            vendor_address   
13  00136a27c7774c1e8dc6b2f2               vendor_name   
14  002e3cf97973428f905671b3                amount_due   

                                                value  page 

In [14]:
print(
    canonical_df["field"]
    .value_counts()
)

field
customer_billing_name       3736
amount_due                  3674
date_issue                  3653
amount_total_gross          3649
vendor_address              3487
customer_billing_address    3430
vendor_name                 3372
Name: count, dtype: int64


In [15]:
with zipfile.ZipFile(docile_zip, "r") as z:

    sample_id = invoice_gt_df["document_id"].iloc[0]

    ocr_data = json.loads(
        z.read(f"ocr/{sample_id}.json")
    )

words = []

for page_idx, page in enumerate(ocr_data["pages"]):

    for block in page["blocks"]:

        for line in block["lines"]:

            for word in line["words"]:

                words.append({
                    "page": page_idx,
                    "text": word["value"],
                    "bbox": word["bbox"]
                })

ocr_df = pd.DataFrame(words)

print("OCR words:", ocr_df.shape)
print(ocr_df.head(20))

KeyError: 'bbox'

In [16]:
with zipfile.ZipFile(docile_zip, "r") as z:

    sample_id = invoice_gt_df["document_id"].iloc[0]

    ocr_data = json.loads(
        z.read(f"ocr/{sample_id}.json")
    )

sample_word = (
    ocr_data["pages"][0]
    ["blocks"][0]
    ["lines"][0]
    ["words"][0]
)

print(sample_word)

{'value': 'WIRE', 'confidence': 0.9773061871528625, 'geometry': [[0.5224609375, 0.1689453125], [0.5595703125, 0.1806640625]], 'snapped_geometry': [[0.5254137115839244, 0.17015590200445435], [0.5567375886524822, 0.17817371937639198]]}


In [17]:
words = []

for page_idx, page in enumerate(ocr_data["pages"]):

    for block in page["blocks"]:

        for line in block["lines"]:

            for word in line["words"]:

                geometry = word["geometry"]

                x1 = geometry[0][0]
                y1 = geometry[0][1]
                x2 = geometry[1][0]
                y2 = geometry[1][1]

                words.append({
                    "page": page_idx,
                    "text": word["value"],
                    "confidence": word["confidence"],
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                })

ocr_df = pd.DataFrame(words)

print("OCR words:", ocr_df.shape)
print(ocr_df.head(20))

OCR words: (175, 7)
    page           text  confidence        x1        y1        x2        y2
0      0           WIRE    0.977306  0.522461  0.168945  0.559570  0.180664
1      0  INSTRUCTIONS:    0.734411  0.559570  0.167969  0.660156  0.181641
2      0   NationsBank,    0.887772  0.523438  0.179688  0.599609  0.192383
3      0           N.A.    0.991446  0.599609  0.179688  0.626953  0.191406
4      0     Baltimore,    0.987523  0.522461  0.202148  0.580078  0.213867
5      0             MD    0.999779  0.580078  0.201172  0.604492  0.213867
6      0          21202    0.999980  0.603516  0.201172  0.641602  0.212891
7      0        Routing    0.993045  0.522461  0.212891  0.567383  0.224609
8      0            No.    0.999560  0.567383  0.211914  0.590820  0.224609
9      0      052001633    0.947946  0.589844  0.212891  0.655273  0.223633
10     0          REMIT    0.995124  0.698242  0.169922  0.741211  0.181641
11     0            TO:    0.999506  0.739258  0.169922  0.764648  0

In [18]:
def bbox_iou(box1, box2):
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
        return 0.0

    intersection = (
        (inter_x2 - inter_x1) *
        (inter_y2 - inter_y1)
    )

    area1 = (ax2 - ax1) * (ay2 - ay1)
    area2 = (bx2 - bx1) * (by2 - by1)

    union = area1 + area2 - intersection

    return intersection / union


sample_doc = invoice_extraction_df[
    invoice_extraction_df["document_id"] == sample_id
].reset_index(drop=True)

overlaps = []

for i in range(len(sample_doc)):
    for j in range(i + 1, len(sample_doc)):

        if sample_doc.loc[i, "page"] != sample_doc.loc[j, "page"]:
            continue

        iou = bbox_iou(
            sample_doc.loc[i, "bbox"],
            sample_doc.loc[j, "bbox"]
        )

        if iou > 0:
            overlaps.append({
                "field_1": sample_doc.loc[i, "field"],
                "field_2": sample_doc.loc[j, "field"],
                "iou": iou,
                "value_1": sample_doc.loc[i, "value"],
                "value_2": sample_doc.loc[j, "value"]
            })

overlap_df = pd.DataFrame(overlaps)

print(overlap_df)

                 field_1                   field_2       iou  \
0            vendor_name            vendor_address  0.644457   
1  customer_billing_name  customer_billing_address  0.187641   
2     amount_total_gross                amount_due  0.848780   

                                    value_1  \
0  BIO RELIANCE Testing & Development, Inc.   
1                 LORILLARD RESEARCH CENTER   
2                                $21,000.00   

                                             value_2  
0  BIORELIANCE- Testing & Development, Inc.\n1492...  
1  LORILLARD RESEARCH CENTER\nATTN: MS MELANEE BE...  
2                                         $21,000.00  


In [19]:
vendor_annotations = invoice_extraction_df[
    (invoice_extraction_df["document_id"] == sample_id) &
    (invoice_extraction_df["field"] == "vendor_name")
].reset_index(drop=True)

print(vendor_annotations[["value", "page", "bbox"]])

                                      value  page  \
0  BIO RELIANCE Testing & Development, Inc.     0   
1   BioReliance Testing & Development, Inc.     0   
2   BioReliance Testing & Development, Inc.     0   

                                                bbox  
0  [0.0726911179849757, 0.1801480926047427, 0.463...  
1  [0.6949711431593889, 0.17973300483376403, 0.91...  
2  [0.6933190722960939, 0.26316564680047666, 0.91...  


In [20]:
def word_center(word_row):
    return (
        (word_row["x1"] + word_row["x2"]) / 2,
        (word_row["y1"] + word_row["y2"]) / 2
    )


def inside_bbox(word_row, bbox):
    cx, cy = word_center(word_row)

    x1, y1, x2, y2 = bbox

    return (
        x1 <= cx <= x2
        and
        y1 <= cy <= y2
    )


for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched_words = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if inside_bbox(word, annotation["bbox"]):
            matched_words.append(word["text"])

    print("\nAnnotated value:")
    print(annotation["value"])

    print("Matched OCR words:")
    print(matched_words)


Annotated value:
BIO RELIANCE Testing & Development, Inc.
Matched OCR words:
['BIORELIANCE', 'Testing', '&', 'Development,', 'Inc.']

Annotated value:
BioReliance Testing & Development, Inc.
Matched OCR words:
['BioReliance', 'Testing', '&', 'Development,', 'Inc.']

Annotated value:
BioReliance Testing & Development, Inc.
Matched OCR words:
['BioReliance', '&', 'Testing', 'Development,']


In [21]:
def bbox_iou(box1, box2):
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    if inter_x2 <= inter_x1 or inter_y2 <= inter_y1:
        return 0.0

    intersection = (
        (inter_x2 - inter_x1) *
        (inter_y2 - inter_y1)
    )

    area1 = (ax2 - ax1) * (ay2 - ay1)
    area2 = (bx2 - bx1) * (by2 - by1)

    union = area1 + area2 - intersection

    return intersection / union


def word_matches_annotation(word_row, annotation_bbox):
    word_bbox = [
        word_row["x1"],
        word_row["y1"],
        word_row["x2"],
        word_row["y2"]
    ]

    # Accept either substantial overlap or word center inside annotation
    iou = bbox_iou(word_bbox, annotation_bbox)

    cx = (word_row["x1"] + word_row["x2"]) / 2
    cy = (word_row["y1"] + word_row["y2"]) / 2

    x1, y1, x2, y2 = annotation_bbox

    center_inside = (
        x1 <= cx <= x2 and
        y1 <= cy <= y2
    )

    return iou > 0.1 or center_inside

In [22]:
for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched_words = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched_words.append(word["text"])

    print("\nAnnotated:")
    print(annotation["value"])

    print("Matched:")
    print(matched_words)


Annotated:
BIO RELIANCE Testing & Development, Inc.
Matched:
['BIORELIANCE', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', '&', 'Testing', 'Development,']


In [23]:
for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word)

    matched_df = pd.DataFrame(matched)

    if not matched_df.empty:
        matched_df = matched_df.sort_values(
            ["y1", "x1"]
        )

    print("\nAnnotated:")
    print(annotation["value"])

    print("Matched in visual order:")
    print(matched_df["text"].tolist())


Annotated:
BIO RELIANCE Testing & Development, Inc.
Matched in visual order:
['BIORELIANCE', 'Testing', '&', 'Inc.', 'Development,']

Annotated:
BioReliance Testing & Development, Inc.
Matched in visual order:
['Testing', '&', 'BioReliance', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched in visual order:
['Testing', '&', 'BioReliance', 'Development,']


In [24]:
ocr_words = []

for page_idx, page in enumerate(ocr_data["pages"]):

    for block_idx, block in enumerate(page["blocks"]):

        for line_idx, line in enumerate(block["lines"]):

            for word_idx, word in enumerate(line["words"]):

                geometry = word["geometry"]

                ocr_words.append({
                    "page": page_idx,
                    "block_id": block_idx,
                    "line_id": line_idx,
                    "word_id": word_idx,
                    "text": word["value"],
                    "confidence": word["confidence"],
                    "x1": geometry[0][0],
                    "y1": geometry[0][1],
                    "x2": geometry[1][0],
                    "y2": geometry[1][1]
                })

ocr_df = pd.DataFrame(ocr_words)

print(ocr_df.head(20))

    page  block_id  line_id  word_id           text  confidence        x1  \
0      0         0        0        0           WIRE    0.977306  0.522461   
1      0         0        0        1  INSTRUCTIONS:    0.734411  0.559570   
2      0         0        1        0   NationsBank,    0.887772  0.523438   
3      0         0        1        1           N.A.    0.991446  0.599609   
4      0         0        2        0     Baltimore,    0.987523  0.522461   
5      0         0        2        1             MD    0.999779  0.580078   
6      0         0        2        2          21202    0.999980  0.603516   
7      0         0        3        0        Routing    0.993045  0.522461   
8      0         0        3        1            No.    0.999560  0.567383   
9      0         0        3        2      052001633    0.947946  0.589844   
10     0         1        0        0          REMIT    0.995124  0.698242   
11     0         1        0        1            TO:    0.999506  0.739258   

In [25]:
for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word)

    matched_df = pd.DataFrame(matched)

    if not matched_df.empty:
        matched_df = matched_df.sort_values(
            ["block_id", "line_id", "word_id"]
        )

    print("\nAnnotated:")
    print(annotation["value"])

    print("Matched:")
    print(matched_df["text"].tolist())


Annotated:
BIO RELIANCE Testing & Development, Inc.
Matched:
['BIORELIANCE', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', '&', 'Testing', 'Development,']


In [26]:
annotation = vendor_annotations.iloc[2]

matched = []

for j in range(len(ocr_df)):
    word = ocr_df.iloc[j]

    if word["page"] != annotation["page"]:
        continue

    if word_matches_annotation(
        word,
        annotation["bbox"]
    ):
        matched.append(word)

matched_df = pd.DataFrame(matched)

print("ANNOTATION:")
print(annotation["value"])
print("\nANNOTATION BBOX:")
print(annotation["bbox"])

print("\nMATCHED OCR WORDS:")
print(
    matched_df[
        [
            "block_id",
            "line_id",
            "word_id",
            "text",
            "x1",
            "y1",
            "x2",
            "y2"
        ]
    ].to_string(index=False)
)

ANNOTATION:
BioReliance Testing & Development, Inc.

ANNOTATION BBOX:
[0.6933190722960939, 0.26316564680047666, 0.9174500194164358, 0.2743730166169007]

MATCHED OCR WORDS:
 block_id  line_id  word_id         text       x1       y1       x2       y2
        5        0        4  BioReliance 0.694336 0.263672 0.763672 0.274414
        6        0        0            & 0.804688 0.262695 0.819336 0.276367
        6        1        0      Testing 0.762695 0.262695 0.806641 0.277344
        6        1        1 Development, 0.817383 0.263672 0.897461 0.277344


In [27]:
def expand_bbox(bbox, margin=0.005):
    x1, y1, x2, y2 = bbox

    return [
        max(0, x1 - margin),
        max(0, y1 - margin),
        min(1, x2 + margin),
        min(1, y2 + margin)
    ]


def word_matches_annotation(word_row, annotation_bbox):
    expanded = expand_bbox(annotation_bbox, margin=0.005)

    word_bbox = [
        word_row["x1"],
        word_row["y1"],
        word_row["x2"],
        word_row["y2"]
    ]

    iou = bbox_iou(word_bbox, expanded)

    cx = (word_row["x1"] + word_row["x2"]) / 2
    cy = (word_row["y1"] + word_row["y2"]) / 2

    x1, y1, x2, y2 = expanded

    center_inside = (
        x1 <= cx <= x2 and
        y1 <= cy <= y2
    )

    return iou > 0.05 or center_inside

In [28]:
for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word)

    matched_df = pd.DataFrame(matched)

    if not matched_df.empty:
        matched_df = matched_df.sort_values(
            ["block_id", "line_id", "word_id"]
        )

    print("\nAnnotated:")
    print(annotation["value"])

    print("Matched:")
    print(matched_df["text"].tolist())


Annotated:
BIO RELIANCE Testing & Development, Inc.
Matched:
['BIORELIANCE', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['REMIT', 'TO:', 'BioReliance', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', '&', 'Testing', 'Development,']


In [29]:
def word_matches_annotation(word_row, annotation_bbox):
    cx = (word_row["x1"] + word_row["x2"]) / 2
    cy = (word_row["y1"] + word_row["y2"]) / 2

    x1, y1, x2, y2 = annotation_bbox

    return (
        x1 <= cx <= x2
        and
        y1 <= cy <= y2
    )

In [30]:
for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word)

    matched_df = pd.DataFrame(matched)

    if not matched_df.empty:
        matched_df = matched_df.sort_values(
            ["block_id", "line_id", "word_id"]
        )

    print("\nAnnotated:")
    print(annotation["value"])

    print("Matched:")
    print(matched_df["text"].tolist())


Annotated:
BIO RELIANCE Testing & Development, Inc.
Matched:
['BIORELIANCE', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', 'Testing', '&', 'Development,', 'Inc.']

Annotated:
BioReliance Testing & Development, Inc.
Matched:
['BioReliance', '&', 'Testing', 'Development,']


In [31]:
def normalize_tokens(text):
    return re.findall(r"\b\w+\b", str(text).lower())


results = []

for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word["text"])

    annotation_tokens = normalize_tokens(annotation["value"])
    matched_tokens = normalize_tokens(" ".join(matched))

    matched_count = sum(
        1 for token in annotation_tokens
        if token in matched_tokens
    )

    coverage = (
        matched_count / len(annotation_tokens)
        if annotation_tokens
        else 0
    )

    results.append({
        "annotation": annotation["value"],
        "matched": " ".join(matched),
        "coverage": coverage
    })

match_quality_df = pd.DataFrame(results)

print(match_quality_df)

NameError: name 're' is not defined

In [32]:
import re

def normalize_tokens(text):
    return re.findall(r"\b\w+\b", str(text).lower())


results = []

for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word["text"])

    annotation_tokens = normalize_tokens(annotation["value"])
    matched_tokens = normalize_tokens(" ".join(matched))

    matched_count = sum(
        1 for token in annotation_tokens
        if token in matched_tokens
    )

    coverage = (
        matched_count / len(annotation_tokens)
        if annotation_tokens
        else 0
    )

    results.append({
        "annotation": annotation["value"],
        "matched": " ".join(matched),
        "coverage": coverage
    })

match_quality_df = pd.DataFrame(results)

print(match_quality_df)

                                 annotation  \
0  BIO RELIANCE Testing & Development, Inc.   
1   BioReliance Testing & Development, Inc.   
2   BioReliance Testing & Development, Inc.   

                                   matched  coverage  
0  BIORELIANCE Testing & Development, Inc.      0.60  
1  BioReliance Testing & Development, Inc.      1.00  
2       BioReliance & Testing Development,      0.75  


In [33]:
from difflib import SequenceMatcher
import re


def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text


results = []

for i in range(len(vendor_annotations)):

    annotation = vendor_annotations.iloc[i]

    matched = []

    for j in range(len(ocr_df)):

        word = ocr_df.iloc[j]

        if word["page"] != annotation["page"]:
            continue

        if word_matches_annotation(
            word,
            annotation["bbox"]
        ):
            matched.append(word["text"])

    annotated_normalized = normalize_text(
        annotation["value"]
    )

    matched_normalized = normalize_text(
        " ".join(matched)
    )

    similarity = SequenceMatcher(
        None,
        annotated_normalized,
        matched_normalized
    ).ratio()

    results.append({
        "annotation": annotation["value"],
        "matched": " ".join(matched),
        "similarity": round(similarity, 3)
    })

match_quality_df = pd.DataFrame(results)

print(match_quality_df)

                                 annotation  \
0  BIO RELIANCE Testing & Development, Inc.   
1   BioReliance Testing & Development, Inc.   
2   BioReliance Testing & Development, Inc.   

                                   matched  similarity  
0  BIORELIANCE Testing & Development, Inc.       1.000  
1  BioReliance Testing & Development, Inc.       1.000  
2       BioReliance & Testing Development,       0.951  


In [34]:
import numpy as np

test_doc_ids = invoice_gt_df["document_id"].drop_duplicates().head(100)

quality_results = []

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_id in test_doc_ids:

        # Load OCR
        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

        ocr_words = []

        for page_idx, page in enumerate(ocr_data["pages"]):

            for block_idx, block in enumerate(page["blocks"]):

                for line_idx, line in enumerate(block["lines"]):

                    for word_idx, word in enumerate(line["words"]):

                        geometry = word["geometry"]

                        ocr_words.append({
                            "page": page_idx,
                            "block_id": block_idx,
                            "line_id": line_idx,
                            "word_id": word_idx,
                            "text": word["value"],
                            "confidence": word["confidence"],
                            "x1": geometry[0][0],
                            "y1": geometry[0][1],
                            "x2": geometry[1][0],
                            "y2": geometry[1][1]
                        })

        ocr_df_sample = pd.DataFrame(ocr_words)

        annotations = invoice_extraction_df[
            invoice_extraction_df["document_id"] == doc_id
        ]

        for i in range(len(annotations)):

            annotation = annotations.iloc[i]

            matched = []

            for j in range(len(ocr_df_sample)):

                word = ocr_df_sample.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    matched.append(word)

            matched_df = pd.DataFrame(matched)

            if not matched_df.empty:
                matched_df = matched_df.sort_values(
                    ["block_id", "line_id", "word_id"]
                )

                matched_text = " ".join(
                    matched_df["text"].tolist()
                )
            else:
                matched_text = ""

            similarity = SequenceMatcher(
                None,
                normalize_text(annotation["value"]),
                normalize_text(matched_text)
            ).ratio()

            quality_results.append({
                "document_id": doc_id,
                "field": annotation["field"],
                "similarity": similarity,
                "matched": matched_text
            })

quality_df = pd.DataFrame(quality_results)

print("Annotations checked:", len(quality_df))
print()
print(
    quality_df.groupby("field")["similarity"]
    .agg(["count", "mean", "min", "median"])
)

Annotations checked: 805

                          count      mean       min  median
field                                                      
amount_due                  112  0.982909  0.500000     1.0
amount_total_gross          101  0.975272  0.500000     1.0
customer_billing_address     87  0.977364  0.647059     1.0
customer_billing_name       100  0.993920  0.851064     1.0
date_issue                  116  0.997100  0.888889     1.0
vendor_address              142  0.967576  0.571429     1.0
vendor_name                 147  0.983569  0.659091     1.0


In [35]:
quality_summary = (
    quality_df
    .groupby("field")["similarity"]
    .agg(
        total="count",
        good=lambda x: (x >= 0.80).sum(),
        good_percent=lambda x: (x >= 0.80).mean() * 100
    )
)

print(quality_summary)

                          total  good  good_percent
field                                              
amount_due                  112   110     98.214286
amount_total_gross          101    98     97.029703
customer_billing_address     87    84     96.551724
customer_billing_name       100   100    100.000000
date_issue                  116   116    100.000000
vendor_address              142   132     92.957746
vendor_name                 147   143     97.278912


In [36]:
quality_good_df = quality_df[
    quality_df["similarity"] >= 0.80
].copy()

quality_bad_df = quality_df[
    quality_df["similarity"] < 0.80
].copy()

print("Accepted annotations:", len(quality_good_df))
print("Rejected annotations:", len(quality_bad_df))

print("\nAccepted by field:")
print(
    quality_good_df["field"]
    .value_counts()
)

Accepted annotations: 783
Rejected annotations: 22

Accepted by field:
field
vendor_name                 143
vendor_address              132
date_issue                  116
amount_due                  110
customer_billing_name       100
amount_total_gross           98
customer_billing_address     84
Name: count, dtype: int64


In [37]:
full_quality_results = []

target_fields = [
    "date_issue",
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "amount_total_gross",
    "amount_due"
]

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_num, doc_id in enumerate(all_ids):

        annotation_path = f"annotations/{doc_id}.json"

        annotation = json.loads(
            z.read(annotation_path)
        )

        if annotation["metadata"]["document_type"] != "tax_invoice":
            continue

        # Load OCR
        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

        ocr_words = []

        for page_idx, page in enumerate(ocr_data["pages"]):

            for block_idx, block in enumerate(page["blocks"]):

                for line_idx, line in enumerate(block["lines"]):

                    for word_idx, word in enumerate(line["words"]):

                        geometry = word["geometry"]

                        ocr_words.append({
                            "page": page_idx,
                            "block_id": block_idx,
                            "line_id": line_idx,
                            "word_id": word_idx,
                            "text": word["value"],
                            "x1": geometry[0][0],
                            "y1": geometry[0][1],
                            "x2": geometry[1][0],
                            "y2": geometry[1][1]
                        })

        ocr_df_doc = pd.DataFrame(ocr_words)

        annotations = [
            item
            for item in annotation["field_extractions"]
            if item["fieldtype"] in target_fields
        ]

        for item in annotations:

            matched = []

            for j in range(len(ocr_df_doc)):

                word = ocr_df_doc.iloc[j]

                if word["page"] != item["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    item["bbox"]
                ):
                    matched.append(word)

            matched_df = pd.DataFrame(matched)

            if not matched_df.empty:

                matched_df = matched_df.sort_values(
                    ["block_id", "line_id", "word_id"]
                )

                matched_text = " ".join(
                    matched_df["text"].tolist()
                )

            else:
                matched_text = ""

            similarity = SequenceMatcher(
                None,
                normalize_text(item["text"]),
                normalize_text(matched_text)
            ).ratio()

            full_quality_results.append({
                "document_id": doc_id,
                "field": item["fieldtype"],
                "value": item["text"],
                "matched_text": matched_text,
                "similarity": similarity,
                "page": item["page"],
                "bbox": item["bbox"]
            })

        if (doc_num + 1) % 250 == 0:
            print(f"Processed: {doc_num + 1} documents")

full_quality_df = pd.DataFrame(full_quality_results)

print("\nTotal annotations:", len(full_quality_df))

Processed: 250 documents
Processed: 500 documents
Processed: 750 documents
Processed: 1000 documents
Processed: 1500 documents
Processed: 1750 documents
Processed: 2500 documents
Processed: 2750 documents
Processed: 3250 documents
Processed: 3500 documents
Processed: 4250 documents
Processed: 4500 documents
Processed: 4750 documents
Processed: 5250 documents
Processed: 5500 documents

Total annotations: 30823


In [38]:
quality_stats = (
    full_quality_df
    .groupby("field")["similarity"]
    .agg(
        total="count",
        mean="mean",
        min="min",
        median="median",
        accepted=lambda x: (x >= 0.80).sum(),
        accepted_percent=lambda x: (x >= 0.80).mean() * 100
    )
)

print(quality_stats)

                          total      mean       min  median  accepted  \
field                                                                   
amount_due                 4270  0.966981  0.000000     1.0      4081   
amount_total_gross         3910  0.958553  0.000000     1.0      3694   
customer_billing_address   3706  0.982986  0.500000     1.0      3626   
customer_billing_name      4081  0.989363  0.384615     1.0      4036   
date_issue                 4431  0.984889  0.000000     1.0      4341   
vendor_address             5285  0.969160  0.000000     1.0      5055   
vendor_name                5140  0.986170  0.000000     1.0      5069   

                          accepted_percent  
field                                       
amount_due                       95.573770  
amount_total_gross               94.475703  
customer_billing_address         97.841338  
customer_billing_name            98.897329  
date_issue                       97.968856  
vendor_address             

In [39]:
vendor_training_df = full_quality_df[
    (full_quality_df["field"] == "vendor_name") &
    (full_quality_df["similarity"] >= 0.80)
].copy()

print("Vendor-name annotations:", len(vendor_training_df))
print(vendor_training_df.head())

Vendor-name annotations: 5069
                 document_id        field  \
1   00134dd365a24343b35b78c6  vendor_name   
3   00134dd365a24343b35b78c6  vendor_name   
4   00134dd365a24343b35b78c6  vendor_name   
11  00136a27c7774c1e8dc6b2f2  vendor_name   
12  00136a27c7774c1e8dc6b2f2  vendor_name   

                                       value  \
1   BIO RELIANCE Testing & Development, Inc.   
3    BioReliance Testing & Development, Inc.   
4    BioReliance Testing & Development, Inc.   
11                       KMOZ 92.3 The Moose   
12                       KMOZ 92.3 The Moose   

                               matched_text  similarity  page  \
1   BIORELIANCE Testing & Development, Inc.     1.00000     0   
3   BioReliance Testing & Development, Inc.     1.00000     0   
4        BioReliance & Testing Development,     0.95082     0   
11                      KMOZ 92.3 The Moose     1.00000     0   
12                      KMOZ 92.3 The Moose     1.00000     1   

                   

In [40]:
def create_vendor_labels(doc_id):
    with zipfile.ZipFile(docile_zip, "r") as z:

        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    words.append({
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1],
                        "label": "O"
                    })

    words_df = pd.DataFrame(words)

    annotations = vendor_training_df[
        vendor_training_df["document_id"] == doc_id
    ]

    for _, annotation in annotations.iterrows():

        matched_indices = []

        for j in range(len(words_df)):

            word = words_df.iloc[j]

            if word["page"] != annotation["page"]:
                continue

            if word_matches_annotation(
                word,
                annotation["bbox"]
            ):
                matched_indices.append(j)

        for position, index in enumerate(matched_indices):

            if position == 0:
                words_df.loc[index, "label"] = "B-VENDOR_NAME"
            else:
                words_df.loc[index, "label"] = "I-VENDOR_NAME"

    return words_df

In [41]:
sample_vendor_doc = vendor_training_df["document_id"].iloc[0]

vendor_labeled_df = create_vendor_labels(
    sample_vendor_doc
)

print(
    vendor_labeled_df[
        vendor_labeled_df["label"] != "O"
    ][["text", "label"]]
)

            text          label
12   BioReliance  B-VENDOR_NAME
13       Testing  I-VENDOR_NAME
14             &  I-VENDOR_NAME
15  Development,  I-VENDOR_NAME
16          Inc.  I-VENDOR_NAME
33   BIORELIANCE  B-VENDOR_NAME
54       Testing  I-VENDOR_NAME
55             &  I-VENDOR_NAME
56  Development,  I-VENDOR_NAME
57          Inc.  I-VENDOR_NAME
69   BioReliance  B-VENDOR_NAME
72             &  I-VENDOR_NAME
73       Testing  I-VENDOR_NAME
74  Development,  I-VENDOR_NAME


In [42]:
from sklearn.model_selection import train_test_split

invoice_doc_ids = invoice_gt_df[
    "document_id"
].drop_duplicates().tolist()

meta_train_ids, meta_temp_ids = train_test_split(
    invoice_doc_ids,
    test_size=0.30,
    random_state=42
)

meta_val_ids, meta_test_ids = train_test_split(
    meta_temp_ids,
    test_size=0.50,
    random_state=42
)

print("Train documents:", len(meta_train_ids))
print("Validation documents:", len(meta_val_ids))
print("Test documents:", len(meta_test_ids))

Train documents: 2695
Validation documents: 577
Test documents: 578


In [43]:
vendor_labels = []

with zipfile.ZipFile(docuile_zip, "r") as z:
    pass

NameError: name 'docuile_zip' is not defined

In [44]:
vendor_labels = []

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_id in invoice_doc_ids:

        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

        words = []

        for page_idx, page in enumerate(ocr_data["pages"]):

            for block_idx, block in enumerate(page["blocks"]):

                for line_idx, line in enumerate(block["lines"]):

                    for word_idx, word in enumerate(line["words"]):

                        geometry = word["geometry"]

                        words.append({
                            "document_id": doc_id,
                            "page": page_idx,
                            "block_id": block_idx,
                            "line_id": line_idx,
                            "word_id": word_idx,
                            "text": word["value"],
                            "confidence": word["confidence"],
                            "x1": geometry[0][0],
                            "y1": geometry[0][1],
                            "x2": geometry[1][0],
                            "y2": geometry[1][1],
                            "label": "O"
                        })

        words_df = pd.DataFrame(words)

        annotations = vendor_training_df[
            vendor_training_df["document_id"] == doc_id
        ]

        for _, annotation in annotations.iterrows():

            matched_indices = []

            for j in range(len(words_df)):

                word = words_df.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    matched_indices.append(j)

            for position, index in enumerate(matched_indices):

                if position == 0:
                    words_df.loc[index, "label"] = "B-VENDOR_NAME"
                else:
                    words_df.loc[index, "label"] = "I-VENDOR_NAME"

        vendor_labels.extend(
            words_df.to_dict("records")
        )

        if len(vendor_labels) % 1000 == 0:
            print("Labeled OCR words:", len(vendor_labels))

vendor_labels_df = pd.DataFrame(vendor_labels)

print("\nTotal OCR words:", len(vendor_labels_df))
print(vendor_labels_df["label"].value_counts())

Labeled OCR words: 71000
Labeled OCR words: 248000
Labeled OCR words: 512000
Labeled OCR words: 521000
Labeled OCR words: 622000
Labeled OCR words: 767000
Labeled OCR words: 847000

Total OCR words: 940348
label
O                923878
I-VENDOR_NAME     11403
B-VENDOR_NAME      5067
Name: count, dtype: int64


In [45]:
train_vendor_df = vendor_labels_df[
    vendor_labels_df["document_id"].isin(meta_train_ids)
].copy()

val_vendor_df = vendor_labels_df[
    vendor_labels_df["document_id"].isin(meta_val_ids)
].copy()

test_vendor_df = vendor_labels_df[
    vendor_labels_df["document_id"].isin(meta_test_ids)
].copy()

print("Train:", train_vendor_df.shape)
print("Validation:", val_vendor_df.shape)
print("Test:", test_vendor_df.shape)

print("\nTrain labels:")
print(train_vendor_df["label"].value_counts())

print("\nValidation labels:")
print(val_vendor_df["label"].value_counts())

print("\nTest labels:")
print(test_vendor_df["label"].value_counts())

Train: (659937, 12)
Validation: (143347, 12)
Test: (137064, 12)

Train labels:
label
O                648585
I-VENDOR_NAME      7838
B-VENDOR_NAME      3514
Name: count, dtype: int64

Validation labels:
label
O                140860
I-VENDOR_NAME      1721
B-VENDOR_NAME       766
Name: count, dtype: int64

Test labels:
label
O                134433
I-VENDOR_NAME      1844
B-VENDOR_NAME       787
Name: count, dtype: int64


In [46]:
def vendor_document_stats(df):
    total_docs = df["document_id"].nunique()

    positive_docs = df[
        df["label"].isin(["B-VENDOR_NAME", "I-VENDOR_NAME"])
    ]["document_id"].nunique()

    return {
        "total_documents": total_docs,
        "documents_with_vendor_name": positive_docs,
        "documents_without_vendor_name": total_docs - positive_docs
    }


print("Train:", vendor_document_stats(train_vendor_df))
print("Validation:", vendor_document_stats(val_vendor_df))
print("Test:", vendor_document_stats(test_vendor_df))

Train: {'total_documents': 2695, 'documents_with_vendor_name': 2331, 'documents_without_vendor_name': 364}
Validation: {'total_documents': 577, 'documents_with_vendor_name': 496, 'documents_without_vendor_name': 81}
Test: {'total_documents': 578, 'documents_with_vendor_name': 500, 'documents_without_vendor_name': 78}


In [47]:
all_vendor_annotated_docs = set(
    full_quality_df[
        full_quality_df["field"] == "vendor_name"
    ]["document_id"]
)

accepted_vendor_docs = set(
    vendor_training_df["document_id"]
)

rejected_vendor_docs = (
    all_vendor_annotated_docs
    - accepted_vendor_docs
)

print("Documents with vendor annotations:",
      len(all_vendor_annotated_docs))

print("Documents with accepted vendor annotations:",
      len(accepted_vendor_docs))

print("Documents with rejected/uncertain vendor annotations:",
      len(rejected_vendor_docs))

print("Documents with no vendor annotation:",
      len(invoice_doc_ids) - len(all_vendor_annotated_docs))

Documents with vendor annotations: 3372
Documents with accepted vendor annotations: 3327
Documents with rejected/uncertain vendor annotations: 45
Documents with no vendor annotation: 478


In [48]:
uncertain_docs = set(rejected_vendor_docs)

vendor_model_df = vendor_labels_df[
    ~vendor_labels_df["document_id"].isin(uncertain_docs)
].copy()

print("Original rows:", len(vendor_labels_df))
print("After removing uncertain documents:", len(vendor_model_df))

print("\nDocuments:")
print(vendor_model_df["document_id"].nunique())

print("\nLabels:")
print(vendor_model_df["label"].value_counts())


Original rows: 940348
After removing uncertain documents: 932700

Documents:
3805

Labels:
label
O                916230
I-VENDOR_NAME     11403
B-VENDOR_NAME      5067
Name: count, dtype: int64


In [49]:
from transformers import LayoutLMTokenizerFast

layout_tokenizer = LayoutLMTokenizerFast.from_pretrained(
    "microsoft/layoutlm-base-uncased"
)

print("LayoutLM tokenizer loaded")

LayoutLM tokenizer loaded


In [50]:
sample_doc_id = (
    vendor_model_df["document_id"]
    .iloc[0]
)

sample_words_df = vendor_model_df[
    vendor_model_df["document_id"] == sample_doc_id
].copy()

sample_words_df = sample_words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
)

words = sample_words_df["text"].tolist()
word_labels = sample_words_df["label"].tolist()

# LayoutLM expects coordinates between 0 and 1000
boxes = []

for _, row in sample_words_df.iterrows():
    boxes.append([
        int(row["x1"] * 1000),
        int(row["y1"] * 1000),
        int(row["x2"] * 1000),
        int(row["y2"] * 1000)
    ])

encoded = layout_tokenizer(
    words,
    boxes=boxes,
    is_split_into_words=True,
    truncation=True,
    max_length=512,
    return_attention_mask=True
)

print("Document:", sample_doc_id)
print("Words:", len(words))
print("Tokens:", len(encoded["input_ids"]))
print("First 15 tokens:")
print(layout_tokenizer.convert_ids_to_tokens(
    encoded["input_ids"][:15]
))
print("\nFirst 15 boxes:")
print(encoded["bbox"][:15])

Document: 00134dd365a24343b35b78c6
Words: 175
Tokens: 377
First 15 tokens:
['[CLS]', 'wire', 'instructions', ':', 'nations', '##bank', ',', 'n', '.', 'a', '.', 'baltimore', ',', 'md', '212']

First 15 boxes:


KeyError: 'bbox'

In [ ]:
sample_doc_id = vendor_model_df["document_id"].iloc[0]

sample_words_df = vendor_model_df[
    vendor_model_df["document_id"] == sample_doc_id
].copy()

sample_words_df = sample_words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
)

words = sample_words_df["text"].tolist()
word_labels = sample_words_df["label"].tolist()

# Convert coordinates from 0-1 to LayoutLM's 0-1000 range
word_boxes = []

for _, row in sample_words_df.iterrows():

    word_boxes.append([
        int(row["x1"] * 1000),
        int(row["y1"] * 1000),
        int(row["x2"] * 1000),
        int(row["y2"] * 1000)
    ])

# Tokenize
encoded = layout_tokenizer(
    words,
    boxes=word_boxes,
    is_split_into_words=True,
    truncation=True,
    max_length=256,
    return_attention_mask=True
)

# Map each token back to its original OCR word
token_word_ids = encoded.word_ids()

token_boxes = []
token_labels = []

for word_id in token_word_ids:

    if word_id is None:
        # Special tokens such as [CLS], [SEP]
        token_boxes.append([0, 0, 0, 0])
        token_labels.append("O")

    else:
        token_boxes.append(word_boxes[word_id])
        token_labels.append(word_labels[word_id])

print("Document:", sample_doc_id)
print("OCR words:", len(words))
print("Tokens:", len(encoded["input_ids"]))

print("\nFirst 20 tokens:")
print(
    layout_tokenizer.convert_ids_to_tokens(
        encoded["input_ids"][:20]
    )
)

print("\nFirst 20 token boxes:")
print(token_boxes[:20])

print("\nFirst 20 token labels:")
print(token_labels[:20])

Document: 00134dd365a24343b35b78c6
OCR words: 175
Tokens: 377

First 20 tokens:
['[CLS]', 'wire', 'instructions', ':', 'nations', '##bank', ',', 'n', '.', 'a', '.', 'baltimore', ',', 'md', '212', '##0', '##2', 'routing', 'no', '.']

First 20 token boxes:
[[0, 0, 0, 0], [522, 168, 559, 180], [559, 167, 660, 181], [559, 167, 660, 181], [523, 179, 599, 192], [523, 179, 599, 192], [523, 179, 599, 192], [599, 179, 626, 191], [599, 179, 626, 191], [599, 179, 626, 191], [599, 179, 626, 191], [522, 202, 580, 213], [522, 202, 580, 213], [580, 201, 604, 213], [603, 201, 641, 212], [603, 201, 641, 212], [603, 201, 641, 212], [522, 212, 567, 224], [567, 211, 590, 224], [567, 211, 590, 224]]

First 20 token labels:
['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']


In [52]:
for i in range(len(token_word_ids)):

    word_id = token_word_ids[i]

    if word_id is None:
        continue

    if token_labels[i] != "O":
        token = layout_tokenizer.convert_ids_to_tokens(
            [encoded["input_ids"][i]]
        )[0]

        print(
            i,
            "token:", token,
            "word:", words[word_id],
            "label:", token_labels[i],
            "bbox:", token_boxes[i]
        )

28 token: bio word: BioReliance label: B-VENDOR_NAME bbox: [697, 180, 765, 192]
29 token: ##rel word: BioReliance label: B-VENDOR_NAME bbox: [697, 180, 765, 192]
30 token: ##iance word: BioReliance label: B-VENDOR_NAME bbox: [697, 180, 765, 192]
31 token: testing word: Testing label: I-VENDOR_NAME bbox: [765, 179, 809, 194]
32 token: & word: & label: I-VENDOR_NAME bbox: [807, 179, 821, 193]
33 token: development word: Development, label: I-VENDOR_NAME bbox: [819, 180, 898, 194]
34 token: , word: Development, label: I-VENDOR_NAME bbox: [819, 180, 898, 194]
35 token: inc word: Inc. label: I-VENDOR_NAME bbox: [898, 181, 921, 193]
36 token: . word: Inc. label: I-VENDOR_NAME bbox: [898, 181, 921, 193]
76 token: bio word: BIORELIANCE label: B-VENDOR_NAME bbox: [76, 182, 266, 203]
77 token: ##rel word: BIORELIANCE label: B-VENDOR_NAME bbox: [76, 182, 266, 203]
78 token: ##iance word: BIORELIANCE label: B-VENDOR_NAME bbox: [76, 182, 266, 203]
131 token: testing word: Testing label: I-VENDOR_NA

In [53]:
token_labels = []

previous_word_id = None

for word_id in token_word_ids:

    if word_id is None:
        token_labels.append("O")
        previous_word_id = None
        continue

    original_label = word_labels[word_id]

    if original_label == "B-VENDOR_NAME":
        if word_id == previous_word_id:
            token_labels.append("I-VENDOR_NAME")
        else:
            token_labels.append("B-VENDOR_NAME")

    elif original_label == "I-VENDOR_NAME":
        token_labels.append("I-VENDOR_NAME")

    else:
        token_labels.append("O")

    previous_word_id = word_id

In [54]:
for i in range(len(token_word_ids)):

    word_id = token_word_ids[i]

    if word_id is None:
        continue

    if token_labels[i] != "O":

        token = layout_tokenizer.convert_ids_to_tokens(
            [encoded["input_ids"][i]]
        )[0]

        print(
            i,
            "token:", token,
            "word:", words[word_id],
            "label:", token_labels[i]
        )

28 token: bio word: BioReliance label: B-VENDOR_NAME
29 token: ##rel word: BioReliance label: I-VENDOR_NAME
30 token: ##iance word: BioReliance label: I-VENDOR_NAME
31 token: testing word: Testing label: I-VENDOR_NAME
32 token: & word: & label: I-VENDOR_NAME
33 token: development word: Development, label: I-VENDOR_NAME
34 token: , word: Development, label: I-VENDOR_NAME
35 token: inc word: Inc. label: I-VENDOR_NAME
36 token: . word: Inc. label: I-VENDOR_NAME
76 token: bio word: BIORELIANCE label: B-VENDOR_NAME
77 token: ##rel word: BIORELIANCE label: I-VENDOR_NAME
78 token: ##iance word: BIORELIANCE label: I-VENDOR_NAME
131 token: testing word: Testing label: I-VENDOR_NAME
132 token: & word: & label: I-VENDOR_NAME
133 token: development word: Development, label: I-VENDOR_NAME
134 token: , word: Development, label: I-VENDOR_NAME
135 token: inc word: Inc. label: I-VENDOR_NAME
136 token: . word: Inc. label: I-VENDOR_NAME
151 token: bio word: BioReliance label: B-VENDOR_NAME
152 token: ##r

In [55]:
def create_field_labels(doc_id, field_name):

    with zipfile.ZipFile(docile_zip, "r") as z:

        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    words.append({
                        "document_id": doc_id,
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1],
                        "label": "O"
                    })

    words_df = pd.DataFrame(words)

    annotations = full_quality_df[
        (full_quality_df["document_id"] == doc_id) &
        (full_quality_df["field"] == field_name) &
        (full_quality_df["similarity"] >= 0.80)
    ]

    for _, annotation in annotations.iterrows():

        matched_indices = []

        for j in range(len(words_df)):

            word = words_df.iloc[j]

            if word["page"] != annotation["page"]:
                continue

            if word_matches_annotation(
                word,
                annotation["bbox"]
            ):
                matched_indices.append(j)

        for position, index in enumerate(matched_indices):

            if position == 0:
                words_df.loc[index, "label"] = f"B-{field_name}"
            else:
                words_df.loc[index, "label"] = f"I-{field_name}"

    return words_df

In [56]:
test_field_df = create_field_labels(
    sample_doc_id,
    "vendor_name"
)

print(
    test_field_df[
        test_field_df["label"] != "O"
    ][["text", "label"]]
)

            text          label
12   BioReliance  B-vendor_name
13       Testing  I-vendor_name
14             &  I-vendor_name
15  Development,  I-vendor_name
16          Inc.  I-vendor_name
33   BIORELIANCE  B-vendor_name
54       Testing  I-vendor_name
55             &  I-vendor_name
56  Development,  I-vendor_name
57          Inc.  I-vendor_name
69   BioReliance  B-vendor_name
72             &  I-vendor_name
73       Testing  I-vendor_name
74  Development,  I-vendor_name


In [57]:
def create_field_labels(doc_id, field_name):

    # -----------------------------
    # 1. Load OCR data
    # -----------------------------
    with zipfile.ZipFile(docile_zip, "r") as z:

        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    words.append({
                        "document_id": doc_id,
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1],
                        "label": "O"
                    })

    words_df = pd.DataFrame(words)

    # -----------------------------
    # 2. Get accepted annotations
    # -----------------------------
    annotations = full_quality_df[
        (full_quality_df["document_id"] == doc_id) &
        (full_quality_df["field"] == field_name) &
        (full_quality_df["similarity"] >= 0.80)
    ]

    # -----------------------------
    # 3. Match OCR words to fields
    # -----------------------------
    for _, annotation in annotations.iterrows():

        matched_indices = []

        for j in range(len(words_df)):

            word = words_df.iloc[j]

            # Same page only
            if word["page"] != annotation["page"]:
                continue

            # Check whether OCR word belongs to annotation bbox
            if word_matches_annotation(
                word,
                annotation["bbox"]
            ):
                matched_indices.append(j)

        # -----------------------------
        # 4. Assign BIO labels
        # -----------------------------
        for position, index in enumerate(matched_indices):

            if position == 0:
                words_df.loc[index, "label"] = (
                    f"B-{field_name.upper()}"
                )
            else:
                words_df.loc[index, "label"] = (
                    f"I-{field_name.upper()}"
                )

    return words_df

In [58]:
sample_doc_id = vendor_training_df["document_id"].iloc[0]

test_field_df = create_field_labels(
    sample_doc_id,
    "vendor_name"
)

print(
    test_field_df[
        test_field_df["label"] != "O"
    ][["text", "label"]]
)

            text          label
12   BioReliance  B-VENDOR_NAME
13       Testing  I-VENDOR_NAME
14             &  I-VENDOR_NAME
15  Development,  I-VENDOR_NAME
16          Inc.  I-VENDOR_NAME
33   BIORELIANCE  B-VENDOR_NAME
54       Testing  I-VENDOR_NAME
55             &  I-VENDOR_NAME
56  Development,  I-VENDOR_NAME
57          Inc.  I-VENDOR_NAME
69   BioReliance  B-VENDOR_NAME
72             &  I-VENDOR_NAME
73       Testing  I-VENDOR_NAME
74  Development,  I-VENDOR_NAME


In [59]:
target_fields = [
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "date_issue",
    "amount_total_gross",
    "amount_due"
]

print("Target fields:")
for field in target_fields:
    print("-", field)

Target fields:
- vendor_name
- vendor_address
- customer_billing_name
- customer_billing_address
- date_issue
- amount_total_gross
- amount_due


In [60]:
def build_field_dataset(field_name):

    # Documents where this field has at least one
    # accepted ground-truth annotation
    accepted_docs = set(
        full_quality_df[
            (full_quality_df["field"] == field_name) &
            (full_quality_df["similarity"] >= 0.80)
        ]["document_id"]
    )

    # Documents where this field had an annotation
    # but all matches were rejected/uncertain
    annotated_docs = set(
        full_quality_df[
            full_quality_df["field"] == field_name
        ]["document_id"]
    )

    uncertain_docs = annotated_docs - accepted_docs

    # Remove uncertain documents from this task
    usable_docs = [
        doc_id
        for doc_id in invoice_doc_ids
        if doc_id not in uncertain_docs
    ]

    return {
        "accepted_docs": accepted_docs,
        "uncertain_docs": uncertain_docs,
        "usable_docs": usable_docs
    }

In [61]:
field_dataset_summary = []

for field in target_fields:

    info = build_field_dataset(field)

    field_dataset_summary.append({
        "field": field,
        "accepted_docs": len(info["accepted_docs"]),
        "uncertain_docs": len(info["uncertain_docs"]),
        "usable_docs": len(info["usable_docs"])
    })

field_dataset_summary_df = pd.DataFrame(
    field_dataset_summary
)

print(field_dataset_summary_df)

                      field  accepted_docs  uncertain_docs  usable_docs
0               vendor_name           3327              45         3805
1            vendor_address           3383             104         3746
2     customer_billing_name           3698              38         3812
3  customer_billing_address           3360              70         3780
4                date_issue           3589              64         3786
5        amount_total_gross           3469             180         3670
6                amount_due           3581              93         3757


In [62]:
def build_field_token_dataset(field_name):

    # Accepted annotations for this field
    accepted = full_quality_df[
        (full_quality_df["field"] == field_name) &
        (full_quality_df["similarity"] >= 0.80)
    ].copy()

    # Documents with uncertain annotations for this field
    annotated_docs = set(
        full_quality_df[
            full_quality_df["field"] == field_name
        ]["document_id"]
    )

    accepted_docs = set(
        accepted["document_id"]
    )

    uncertain_docs = annotated_docs - accepted_docs

    usable_docs = [
        doc_id
        for doc_id in invoice_doc_ids
        if doc_id not in uncertain_docs
    ]

    results = []

    with zipfile.ZipFile(docile_zip, "r") as z:

        for count, doc_id in enumerate(usable_docs, start=1):

            # -----------------------------
            # Load OCR
            # -----------------------------
            ocr_data = json.loads(
                z.read(f"ocr/{doc_id}.json")
            )

            words = []

            for page_idx, page in enumerate(ocr_data["pages"]):

                for block_idx, block in enumerate(page["blocks"]):

                    for line_idx, line in enumerate(block["lines"]):

                        for word_idx, word in enumerate(line["words"]):

                            geometry = word["geometry"]

                            words.append({
                                "document_id": doc_id,
                                "page": page_idx,
                                "block_id": block_idx,
                                "line_id": line_idx,
                                "word_id": word_idx,
                                "text": word["value"],
                                "confidence": word["confidence"],
                                "x1": geometry[0][0],
                                "y1": geometry[0][1],
                                "x2": geometry[1][0],
                                "y2": geometry[1][1],
                                "label": "O"
                            })

            words_df = pd.DataFrame(words)

            # -----------------------------
            # Accepted annotations
            # -----------------------------
            annotations = accepted[
                accepted["document_id"] == doc_id
            ]

            # -----------------------------
            # Match OCR words to annotations
            # -----------------------------
            for _, annotation in annotations.iterrows():

                matched_indices = []

                for j in range(len(words_df)):

                    word = words_df.iloc[j]

                    if word["page"] != annotation["page"]:
                        continue

                    if word_matches_annotation(
                        word,
                        annotation["bbox"]
                    ):
                        matched_indices.append(j)

                # Reading order
                matched_indices = sorted(
                    matched_indices,
                    key=lambda idx: (
                        words_df.loc[idx, "block_id"],
                        words_df.loc[idx, "line_id"],
                        words_df.loc[idx, "word_id"]
                    )
                )

                # BIO labels
                for position, index in enumerate(matched_indices):

                    if position == 0:
                        words_df.loc[index, "label"] = (
                            f"B-{field_name.upper()}"
                        )
                    else:
                        words_df.loc[index, "label"] = (
                            f"I-{field_name.upper()}"
                        )

            results.extend(
                words_df.to_dict("records")
            )

            if count % 500 == 0:
                print(
                    f"{field_name}: processed {count}/{len(usable_docs)} documents"
                )

    return pd.DataFrame(results)

In [63]:
vendor_full_df = build_field_token_dataset("vendor_name")

print("\nShape:", vendor_full_df.shape)

print("\nLabels:")
print(
    vendor_full_df["label"].value_counts()
)

print("\nDocuments:")
print(
    vendor_full_df["document_id"].nunique()
)

vendor_name: processed 500/3805 documents
vendor_name: processed 1000/3805 documents
vendor_name: processed 1500/3805 documents
vendor_name: processed 2000/3805 documents
vendor_name: processed 2500/3805 documents
vendor_name: processed 3000/3805 documents
vendor_name: processed 3500/3805 documents

Shape: (932700, 12)

Labels:
label
O                916230
I-VENDOR_NAME     11403
B-VENDOR_NAME      5067
Name: count, dtype: int64

Documents:
3805


In [64]:
label2id_layout = {
    "O": 0,
    "B-VENDOR_NAME": 1,
    "I-VENDOR_NAME": 2
}

id2label_layout = {
    0: "O",
    1: "B-VENDOR_NAME",
    2: "I-VENDOR_NAME"
}

sample_doc_id = vendor_full_df["document_id"].iloc[0]

sample_df = vendor_full_df[
    vendor_full_df["document_id"] == sample_doc_id
].copy()

sample_df = sample_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
)

words = sample_df["text"].tolist()
boxes = []

for _, row in sample_df.iterrows():
    boxes.append([
        int(row["x1"] * 1000),
        int(row["y1"] * 1000),
        int(row["x2"] * 1000),
        int(row["y2"] * 1000)
    ])

word_label_ids = [
    label2id_layout[label]
    for label in sample_df["label"]
]

encoded = layout_tokenizer(
    words,
    boxes=boxes,
    is_split_into_words=True,
    truncation=True,
    max_length=512,
    return_attention_mask=True
)

token_word_ids = encoded.word_ids()

token_labels = []
previous_word_id = None

for word_id in token_word_ids:

    if word_id is None:
        token_labels.append(-100)
        previous_word_id = None
        continue

    word_label = word_label_ids[word_id]

    if word_id == previous_word_id:

        if word_label == label2id_layout["B-VENDOR_NAME"]:
            token_labels.append(
                label2id_layout["I-VENDOR_NAME"]
            )
        else:
            token_labels.append(word_label)

    else:
        token_labels.append(word_label)

    previous_word_id = word_id

print("Document:", sample_doc_id)
print("OCR words:", len(words))
print("LayoutLM tokens:", len(encoded["input_ids"]))

print("\nFirst vendor tokens:")

for i in range(len(token_labels)):

    if token_labels[i] != -100 and token_labels[i] != 0:

        token = layout_tokenizer.convert_ids_to_tokens(
            [encoded["input_ids"][i]]
        )[0]

        print(
            token,
            "→",
            id2label_layout[token_labels[i]]
        )

Document: 00134dd365a24343b35b78c6
OCR words: 175
LayoutLM tokens: 377

First vendor tokens:
bio → B-VENDOR_NAME
##rel → I-VENDOR_NAME
##iance → I-VENDOR_NAME
testing → I-VENDOR_NAME
& → I-VENDOR_NAME
development → I-VENDOR_NAME
, → I-VENDOR_NAME
inc → I-VENDOR_NAME
. → I-VENDOR_NAME
bio → B-VENDOR_NAME
##rel → I-VENDOR_NAME
##iance → I-VENDOR_NAME
testing → I-VENDOR_NAME
& → I-VENDOR_NAME
development → I-VENDOR_NAME
, → I-VENDOR_NAME
inc → I-VENDOR_NAME
. → I-VENDOR_NAME
bio → B-VENDOR_NAME
##rel → I-VENDOR_NAME
##iance → I-VENDOR_NAME
& → I-VENDOR_NAME
testing → I-VENDOR_NAME
development → I-VENDOR_NAME
, → I-VENDOR_NAME


In [65]:
def prepare_layoutlm_examples(
    words_df,
    tokenizer,
    label2id,
    max_length=512,
    stride=128
):
    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    words = words_df["text"].tolist()

    boxes = []

    for _, row in words_df.iterrows():
        boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    word_label_ids = [
        label2id[label]
        for label in words_df["label"]
    ]

    encoded = tokenizer(
        words,
        boxes=boxes,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True
    )

    examples = []

    for chunk_idx in range(len(encoded["input_ids"])):

        token_word_ids = encoded.word_ids(
            batch_index=chunk_idx
        )

        token_labels = []
        token_boxes = []

        previous_word_id = None

        for word_id in token_word_ids:

            if word_id is None:
                token_labels.append(-100)
                token_boxes.append([0, 0, 0, 0])
                previous_word_id = None
                continue

            word_label = word_label_ids[word_id]

            if word_id == previous_word_id:

                if word_label == label2id["B-VENDOR_NAME"]:
                    token_labels.append(
                        label2id["I-VENDOR_NAME"]
                    )
                else:
                    token_labels.append(word_label)

            else:
                token_labels.append(word_label)

            token_boxes.append(boxes[word_id])
            previous_word_id = word_id

        examples.append({
            "input_ids": encoded["input_ids"][chunk_idx],
            "attention_mask": encoded["attention_mask"][chunk_idx],
            "bbox": token_boxes,
            "labels": token_labels
        })

    return examples

In [66]:
sample_examples = prepare_layoutlm_examples(
    sample_df,
    layout_tokenizer,
    label2id_layout
)

print("Chunks created:", len(sample_examples))

print("\nFirst chunk:")
print("Tokens:", len(sample_examples[0]["input_ids"]))
print("Boxes:", len(sample_examples[0]["bbox"]))
print("Labels:", len(sample_examples[0]["labels"]))

Chunks created: 1

First chunk:
Tokens: 377
Boxes: 377
Labels: 377


In [67]:
from datasets import Dataset

# Keep only documents belonging to the metadata-training split
train_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_train_ids)
].copy()


def generate_layoutlm_examples(split_df):
    for doc_id, words_df in split_df.groupby(
        "document_id",
        sort=False
    ):

        examples = prepare_layoutlm_examples(
            words_df,
            layout_tokenizer,
            label2id_layout
        )

        for example in examples:
            yield example


train_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(train_vendor_df)
)

print(train_vendor_hf)

Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 4482
})


In [68]:
from collections import Counter

label_counts = Counter()

for example in train_vendor_hf:
    label_counts.update(example["labels"])

print("Training token labels:")

for label_id, count in sorted(label_counts.items()):
    label_name = id2label_layout.get(label_id, "IGNORED")
    print(label_id, label_name, count)

Training token labels:
-100 IGNORED 8964
0 O 1662906
1 B-VENDOR_NAME 3693
2 I-VENDOR_NAME 17955


In [69]:
val_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_val_ids)
].copy()


val_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(val_vendor_df)
)

print(val_vendor_hf)

Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 944
})


In [70]:
label_counts = Counter()

for example in val_vendor_hf:
    label_counts.update(example["labels"])

print("Validation token labels:")

for label_id, count in sorted(label_counts.items()):
    label_name = id2label_layout.get(label_id, "IGNORED")
    print(label_id, label_name, count)

Validation token labels:
-100 IGNORED 1888
0 O 348655
1 B-VENDOR_NAME 806
2 I-VENDOR_NAME 4068


In [71]:
test_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_test_ids)
].copy()

test_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(test_vendor_df)
)

print(test_vendor_hf)

Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 919
})


In [72]:
label_counts = Counter()

for example in test_vendor_hf:
    label_counts.update(example["labels"])

print("Test token labels:")

for label_id, count in sorted(label_counts.items()):
    label_name = id2label_layout.get(label_id, "IGNORED")
    print(label_id, label_name, count)

Test token labels:
-100 IGNORED 1838
0 O 335287
1 B-VENDOR_NAME 816
2 I-VENDOR_NAME 4078


In [74]:
from transformers import LayoutLMForTokenClassification

layout_model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=3,
    label2id=label2id_layout,
    id2label=id2label_layout
)

layout_model.to(device)

print("LayoutLM model loaded")
print("Device:", next(layout_model.parameters()).device)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] LayoutLMForTokenClassification LOAD REPORT from: microsoft/layoutlm-base-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


NameError: name 'device' is not defined

In [75]:
import torch
from transformers import LayoutLMForTokenClassification

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

layout_model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=3,
    label2id=label2id_layout,
    id2label=id2label_layout
)

layout_model.to(device)

print("LayoutLM model loaded")
print("Model device:", next(layout_model.parameters()).device)

Using device: cuda


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] LayoutLMForTokenClassification LOAD REPORT from: microsoft/layoutlm-base-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LayoutLM model loaded
Model device: cuda:0


In [76]:
import torch

from transformers import LayoutLMForTokenClassification

# Label mapping
label2id_layout = {
    "O": 0,
    "B-VENDOR_NAME": 1,
    "I-VENDOR_NAME": 2
}

id2label_layout = {
    0: "O",
    1: "B-VENDOR_NAME",
    2: "I-VENDOR_NAME"
}

# Select device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using device:", device)

# Load LayoutLM
layout_model = LayoutLMForTokenClassification.from_pretrained(
    "microsoft/layoutlm-base-uncased",
    num_labels=3,
    label2id=label2id_layout,
    id2label=id2label_layout
)

layout_model.to(device)

print("LayoutLM model loaded")
print("Model device:", next(layout_model.parameters()).device)

Using device: cuda


Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] LayoutLMForTokenClassification LOAD REPORT from: microsoft/layoutlm-base-uncased
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


LayoutLM model loaded
Model device: cuda:0


In [77]:
import torch
import torch.nn as nn
from transformers import Trainer


class WeightedTrainer(Trainer):

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None
    ):
        labels = inputs.pop("labels")

        outputs = model(**inputs)

        logits = outputs.logits

        # Moderate class weights.
        # O is given the lowest weight because it dominates.
        class_weights = torch.tensor(
            [1.0, 8.0, 4.0],
            dtype=torch.float32,
            device=logits.device
        )

        loss_function = nn.CrossEntropyLoss(
            weight=class_weights,
            ignore_index=-100
        )

        loss = loss_function(
            logits.view(-1, 3),
            labels.view(-1)
        )

        return (
            (loss, outputs)
            if return_outputs
            else loss
        )


print("WeightedTrainer ready")

WeightedTrainer ready


In [78]:
from transformers import TrainingArguments

layout_training_args = TrainingArguments(
    output_dir="../models/layoutlm_vendor_name",

    num_train_epochs=5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=3e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_steps=100,

    report_to="none"
)

print("LayoutLM training configuration ready")

LayoutLM training configuration ready


In [79]:
layout_trainer = WeightedTrainer(
    model=layout_model,
    args=layout_training_args,

    train_dataset=train_vendor_hf,
    eval_dataset=val_vendor_hf,

    processing_class=layout_tokenizer,

    compute_metrics=compute_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("LayoutLM trainer ready")

NameError: name 'compute_metrics' is not defined

In [80]:
from datasets import Dataset

# Recreate split DataFrames
train_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_train_ids)
].copy()

val_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_val_ids)
].copy()

test_vendor_df = vendor_model_df[
    vendor_model_df["document_id"].isin(meta_test_ids)
].copy()


def generate_layoutlm_examples(split_df):
    for doc_id, words_df in split_df.groupby(
        "document_id",
        sort=False
    ):

        examples = prepare_layoutlm_examples(
            words_df,
            layout_tokenizer,
            label2id_layout
        )

        for example in examples:
            yield example


train_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(train_vendor_df)
)

val_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(val_vendor_df)
)

test_vendor_hf = Dataset.from_generator(
    lambda: generate_layoutlm_examples(test_vendor_df)
)

print("Train:", train_vendor_hf)
print("Validation:", val_vendor_hf)
print("Test:", test_vendor_hf)

Train: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 4482
})
Validation: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 944
})
Test: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 919
})


In [81]:
full_quality_df.to_csv(
    "../processed/invoice_field_quality.csv",
    index=False
)

vendor_model_df.to_csv(
    "../processed/vendor_model_tokens.csv",
    index=False
)

print("Metadata datasets saved.")

Metadata datasets saved.


In [82]:
from pathlib import Path

layout_data_dir = Path("../processed/layoutlm_vendor_name")
layout_data_dir.mkdir(parents=True, exist_ok=True)

train_vendor_hf.save_to_disk(
    layout_data_dir / "train"
)

val_vendor_hf.save_to_disk(
    layout_data_dir / "validation"
)

test_vendor_hf.save_to_disk(
    layout_data_dir / "test"
)

print("LayoutLM datasets saved successfully.")

Saving the dataset (0/1 shards):   0%|          | 0/4482 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/944 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/919 [00:00<?, ? examples/s]

LayoutLM datasets saved successfully.


In [83]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

def compute_layout_metrics(eval_pred):

    predictions, labels = eval_pred

    predictions = np.argmax(predictions, axis=-1)

    # Ignore special/padding tokens marked as -100
    true_labels = []
    true_predictions = []

    for i in range(len(labels)):

        for j in range(len(labels[i])):

            if labels[i][j] == -100:
                continue

            true_labels.append(labels[i][j])
            true_predictions.append(predictions[i][j])

    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels,
        true_predictions,
        labels=[1, 2],
        average="macro",
        zero_division=0
    )

    return {
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

print("LayoutLM metrics ready")

LayoutLM metrics ready


In [84]:
layout_training_args = TrainingArguments(
    output_dir="../models/layoutlm_vendor_name",

    num_train_epochs=5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=3e-5,
    weight_decay=0.01,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",
    greater_is_better=False,

    fp16=True,

    logging_steps=100,

    report_to="none"
)

print("LayoutLM training configuration ready")

LayoutLM training configuration ready


In [85]:
layout_trainer = WeightedTrainer(
    model=layout_model,
    args=layout_training_args,

    train_dataset=train_vendor_hf,
    eval_dataset=val_vendor_hf,

    processing_class=layout_tokenizer,

    compute_metrics=compute_layout_metrics,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("LayoutLM trainer ready")

NameError: name 'EarlyStoppingCallback' is not defined

In [86]:
from transformers import EarlyStoppingCallback

print("EarlyStoppingCallback imported")

EarlyStoppingCallback imported


In [87]:
layout_trainer = WeightedTrainer(
    model=layout_model,
    args=layout_training_args,
    train_dataset=train_vendor_hf,
    eval_dataset=val_vendor_hf,
    processing_class=layout_tokenizer,
    compute_metrics=compute_layout_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("LayoutLM trainer ready")

LayoutLM trainer ready


In [88]:
print("Starting LayoutLM training...")

layout_train_output = layout_trainer.train()

print("\nTraining completed.")
print(layout_train_output)

Starting LayoutLM training...


ValueError: Unable to create tensor, you should probably activate truncation and/or padding with 'padding=True' 'truncation=True' to have batched tensors with the same length. Perhaps your features (`bbox` in this case) have excessive nesting (inputs type `list` where type `int` is expected).

In [89]:
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from torch.nn.utils.rnn import pad_sequence


@dataclass
class LayoutLMDataCollator:
    tokenizer: Any
    label_pad_token_id: int = -100
    bbox_pad_value: int = 0

    def __call__(self, features: List[Dict[str, Any]]):

        input_ids = [
            torch.tensor(feature["input_ids"], dtype=torch.long)
            for feature in features
        ]

        attention_masks = [
            torch.tensor(feature["attention_mask"], dtype=torch.long)
            for feature in features
        ]

        bboxes = [
            torch.tensor(feature["bbox"], dtype=torch.long)
            for feature in features
        ]

        labels = [
            torch.tensor(feature["labels"], dtype=torch.long)
            for feature in features
        ]

        # Pad input IDs
        input_ids = pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        )

        # Pad attention masks
        attention_masks = pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        )

        # Pad labels
        labels = pad_sequence(
            labels,
            batch_first=True,
            padding_value=self.label_pad_token_id
        )

        # Pad bounding boxes to [0, 0, 0, 0]
        max_length = input_ids.size(1)

        padded_bboxes = []

        for bbox in bboxes:

            padding_length = max_length - bbox.size(0)

            if padding_length > 0:
                padding = torch.zeros(
                    (padding_length, 4),
                    dtype=torch.long
                )

                bbox = torch.cat(
                    [bbox, padding],
                    dim=0
                )

            padded_bboxes.append(bbox)

        bboxes = torch.stack(padded_bboxes)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks,
            "bbox": bboxes,
            "labels": labels
        }


layout_data_collator = LayoutLMDataCollator(
    tokenizer=layout_tokenizer
)

print("Custom LayoutLM data collator ready")

Custom LayoutLM data collator ready


In [90]:
test_batch = layout_data_collator([
    train_vendor_hf[0],
    train_vendor_hf[1]
])

for key, value in test_batch.items():
    print(key, value.shape)

input_ids torch.Size([2, 512])
attention_mask torch.Size([2, 512])
bbox torch.Size([2, 512, 4])
labels torch.Size([2, 512])


In [91]:
data_collator=layout_data_collator

In [92]:
layout_trainer = WeightedTrainer(
    model=layout_model,
    args=layout_training_args,
    train_dataset=train_vendor_hf,
    eval_dataset=val_vendor_hf,
    data_collator=layout_data_collator,
    processing_class=layout_tokenizer,
    compute_metrics=compute_layout_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2
        )
    ]
)

print("LayoutLM trainer ready with custom collator")

LayoutLM trainer ready with custom collator


In [93]:
print("Starting LayoutLM vendor-name training...")

layout_train_output = layout_trainer.train()

print("\nTraining completed.")
print(layout_train_output)


Starting LayoutLM vendor-name training...


Epoch,Training Loss,Validation Loss,Precision,Recall,Macro F1
1,0.071728,0.029601,0.737947,0.920410,0.819140
2,0.031940,0.029134,0.800373,0.911069,0.851965
3,0.023344,0.036431,0.824341,0.902413,0.861516
4,0.017500,0.041815,0.824574,0.905380,0.863062


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training completed.
TrainOutput(global_step=2244, training_loss=0.05087854733122861, metrics={'train_runtime': 7275.1089, 'train_samples_per_second': 3.08, 'train_steps_per_second': 0.386, 'total_flos': 4593092821481808.0, 'train_loss': 0.05087854733122861, 'epoch': 4.0})


In [94]:
print("Best checkpoint:")
print(layout_trainer.state.best_model_checkpoint)

print("\nBest validation metric:")
print(layout_trainer.state.best_metric)

Best checkpoint:
../models/layoutlm_vendor_name\checkpoint-1122

Best validation metric:
0.029133543372154236


In [95]:
test_output = layout_trainer.predict(
    test_vendor_hf
)

print("Test prediction completed")
print(test_output.metrics)

Test prediction completed
{'test_loss': 0.029321927577257156, 'test_precision': 0.7711125457284269, 'test_recall': 0.9113704334112263, 'test_macro_f1': 0.8351211138179568, 'test_runtime': 77.2216, 'test_samples_per_second': 11.901, 'test_steps_per_second': 2.978}


In [96]:
%pip install seqeval

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Installing backend dependencies: started
  Installing backend dependencies: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16282 sha256=6fcfe20cbdb03219990df424602e3026319c99570623606ea671e8af5f4015da
  Stored in directory: c:\users\sudee\appdata\local\pip\cache\wheels\5f\b8\73\0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval
Note: you may need to restart the kernel to use updated packages.


In [97]:
from seqeval.metrics import precision_score, recall_score, f1_score

print("seqeval ready")

seqeval ready


In [98]:
import numpy as np
from seqeval.metrics import (
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Predictions from the best checkpoint
predictions = test_output.predictions
predicted_ids = np.argmax(predictions, axis=-1)

true_sequences = []
predicted_sequences = []

for i in range(len(test_vendor_hf)):

    true_labels = []
    pred_labels = []

    labels_row = test_vendor_hf[i]["labels"]

    for j in range(len(labels_row)):

        true_id = labels_row[j]

        # Ignore special/padding tokens
        if true_id == -100:
            continue

        pred_id = predicted_ids[i][j]

        true_labels.append(
            id2label_layout[true_id]
        )

        pred_labels.append(
            id2label_layout[pred_id]
        )

    true_sequences.append(true_labels)
    predicted_sequences.append(pred_labels)


print("Entity-level Precision:",
      precision_score(
          true_sequences,
          predicted_sequences,
          zero_division=0
      ))

print("Entity-level Recall:",
      recall_score(
          true_sequences,
          predicted_sequences,
          zero_division=0
      ))

print("Entity-level F1:",
      f1_score(
          true_sequences,
          predicted_sequences,
          zero_division=0
      ))

print("\nDetailed report:")
print(
    classification_report(
        true_sequences,
        predicted_sequences,
        zero_division=0
    )
)

Entity-level Precision: 0.6446280991735537
Entity-level Recall: 0.8347205707491082
Entity-level F1: 0.7274611398963731

Detailed report:
              precision    recall  f1-score   support

 VENDOR_NAME       0.64      0.83      0.73       841

   micro avg       0.64      0.83      0.73       841
   macro avg       0.64      0.83      0.73       841
weighted avg       0.64      0.83      0.73       841



In [99]:
import numpy as np

predicted_ids = np.argmax(
    test_output.predictions,
    axis=-1
)


def extract_entities(tokens, label_ids):

    entities = []
    current_tokens = []

    for token, label_id in zip(tokens, label_ids):

        if label_id == -100:
            continue

        label = id2label_layout[label_id]

        if label == "B-VENDOR_NAME":

            if current_tokens:
                entities.append(
                    layout_tokenizer.convert_tokens_to_string(
                        current_tokens
                    )
                )

            current_tokens = [token]

        elif label == "I-VENDOR_NAME":

            if current_tokens:
                current_tokens.append(token)
            else:
                current_tokens = [token]

        else:

            if current_tokens:
                entities.append(
                    layout_tokenizer.convert_tokens_to_string(
                        current_tokens
                    )
                )
                current_tokens = []

    if current_tokens:
        entities.append(
            layout_tokenizer.convert_tokens_to_string(
                current_tokens
            )
        )

    return entities


error_count = 0

for i in range(len(test_vendor_hf)):

    tokens = layout_tokenizer.convert_ids_to_tokens(
        test_vendor_hf[i]["input_ids"]
    )

    true_labels = test_vendor_hf[i]["labels"]
    pred_labels = predicted_ids[i]

    true_entities = extract_entities(
        tokens,
        true_labels
    )

    predicted_entities = extract_entities(
        tokens,
        pred_labels
    )

    if true_entities != predicted_entities:

        print("\n" + "=" * 80)
        print("Chunk:", i)

        print("True entities:")
        print(true_entities)

        print("\nPredicted entities:")
        print(predicted_entities)

        error_count += 1

        if error_count >= 10:
            break


Chunk: 0
True entities:
['rothstein - lauber, incorporated']

Predicted entities:
['rothstein - lauber, incorporated market research services']

Chunk: 3
True entities:
['bozell worldwide inc.']

Predicted entities:
['bozell', '.', 'bozell worldwide inc.']

Chunk: 5
True entities:
['video monitoring services of america, lpi']

Predicted entities:
['video monitoring services of america, lp', 'video monitoring services of america, lpi']

Chunk: 6
True entities:
['sociery for riek analysis']

Predicted entities:
['##iery for']

Chunk: 7
True entities:
['virginia commonwealth university']

Predicted entities:
['virginia commonwealth university', 'of pharmaceutics']

Chunk: 11
True entities:
['thomas jcfferson university']

Predicted entities:
['thomas jefferson', 'thomas jcfferson university']

Chunk: 14
True entities:
['pontiac! silverdome']

Predicted entities:
['pontiac silverdome', 'pontiac! silverdome', 'pontiac stadium authority']

Chunk: 17
True entities:
[]

Predicted entities:
['

In [100]:
import numpy as np
from collections import defaultdict
from seqeval.metrics import precision_score, recall_score, f1_score


def build_eval_chunks(split_df):
    """
    Create LayoutLM chunks while keeping the mapping from
    each token back to the original OCR word position.
    """

    examples = []
    mappings = []

    for doc_id, words_df in split_df.groupby(
        "document_id",
        sort=False
    ):

        words_df = words_df.sort_values(
            ["page", "block_id", "line_id", "word_id"]
        ).reset_index(drop=True)

        words = words_df["text"].tolist()

        boxes = []

        for _, row in words_df.iterrows():
            boxes.append([
                int(row["x1"] * 1000),
                int(row["y1"] * 1000),
                int(row["x2"] * 1000),
                int(row["y2"] * 1000)
            ])

        word_label_ids = [
            label2id_layout[label]
            for label in words_df["label"]
        ]

        encoded = layout_tokenizer(
            words,
            boxes=boxes,
            is_split_into_words=True,
            truncation=True,
            max_length=512,
            stride=128,
            return_overflowing_tokens=True,
            return_attention_mask=True
        )

        for chunk_idx in range(len(encoded["input_ids"])):

            word_ids = encoded.word_ids(
                batch_index=chunk_idx
            )

            token_labels = []
            token_boxes = []
            token_word_positions = []

            previous_word_id = None

            for word_id in word_ids:

                if word_id is None:
                    token_labels.append(-100)
                    token_boxes.append([0, 0, 0, 0])
                    token_word_positions.append(None)
                    previous_word_id = None
                    continue

                word_label = word_label_ids[word_id]

                if word_id == previous_word_id:
                    if word_label == label2id_layout["B-VENDOR_NAME"]:
                        token_labels.append(
                            label2id_layout["I-VENDOR_NAME"]
                        )
                    else:
                        token_labels.append(word_label)
                else:
                    token_labels.append(word_label)

                token_boxes.append(boxes[word_id])
                token_word_positions.append(word_id)

                previous_word_id = word_id

            examples.append({
                "input_ids": encoded["input_ids"][chunk_idx],
                "attention_mask": encoded["attention_mask"][chunk_idx],
                "bbox": token_boxes,
                "labels": token_labels
            })

            mappings.append({
                "document_id": doc_id,
                "word_positions": token_word_positions
            })

    return examples, mappings


test_eval_examples, test_mappings = build_eval_chunks(
    test_vendor_df
)

print("Evaluation chunks:", len(test_eval_examples))
print("Mappings:", len(test_mappings))

Evaluation chunks: 919
Mappings: 919


In [101]:
from datasets import Dataset
from collections import defaultdict
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score


# --------------------------------------------------
# 1. Convert evaluation examples to Hugging Face Dataset
# --------------------------------------------------

test_eval_hf = Dataset.from_list(test_eval_examples)

print("Test evaluation dataset:", test_eval_hf)


# --------------------------------------------------
# 2. Predict all chunks with the best LayoutLM model
# --------------------------------------------------

eval_output = layout_trainer.predict(test_eval_hf)

logits = eval_output.predictions

print("Prediction shape:", logits.shape)


# --------------------------------------------------
# 3. Collect logits for each ORIGINAL OCR word
#
# A word may appear:
# - in multiple subword tokens
# - in overlapping chunks
#
# We average all of those logits.
# --------------------------------------------------

word_logits = defaultdict(list)

for chunk_idx in range(len(test_mappings)):

    document_id = test_mappings[chunk_idx]["document_id"]

    word_positions = test_mappings[chunk_idx]["word_positions"]

    for token_idx, word_pos in enumerate(word_positions):

        if word_pos is None:
            continue

        key = (
            document_id,
            word_pos
        )

        word_logits[key].append(
            logits[chunk_idx][token_idx]
        )


# --------------------------------------------------
# 4. Create one prediction per OCR word
# --------------------------------------------------

document_predictions = defaultdict(dict)

for key, token_logits in word_logits.items():

    document_id, word_pos = key

    averaged_logits = np.mean(
        token_logits,
        axis=0
    )

    predicted_label_id = int(
        np.argmax(averaged_logits)
    )

    document_predictions[
        document_id
    ][word_pos] = predicted_label_id


# --------------------------------------------------
# 5. Build TRUE and PREDICTED BIO sequences
# --------------------------------------------------

true_sequences = []
predicted_sequences = []

for document_id, words_df in test_vendor_df.groupby(
    "document_id",
    sort=False
):

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    true_labels = []
    pred_labels = []

    for word_pos in range(len(words_df)):

        true_label = words_df.loc[
            word_pos,
            "label"
        ]

        predicted_id = document_predictions[
            document_id
        ].get(
            word_pos,
            label2id_layout["O"]
        )

        predicted_label = id2label_layout[
            predicted_id
        ]

        true_labels.append(
            true_label
        )

        pred_labels.append(
            predicted_label
        )

    true_sequences.append(true_labels)
    predicted_sequences.append(pred_labels)


# --------------------------------------------------
# 6. Document-level entity evaluation
# --------------------------------------------------

document_precision = precision_score(
    true_sequences,
    predicted_sequences,
    zero_division=0
)

document_recall = recall_score(
    true_sequences,
    predicted_sequences,
    zero_division=0
)

document_f1 = f1_score(
    true_sequences,
    predicted_sequences,
    zero_division=0
)

print("\n==========================================")
print("DOCUMENT-LEVEL VENDOR NAME EVALUATION")
print("==========================================")

print("Precision:", document_precision)
print("Recall   :", document_recall)
print("F1       :", document_f1)

Test evaluation dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 919
})


Prediction shape: (919, 512, 3)

DOCUMENT-LEVEL VENDOR NAME EVALUATION
Precision: 0.6927835051546392
Recall   : 0.8296296296296296
F1       : 0.7550561797752808


In [102]:
from scipy.special import softmax
from seqeval.metrics import f1_score

# -----------------------------------------
# 1. Build validation evaluation examples
# -----------------------------------------

val_eval_examples, val_mappings = build_eval_chunks(
    val_vendor_df
)

val_eval_hf = Dataset.from_list(
    val_eval_examples
)

print("Validation chunks:", len(val_eval_examples))


# -----------------------------------------
# 2. Get model predictions
# -----------------------------------------

val_output = layout_trainer.predict(
    val_eval_hf
)

val_probabilities = softmax(
    val_output.predictions,
    axis=-1
)


# -----------------------------------------
# 3. Aggregate subwords/chunks back
#    to original OCR words
# -----------------------------------------

val_word_probs = defaultdict(list)

for chunk_idx in range(len(val_mappings)):

    document_id = val_mappings[
        chunk_idx
    ]["document_id"]

    word_positions = val_mappings[
        chunk_idx
    ]["word_positions"]

    for token_idx, word_pos in enumerate(word_positions):

        if word_pos is None:
            continue

        key = (
            document_id,
            word_pos
        )

        val_word_probs[key].append(
            val_probabilities[
                chunk_idx
            ][token_idx]
        )


# -----------------------------------------
# 4. Average overlapping predictions
# -----------------------------------------

val_document_probs = defaultdict(dict)

for key, probabilities in val_word_probs.items():

    document_id, word_pos = key

    val_document_probs[
        document_id
    ][word_pos] = np.mean(
        probabilities,
        axis=0
    )


# -----------------------------------------
# 5. Build true BIO sequences
# -----------------------------------------

val_true_sequences = []
val_probability_sequences = []

for document_id, words_df in val_vendor_df.groupby(
    "document_id",
    sort=False
):

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    true_labels = []
    probability_sequence = []

    for word_pos in range(len(words_df)):

        true_labels.append(
            words_df.loc[
                word_pos,
                "label"
            ]
        )

        probability_sequence.append(
            val_document_probs[
                document_id
            ].get(
                word_pos,
                np.array([1.0, 0.0, 0.0])
            )
        )

    val_true_sequences.append(
        true_labels
    )

    val_probability_sequences.append(
        probability_sequence
    )


# -----------------------------------------
# 6. Try different thresholds
# -----------------------------------------

threshold_results = []

thresholds = [
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90
]

for threshold in thresholds:

    predicted_sequences = []

    for probability_sequence in val_probability_sequences:

        predicted_labels = []

        for probabilities in probability_sequence:

            p_non_o = (
                probabilities[1] +
                probabilities[2]
            )

            if p_non_o < threshold:

                predicted_labels.append("O")

            else:

                if probabilities[1] >= probabilities[2]:
                    predicted_labels.append(
                        "B-VENDOR_NAME"
                    )
                else:
                    predicted_labels.append(
                        "I-VENDOR_NAME"
                    )

        predicted_sequences.append(
            predicted_labels
        )

    f1 = f1_score(
        val_true_sequences,
        predicted_sequences,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "f1": f1
    })


threshold_df = pd.DataFrame(
    threshold_results
)

print(threshold_df)

Validation chunks: 944


   threshold        f1
0        0.3  0.699721
1        0.4  0.730086
2        0.5  0.743888
3        0.6  0.749705
4        0.7  0.768862
5        0.8  0.776284
6        0.9  0.777778


In [103]:
from seqeval.metrics import precision_score, recall_score, f1_score

selected_threshold = 0.90

predicted_sequences = []

for probability_sequence in val_probability_sequences:

    predicted_labels = []

    for probabilities in probability_sequence:

        p_non_o = probabilities[1] + probabilities[2]

        if p_non_o < selected_threshold:
            predicted_labels.append("O")

        elif probabilities[1] >= probabilities[2]:
            predicted_labels.append("B-VENDOR_NAME")

        else:
            predicted_labels.append("I-VENDOR_NAME")

    predicted_sequences.append(predicted_labels)


print("Threshold:", selected_threshold)

print(
    "Precision:",
    precision_score(
        val_true_sequences,
        predicted_sequences,
        zero_division=0
    )
)

print(
    "Recall:",
    recall_score(
        val_true_sequences,
        predicted_sequences,
        zero_division=0
    )
)

print(
    "F1:",
    f1_score(
        val_true_sequences,
        predicted_sequences,
        zero_division=0
    )
)

Threshold: 0.9
Precision: 0.7634803921568627
Recall: 0.7926208651399491
F1: 0.7777777777777777


In [104]:
from collections import defaultdict
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score

selected_threshold = 0.90

# --------------------------------------------------
# 1. Convert test logits to probabilities
# --------------------------------------------------

test_probabilities = softmax(
    test_output.predictions,
    axis=-1
)

# --------------------------------------------------
# 2. Aggregate overlapping chunks back to OCR words
# --------------------------------------------------

test_word_probs = defaultdict(list)

for chunk_idx in range(len(test_mappings)):

    document_id = test_mappings[chunk_idx]["document_id"]
    word_positions = test_mappings[chunk_idx]["word_positions"]

    for token_idx, word_pos in enumerate(word_positions):

        if word_pos is None:
            continue

        key = (
            document_id,
            word_pos
        )

        test_word_probs[key].append(
            test_probabilities[chunk_idx][token_idx]
        )

# --------------------------------------------------
# 3. Average probabilities for each OCR word
# --------------------------------------------------

test_document_probs = defaultdict(dict)

for key, probabilities in test_word_probs.items():

    document_id, word_pos = key

    test_document_probs[document_id][word_pos] = np.mean(
        probabilities,
        axis=0
    )

# --------------------------------------------------
# 4. Create true/predicted BIO sequences
# --------------------------------------------------

test_true_sequences = []
test_pred_sequences = []

for document_id, words_df in test_vendor_df.groupby(
    "document_id",
    sort=False
):

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    true_labels = []
    predicted_labels = []

    for word_pos in range(len(words_df)):

        true_label = words_df.loc[
            word_pos,
            "label"
        ]

        probabilities = test_document_probs[
            document_id
        ].get(
            word_pos,
            np.array([1.0, 0.0, 0.0])
        )

        p_non_o = probabilities[1] + probabilities[2]

        if p_non_o < selected_threshold:

            predicted_label = "O"

        elif probabilities[1] >= probabilities[2]:

            predicted_label = "B-VENDOR_NAME"

        else:

            predicted_label = "I-VENDOR_NAME"

        true_labels.append(true_label)
        predicted_labels.append(predicted_label)

    test_true_sequences.append(true_labels)
    test_pred_sequences.append(predicted_labels)

# --------------------------------------------------
# 5. Final test metrics
# --------------------------------------------------

test_precision = precision_score(
    test_true_sequences,
    test_pred_sequences,
    zero_division=0
)

test_recall = recall_score(
    test_true_sequences,
    test_pred_sequences,
    zero_division=0
)

test_f1 = f1_score(
    test_true_sequences,
    test_pred_sequences,
    zero_division=0
)

print("==========================================")
print("FINAL VENDOR_NAME TEST RESULT")
print("==========================================")
print("Threshold :", selected_threshold)
print("Precision :", test_precision)
print("Recall    :", test_recall)
print("F1        :", test_f1)

FINAL VENDOR_NAME TEST RESULT
Threshold : 0.9
Precision : 0.7494279176201373
Recall    : 0.808641975308642
F1        : 0.7779097387173397


In [105]:
def extract_entity_strings(tokens, labels):
    entities = []
    current = []

    for token, label in zip(tokens, labels):

        if label == "B-VENDOR_NAME":

            if current:
                entities.append(
                    layout_tokenizer.convert_tokens_to_string(current)
                )

            current = [token]

        elif label == "I-VENDOR_NAME":

            if current:
                current.append(token)
            else:
                current = [token]

        else:

            if current:
                entities.append(
                    layout_tokenizer.convert_tokens_to_string(current)
                )
                current = []

    if current:
        entities.append(
            layout_tokenizer.convert_tokens_to_string(current)
        )

    return entities


error_count = 0

for doc_idx, document_id in enumerate(
    test_vendor_df["document_id"].drop_duplicates()
):

    words_df = test_vendor_df[
        test_vendor_df["document_id"] == document_id
    ].copy()

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    true_labels = test_true_sequences[doc_idx]
    pred_labels = test_pred_sequences[doc_idx]

    tokens = words_df["text"].tolist()

    true_entities = extract_entity_strings(
        tokens,
        true_labels
    )

    predicted_entities = extract_entity_strings(
        tokens,
        pred_labels
    )

    if true_entities != predicted_entities:

        print("\n" + "=" * 80)
        print("Document:", document_id)

        print("TRUE:")
        print(true_entities)

        print("PREDICTED:")
        print(predicted_entities)

        error_count += 1

        if error_count >= 15:
            break


Document: 023d32572143486587809f7a
TRUE:
['VIDEO MONITORING SERVICES of AMERICA, LPI']
PREDICTED:
['Video Monitoring Services of America, LP', 'VIDEO MONITORING SERVICES of AMERICA, LPI']

Document: 035fb5c9709e4d928a2e2088
TRUE:
['Sociery for Riek Analysis']
PREDICTED:
[]

Document: 062d94841d1649a5b5a4e720
TRUE:
['PONTIAC! SILVERDOME']
PREDICTED:
['Pontiac', 'PONTIAC! SILVERDOME']

Document: 085507cff07e4c30948f382d
TRUE:
['CHUM-Centre de recherche', 'HOTEL-DIEU (Siège social)', 'HOPITAL NOTRE-DAME', 'HOPITAL SAINT-LUC']
PREDICTED:
['CHUM-Centre de recherche', 'CENTRE HOSPITALIER DEI LUNIVERSITÉ DE MONTRÉAL HOTEL-DIEU', 'HOPITAL NOTRE-DAME', 'HOPITAL SAINT-LUC']

Document: 0998d3bad867466d950533ab
TRUE:
[]
PREDICTED:
['WHOLESALE']

Document: 0cb2ecb2228c469a99cc262a
TRUE:
['KEYW-FM', 'KEYW-FM TSQ Media Tri-Cities', 'KEYW-FM TSQ Media Tri-Cities']
PREDICTED:
['KEYW-FM', 'KEYW-FM TSQ', 'Tri-Cities', 'KEYW-FM TSQ', 'Tri-Cities']

Document: 0d279060c3834c46a50f97d2
TRUE:
['CUMULUS WILMI

In [106]:
def normalize_vendor(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text


def extract_vendor_candidates(
    tokens,
    labels,
    token_probabilities=None
):
    candidates = []
    current_tokens = []
    current_scores = []

    for i in range(len(tokens)):

        label = labels[i]

        if label == "B-VENDOR_NAME":

            if current_tokens:
                candidates.append({
                    "text": layout_tokenizer.convert_tokens_to_string(
                        current_tokens
                    ),
                    "score": (
                        np.mean(current_scores)
                        if current_scores
                        else 0
                    )
                })

            current_tokens = [tokens[i]]

            if token_probabilities is not None:
                current_scores = [
                    token_probabilities[i]
                ]
            else:
                current_scores = []

        elif label == "I-VENDOR_NAME":

            if current_tokens:
                current_tokens.append(tokens[i])

                if token_probabilities is not None:
                    current_scores.append(
                        token_probabilities[i]
                    )

        else:

            if current_tokens:
                candidates.append({
                    "text": layout_tokenizer.convert_tokens_to_string(
                        current_tokens
                    ),
                    "score": (
                        np.mean(current_scores)
                        if current_scores
                        else 0
                    )
                })

                current_tokens = []
                current_scores = []

    if current_tokens:
        candidates.append({
            "text": layout_tokenizer.convert_tokens_to_string(
                current_tokens
            ),
            "score": (
                np.mean(current_scores)
                if current_scores
                else 0
            )
        })

    return candidates

In [107]:
doc_id = "023d32572143486587809f7a"

words_df = test_vendor_df[
    test_vendor_df["document_id"] == doc_id
].copy()

words_df = words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

doc_index = list(
    test_vendor_df["document_id"].drop_duplicates()
).index(doc_id)

tokens = words_df["text"].tolist()

candidates = extract_vendor_candidates(
    tokens,
    test_true_sequences[doc_index]
)

print("True candidates:")
print(candidates)

True candidates:
[{'text': 'VIDEO MONITORING SERVICES of AMERICA, LPI', 'score': 0}]


In [108]:
doc_id = "023d32572143486587809f7a"

words_df = test_vendor_df[
    test_vendor_df["document_id"] == doc_id
].copy()

words_df = words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

doc_index = list(
    test_vendor_df["document_id"].drop_duplicates()
).index(doc_id)

tokens = words_df["text"].tolist()
pred_labels = test_pred_sequences[doc_index]

# Get probability for the predicted vendor/non-vendor decision
token_scores = []

for word_pos in range(len(words_df)):

    probabilities = test_document_probs[
        doc_id
    ].get(
        word_pos,
        np.array([1.0, 0.0, 0.0])
    )

    token_scores.append(
        probabilities[1] + probabilities[2]
    )

predicted_candidates = extract_vendor_candidates(
    tokens,
    pred_labels,
    token_scores
)

print("Predicted candidates:")
for candidate in predicted_candidates:
    print(candidate)

Predicted candidates:
{'text': 'Video Monitoring Services of America, LP', 'score': np.float32(0.995514)}
{'text': 'VIDEO MONITORING SERVICES of AMERICA, LPI', 'score': np.float32(0.99801475)}


In [109]:
from difflib import SequenceMatcher


def normalize_vendor_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]", "", text)
    return text


def get_predicted_candidates_for_document(
    document_id,
    words_df,
    pred_labels,
    probabilities
):
    tokens = words_df["text"].tolist()

    candidates = extract_vendor_candidates(
        tokens,
        pred_labels,
        probabilities
    )

    # Remove exact duplicate candidates
    unique = {}

    for candidate in candidates:

        key = normalize_vendor_text(
            candidate["text"]
        )

        if not key:
            continue

        if (
            key not in unique or
            candidate["score"] > unique[key]["score"]
        ):
            unique[key] = candidate

    candidates = list(unique.values())

    # Highest-confidence candidate first
    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return candidates


results = []

# Accepted ground-truth vendor annotations
vendor_gt = full_quality_df[
    (full_quality_df["field"] == "vendor_name") &
    (full_quality_df["similarity"] >= 0.80)
].copy()

for doc_idx, document_id in enumerate(
    val_vendor_df["document_id"].drop_duplicates()
):

    words_df = val_vendor_df[
        val_vendor_df["document_id"] == document_id
    ].copy()

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    pred_labels = []

    probability_list = []

    for word_pos in range(len(words_df)):

        probabilities = val_document_probs[
            document_id
        ].get(
            word_pos,
            np.array([1.0, 0.0, 0.0])
        )

        probability_list.append(
            probabilities[1] + probabilities[2]
        )

        if probabilities[1] + probabilities[2] < 0.90:
            pred_labels.append("O")

        elif probabilities[1] >= probabilities[2]:
            pred_labels.append("B-VENDOR_NAME")

        else:
            pred_labels.append("I-VENDOR_NAME")

    candidates = get_predicted_candidates_for_document(
        document_id,
        words_df,
        pred_labels,
        probability_list
    )

    predicted = (
        candidates[0]["text"]
        if candidates
        else None
    )

    gt_values = vendor_gt[
        vendor_gt["document_id"] == document_id
    ]["value"].tolist()

    best_similarity = 0

    if predicted is not None:

        predicted_norm = normalize_vendor_text(
            predicted
        )

        for value in gt_values:

            similarity = SequenceMatcher(
                None,
                predicted_norm,
                normalize_vendor_text(value)
            ).ratio()

            best_similarity = max(
                best_similarity,
                similarity
            )

    results.append({
        "document_id": document_id,
        "predicted": predicted,
        "ground_truth_count": len(gt_values),
        "best_similarity": best_similarity
    })


candidate_eval_df = pd.DataFrame(results)

print(
    "Documents evaluated:",
    len(candidate_eval_df)
)

print(
    "Similarity >= 0.80:",
    (
        candidate_eval_df["best_similarity"] >= 0.80
    ).mean()
)

print(
    "Similarity >= 0.90:",
    (
        candidate_eval_df["best_similarity"] >= 0.90
    ).mean()
)

Documents evaluated: 572
Similarity >= 0.80: 0.534965034965035
Similarity >= 0.90: 0.5157342657342657


In [110]:
# Show the 15 worst validation predictions

worst = candidate_eval_df.sort_values(
    "best_similarity"
).head(15)

for _, row in worst.iterrows():

    print("\n" + "=" * 80)
    print("Document:", row["document_id"])
    print("Predicted:", row["predicted"])
    print("Ground-truth count:", row["ground_truth_count"])
    print("Best similarity:", round(row["best_similarity"], 3))


Document: 00134dd365a24343b35b78c6
Predicted: nan
Ground-truth count: 3
Best similarity: 0.0

Document: 00aa98164d264f4e924f55a9
Predicted: nan
Ground-truth count: 4
Best similarity: 0.0

Document: 0100fa8313bf4d3489c55c81
Predicted: nan
Ground-truth count: 5
Best similarity: 0.0

Document: ff34fb7572224f22ba00f01e
Predicted: nan
Ground-truth count: 0
Best similarity: 0.0

Document: fba03542fc794843bed0da6e
Predicted: nan
Ground-truth count: 1
Best similarity: 0.0

Document: f98683177f7e42579b904566
Predicted: nan
Ground-truth count: 1
Best similarity: 0.0

Document: ff0456f8765d4e9385109aa0
Predicted: nan
Ground-truth count: 3
Best similarity: 0.0

Document: 0e9fa414a3054e839c72685a
Predicted: nan
Ground-truth count: 1
Best similarity: 0.0

Document: 1203b6c41035495bbc412de6
Predicted: nan
Ground-truth count: 2
Best similarity: 0.0

Document: 250d4fdc5b114f379d1d2c22
Predicted: nan
Ground-truth count: 0
Best similarity: 0.0

Document: 3ef10e565663409cbbb43990
Predicted: nan
Ground-tr

In [111]:
doc_id = "00134dd365a24343b35b78c6"

words_df = val_vendor_df[
    val_vendor_df["document_id"] == doc_id
].copy()

words_df = words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

probabilities = []

for word_pos in range(len(words_df)):

    probs = val_document_probs[
        doc_id
    ].get(
        word_pos,
        np.array([1.0, 0.0, 0.0])
    )

    probabilities.append({
        "text": words_df.loc[word_pos, "text"],
        "p_O": probs[0],
        "p_B": probs[1],
        "p_I": probs[2],
        "p_vendor": probs[1] + probs[2],
        "true_label": words_df.loc[word_pos, "label"]
    })

prob_df = pd.DataFrame(probabilities)

print(
    prob_df
    .sort_values("p_vendor", ascending=False)
    .head(30)
    .to_string(index=False)
)

            text      p_O      p_B      p_I  p_vendor    true_label
         Testing 0.002038 0.000241 0.997721  0.997962 I-VENDOR_NAME
     BioReliance 0.002080 0.333114 0.664806  0.997920 B-VENDOR_NAME
            Inc. 0.004632 0.000248 0.995120  0.995368 I-VENDOR_NAME
    Development, 0.004774 0.000159 0.995067  0.995226 I-VENDOR_NAME
               & 0.005228 0.000179 0.994593  0.994772 I-VENDOR_NAME
         Testing 0.006142 0.018834 0.975024  0.993859 I-VENDOR_NAME
    Development, 0.006698 0.000224 0.993078  0.993302 I-VENDOR_NAME
            Inc. 0.010702 0.000283 0.989015  0.989298 I-VENDOR_NAME
               & 0.011953 0.000270 0.987777  0.988047 I-VENDOR_NAME
     BIORELIANCE 0.016255 0.332292 0.651452  0.983745 B-VENDOR_NAME
     BioReliance 0.400097 0.260524 0.339380  0.599904 B-VENDOR_NAME
         Testing 0.461654 0.003864 0.534483  0.538346 I-VENDOR_NAME
    Development, 0.464616 0.003784 0.531600  0.535384 I-VENDOR_NAME
               & 0.745040 0.016016 0.238944  0.2

In [112]:
def decode_vendor_entities(tokens, probabilities, threshold=0.90):

    entities = []

    current_tokens = []
    current_scores = []

    for i in range(len(tokens)):

        probs = probabilities[i]

        p_b = probs[1]
        p_i = probs[2]
        p_vendor = p_b + p_i

        if p_vendor >= threshold:

            # Start a new entity
            if p_b >= p_i or not current_tokens:

                if current_tokens:
                    entities.append({
                        "text": layout_tokenizer.convert_tokens_to_string(
                            current_tokens
                        ),
                        "score": float(
                            np.mean(current_scores)
                        )
                    })

                current_tokens = [tokens[i]]
                current_scores = [float(p_vendor)]

            # Continue existing entity
            else:

                current_tokens.append(tokens[i])
                current_scores.append(float(p_vendor))

        else:

            if current_tokens:

                entities.append({
                    "text": layout_tokenizer.convert_tokens_to_string(
                        current_tokens
                    ),
                    "score": float(
                        np.mean(current_scores)
                    )
                })

                current_tokens = []
                current_scores = []

    if current_tokens:

        entities.append({
            "text": layout_tokenizer.convert_tokens_to_string(
                current_tokens
            ),
            "score": float(
                np.mean(current_scores)
            )
        })

    return entities

In [113]:
doc_id = "00134dd365a24343b35b78c6"

words_df = val_vendor_df[
    val_vendor_df["document_id"] == doc_id
].copy()

words_df = words_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

tokens = words_df["text"].tolist()

probability_vectors = []

for word_pos in range(len(words_df)):

    probs = val_document_probs[
        doc_id
    ].get(
        word_pos,
        np.array([1.0, 0.0, 0.0])
    )

    probability_vectors.append(probs)

candidates = decode_vendor_entities(
    tokens,
    probability_vectors,
    threshold=0.90
)

print("Candidates:")

for candidate in candidates:
    print(candidate)

Candidates:
{'text': 'BioReliance Testing & Development, Inc.', 'score': 0.9962494134902954}
{'text': 'BIORELIANCE', 'score': 0.9837448596954346}
{'text': 'Testing & Development, Inc.', 'score': 0.9911264181137085}


In [114]:
from difflib import SequenceMatcher

def text_similarity(a, b):
    return SequenceMatcher(
        None,
        normalize_vendor_text(a),
        normalize_vendor_text(b)
    ).ratio()


match_results = []

vendor_gt = full_quality_df[
    (full_quality_df["field"] == "vendor_name") &
    (full_quality_df["similarity"] >= 0.80)
].copy()

for doc_idx, document_id in enumerate(
    val_vendor_df["document_id"].drop_duplicates()
):

    words_df = val_vendor_df[
        val_vendor_df["document_id"] == document_id
    ].copy()

    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    # Get probabilities for every OCR word
    probability_vectors = []

    for word_pos in range(len(words_df)):

        probability_vectors.append(
            val_document_probs[
                document_id
            ].get(
                word_pos,
                np.array([1.0, 0.0, 0.0])
            )
        )

    # Decode entities
    candidates = decode_vendor_entities(
        words_df["text"].tolist(),
        probability_vectors,
        threshold=0.90
    )

    predicted_texts = [
        candidate["text"]
        for candidate in candidates
    ]

    # Ground truth values
    gt_values = vendor_gt[
        vendor_gt["document_id"] == document_id
    ]["value"].tolist()

    best_similarity = 0.0

    # Compare EVERY predicted candidate against
    # EVERY valid ground-truth vendor annotation
    for predicted in predicted_texts:

        for ground_truth in gt_values:

            similarity = text_similarity(
                predicted,
                ground_truth
            )

            best_similarity = max(
                best_similarity,
                similarity
            )

    match_results.append({
        "document_id": document_id,
        "predicted_candidates": predicted_texts,
        "ground_truth": gt_values,
        "best_similarity": best_similarity
    })


candidate_match_df = pd.DataFrame(match_results)

print(
    "Validation documents:",
    len(candidate_match_df)
)

print(
    "Similarity >= 0.80:",
    (
        candidate_match_df["best_similarity"] >= 0.80
    ).mean()
)

print(
    "Similarity >= 0.90:",
    (
        candidate_match_df["best_similarity"] >= 0.90
    ).mean()
)

Validation documents: 572
Similarity >= 0.80: 0.7744755244755245
Similarity >= 0.90: 0.7412587412587412


In [115]:
baseline_result = pd.DataFrame({
    "model": ["LayoutLM vendor-name baseline"],
    "validation_similarity_80": [0.7744755244755245],
    "validation_similarity_90": [0.7412587412587412],
    "test_precision": [0.7494279176201373],
    "test_recall": [0.808641975308642],
    "test_entity_f1": [0.7779097387173397]
})

print(baseline_result)

                           model  validation_similarity_80  \
0  LayoutLM vendor-name baseline                  0.774476   

   validation_similarity_90  test_precision  test_recall  test_entity_f1  
0                  0.741259        0.749428     0.808642         0.77791  


In [116]:
target_fields = [
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "date_issue",
    "amount_total_gross",
    "amount_due"
]


def create_multilabel_document(doc_id):

    with zipfile.ZipFile(docile_zip, "r") as z:
        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    record = {
                        "document_id": doc_id,
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1]
                    }

                    # One binary column per field
                    for field in target_fields:
                        record[field] = 0

                    words.append(record)

    words_df = pd.DataFrame(words)

    # -----------------------------------------
    # Apply accepted annotations independently
    # -----------------------------------------

    for field in target_fields:

        annotations = full_quality_df[
            (full_quality_df["document_id"] == doc_id) &
            (full_quality_df["field"] == field) &
            (full_quality_df["similarity"] >= 0.80)
        ]

        for _, annotation in annotations.iterrows():

            for j in range(len(words_df)):

                word = words_df.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    words_df.loc[j, field] = 1

    return words_df

In [117]:
multi_test_df = create_multilabel_document(
    sample_doc_id
)

print(
    multi_test_df[
        [
            "text",
            "vendor_name",
            "vendor_address",
            "customer_billing_name",
            "customer_billing_address",
            "date_issue",
            "amount_total_gross",
            "amount_due"
        ]
    ]
    .query(
        "vendor_name == 1 or "
        "vendor_address == 1 or "
        "customer_billing_name == 1 or "
        "customer_billing_address == 1 or "
        "date_issue == 1 or "
        "amount_total_gross == 1 or "
        "amount_due == 1"
    )
    .head(30)
)

             text  vendor_name  vendor_address  customer_billing_name  \
12    BioReliance            1               0                      0   
13        Testing            1               0                      0   
14              &            1               0                      0   
15   Development,            1               0                      0   
16           Inc.            1               0                      0   
33    BIORELIANCE            1               0                      0   
54        Testing            1               0                      0   
55              &            1               0                      0   
56   Development,            1               0                      0   
57           Inc.            1               0                      0   
69    BioReliance            1               0                      0   
72              &            1               0                      0   
73        Testing            1               0     

In [118]:
def create_multilabel_document(doc_id):

    with zipfile.ZipFile(docile_zip, "r") as z:
        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    record = {
                        "document_id": doc_id,
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1]
                    }

                    for field in target_fields:
                        record[field] = 0

                    words.append(record)

    words_df = pd.DataFrame(words)

    # -----------------------------------------
    # Apply each field independently
    # -----------------------------------------

    for field in target_fields:

        field_annotations = full_quality_df[
            (full_quality_df["document_id"] == doc_id) &
            (full_quality_df["field"] == field)
        ]

        accepted_annotations = field_annotations[
            field_annotations["similarity"] >= 0.80
        ]

        # If annotations exist but none are reliable,
        # mark this field as UNKNOWN (-1)
        # rather than falsely marking every token as O.
        if (
            len(field_annotations) > 0 and
            len(accepted_annotations) == 0
        ):
            words_df[field] = -1
            continue

        for _, annotation in accepted_annotations.iterrows():

            for j in range(len(words_df)):

                word = words_df.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    words_df.loc[j, field] = 1

    return words_df

In [119]:
multi_test_df = create_multilabel_document(
    sample_doc_id
)

print(
    multi_test_df[
        [
            "text",
            "vendor_name",
            "vendor_address",
            "customer_billing_name",
            "customer_billing_address",
            "date_issue",
            "amount_total_gross",
            "amount_due"
        ]
    ]
    .query(
        "vendor_name != 0 or "
        "vendor_address != 0 or "
        "customer_billing_name != 0 or "
        "customer_billing_address != 0 or "
        "date_issue != 0 or "
        "amount_total_gross != 0 or "
        "amount_due != 0"
    )
)

              text  vendor_name  vendor_address  customer_billing_name  \
0             WIRE            0              -1                      0   
1    INSTRUCTIONS:            0              -1                      0   
2     NationsBank,            0              -1                      0   
3             N.A.            0              -1                      0   
4       Baltimore,            0              -1                      0   
..             ...          ...             ...                    ...   
170         (REMIT            0              -1                      0   
171             IN            0              -1                      0   
172           U.S.            0              -1                      0   
173       CURRENCY            0              -1                      0   
174          ONLY)            0              -1                      0   

     customer_billing_address  date_issue  amount_total_gross  amount_due  
0                           0      

In [120]:
for field in target_fields:
    print(
        field,
        "=>",
        multi_test_df[field].value_counts().to_dict()
    )

vendor_name => {0: 161, 1: 14}
vendor_address => {-1: 175}
customer_billing_name => {0: 172, 1: 3}
customer_billing_address => {0: 162, 1: 13}
date_issue => {0: 174, 1: 1}
amount_total_gross => {0: 174, 1: 1}
amount_due => {0: 174, 1: 1}


In [121]:
mixed_quality = []

for field in target_fields:

    field_quality = full_quality_df[
        full_quality_df["field"] == field
    ]

    for doc_id, group in field_quality.groupby("document_id"):

        has_accepted = (
            group["similarity"] >= 0.80
        ).any()

        has_rejected = (
            group["similarity"] < 0.80
        ).any()

        if has_accepted and has_rejected:
            mixed_quality.append({
                "document_id": doc_id,
                "field": field
            })

mixed_quality_df = pd.DataFrame(
    mixed_quality
)

print("Documents with both accepted and rejected occurrences:",
      len(mixed_quality_df))

print("\nBy field:")
print(
    mixed_quality_df["field"].value_counts()
)

Documents with both accepted and rejected occurrences: 254

By field:
field
vendor_address              100
amount_due                   74
amount_total_gross           36
vendor_name                  23
date_issue                   11
customer_billing_name         7
customer_billing_address      3
Name: count, dtype: int64


In [122]:
def create_multilabel_document(doc_id):

    with zipfile.ZipFile(docile_zip, "r") as z:
        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

    words = []

    for page_idx, page in enumerate(ocr_data["pages"]):

        for block_idx, block in enumerate(page["blocks"]):

            for line_idx, line in enumerate(block["lines"]):

                for word_idx, word in enumerate(line["words"]):

                    geometry = word["geometry"]

                    record = {
                        "document_id": doc_id,
                        "page": page_idx,
                        "block_id": block_idx,
                        "line_id": line_idx,
                        "word_id": word_idx,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1]
                    }

                    # Default: confidently NOT this field
                    for field in target_fields:
                        record[field] = 0

                    words.append(record)

    words_df = pd.DataFrame(words)

    # ------------------------------------------------
    # Process each metadata field independently
    # ------------------------------------------------

    for field in target_fields:

        field_annotations = full_quality_df[
            (full_quality_df["document_id"] == doc_id) &
            (full_quality_df["field"] == field)
        ].copy()

        accepted = field_annotations[
            field_annotations["similarity"] >= 0.80
        ]

        rejected = field_annotations[
            field_annotations["similarity"] < 0.80
        ]

        # ------------------------------------------------
        # First mark rejected regions as UNKNOWN (-1)
        # ------------------------------------------------

        for _, annotation in rejected.iterrows():

            for j in range(len(words_df)):

                word = words_df.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    words_df.loc[j, field] = -1

        # ------------------------------------------------
        # Then mark accepted regions as POSITIVE (1)
        # Accepted wins over uncertain
        # ------------------------------------------------

        for _, annotation in accepted.iterrows():

            for j in range(len(words_df)):

                word = words_df.iloc[j]

                if word["page"] != annotation["page"]:
                    continue

                if word_matches_annotation(
                    word,
                    annotation["bbox"]
                ):
                    words_df.loc[j, field] = 1

    return words_df

In [123]:
multi_test_df = create_multilabel_document(
    sample_doc_id
)

for field in target_fields:

    print(
        field,
        "=>",
        multi_test_df[field].value_counts().to_dict()
    )

vendor_name => {0: 161, 1: 14}
vendor_address => {0: 162, -1: 13}
customer_billing_name => {0: 172, 1: 3}
customer_billing_address => {0: 162, 1: 13}
date_issue => {0: 174, 1: 1}
amount_total_gross => {0: 174, 1: 1}
amount_due => {0: 174, 1: 1}


In [124]:
all_multilabel_records = []

with zipfile.ZipFile(docile_zip, "r") as z:

    for doc_num, doc_id in enumerate(invoice_doc_ids, start=1):

        ocr_data = json.loads(
            z.read(f"ocr/{doc_id}.json")
        )

        words = []

        for page_idx, page in enumerate(ocr_data["pages"]):

            for block_idx, block in enumerate(page["blocks"]):

                for line_idx, line in enumerate(block["lines"]):

                    for word_idx, word in enumerate(line["words"]):

                        geometry = word["geometry"]

                        record = {
                            "document_id": doc_id,
                            "page": page_idx,
                            "block_id": block_idx,
                            "line_id": line_idx,
                            "word_id": word_idx,
                            "text": word["value"],
                            "confidence": word["confidence"],
                            "x1": geometry[0][0],
                            "y1": geometry[0][1],
                            "x2": geometry[1][0],
                            "y2": geometry[1][1]
                        }

                        for field in target_fields:
                            record[field] = 0

                        words.append(record)

        words_df = pd.DataFrame(words)

        # -----------------------------------------
        # Apply each field independently
        # -----------------------------------------

        for field in target_fields:

            field_annotations = full_quality_df[
                (full_quality_df["document_id"] == doc_id) &
                (full_quality_df["field"] == field)
            ]

            accepted = field_annotations[
                field_annotations["similarity"] >= 0.80
            ]

            rejected = field_annotations[
                field_annotations["similarity"] < 0.80
            ]

            # -------------------------------------
            # Mark uncertain regions as -1
            # -------------------------------------

            for _, annotation in rejected.iterrows():

                for j in range(len(words_df)):

                    word = words_df.iloc[j]

                    if word["page"] != annotation["page"]:
                        continue

                    if word_matches_annotation(
                        word,
                        annotation["bbox"]
                    ):
                        words_df.loc[j, field] = -1

            # -------------------------------------
            # Mark accepted regions as 1
            # -------------------------------------

            for _, annotation in accepted.iterrows():

                for j in range(len(words_df)):

                    word = words_df.iloc[j]

                    if word["page"] != annotation["page"]:
                        continue

                    if word_matches_annotation(
                        word,
                        annotation["bbox"]
                    ):
                        words_df.loc[j, field] = 1

        all_multilabel_records.extend(
            words_df.to_dict("records")
        )

        if doc_num % 250 == 0:
            print(
                f"Processed {doc_num}/{len(invoice_doc_ids)} invoices"
            )


multilabel_df = pd.DataFrame(
    all_multilabel_records
)

print("\nFinal shape:", multilabel_df.shape)
print(
    "Documents:",
    multilabel_df["document_id"].nunique()
)

Processed 250/3850 invoices
Processed 500/3850 invoices
Processed 750/3850 invoices
Processed 1000/3850 invoices
Processed 1250/3850 invoices
Processed 1500/3850 invoices
Processed 1750/3850 invoices
Processed 2000/3850 invoices
Processed 2250/3850 invoices
Processed 2500/3850 invoices
Processed 2750/3850 invoices
Processed 3000/3850 invoices
Processed 3250/3850 invoices
Processed 3500/3850 invoices
Processed 3750/3850 invoices

Final shape: (940348, 18)
Documents: 3850


In [125]:
for field in target_fields:
    print("\n" + "=" * 50)
    print(field)
    print(multilabel_df[field].value_counts().sort_index())


vendor_name
vendor_name
-1       251
 0    923627
 1     16470
Name: count, dtype: int64

vendor_address
vendor_address
-1      2269
 0    889829
 1     48250
Name: count, dtype: int64

customer_billing_name
customer_billing_name
-1       158
 0    923411
 1     16779
Name: count, dtype: int64

customer_billing_address
customer_billing_address
-1       964
 0    893596
 1     45788
Name: count, dtype: int64

date_issue
date_issue
-1       102
 0    933568
 1      6678
Name: count, dtype: int64

amount_total_gross
amount_total_gross
-1       135
 0    936360
 1      3853
Name: count, dtype: int64

amount_due
amount_due
-1       140
 0    935985
 1      4223
Name: count, dtype: int64


In [126]:
processed_dir = Path("../processed")
processed_dir.mkdir(exist_ok=True)

multilabel_path = processed_dir / "invoice_multilabel_tokens.pkl"

multilabel_df.to_pickle(
    multilabel_path
)

print("Saved:", multilabel_path)
print("Rows:", len(multilabel_df))

Saved: ..\processed\invoice_multilabel_tokens.pkl
Rows: 940348


In [127]:
check_df = pd.read_pickle(
    "../processed/invoice_multilabel_tokens.pkl"
)

print(check_df.shape)
print(check_df["document_id"].nunique())

(940348, 18)
3850


In [128]:
train_multi_df = multilabel_df[
    multilabel_df["document_id"].isin(meta_train_ids)
].copy()

val_multi_df = multilabel_df[
    multilabel_df["document_id"].isin(meta_val_ids)
].copy()

test_multi_df = multilabel_df[
    multilabel_df["document_id"].isin(meta_test_ids)
].copy()

print("Train:", train_multi_df.shape)
print("Validation:", val_multi_df.shape)
print("Test:", test_multi_df.shape)

print("\nDocuments:")
print(
    "Train:", train_multi_df["document_id"].nunique()
)
print(
    "Validation:", val_multi_df["document_id"].nunique()
)
print(
    "Test:", test_multi_df["document_id"].nunique()
)

Train: (659937, 18)
Validation: (143347, 18)
Test: (137064, 18)

Documents:
Train: 2695
Validation: 577
Test: 578


In [129]:
train_ids = set(train_multi_df["document_id"])
val_ids = set(val_multi_df["document_id"])
test_ids = set(test_multi_df["document_id"])

print("Train ∩ Validation:", len(train_ids & val_ids))
print("Train ∩ Test:", len(train_ids & test_ids))
print("Validation ∩ Test:", len(val_ids & test_ids))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [130]:
field2id = {
    "vendor_name": 0,
    "vendor_address": 1,
    "customer_billing_name": 2,
    "customer_billing_address": 3,
    "date_issue": 4,
    "amount_total_gross": 5,
    "amount_due": 6
}

id2field = {
    value: key
    for key, value in field2id.items()
}

print(field2id)

{'vendor_name': 0, 'vendor_address': 1, 'customer_billing_name': 2, 'customer_billing_address': 3, 'date_issue': 4, 'amount_total_gross': 5, 'amount_due': 6}


In [131]:
sample_doc_id = train_multi_df["document_id"].iloc[0]

sample_df = train_multi_df[
    train_multi_df["document_id"] == sample_doc_id
].copy()

sample_df = sample_df.sort_values(
    ["page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

words = sample_df["text"].tolist()

boxes = []

for _, row in sample_df.iterrows():
    boxes.append([
        int(row["x1"] * 1000),
        int(row["y1"] * 1000),
        int(row["x2"] * 1000),
        int(row["y2"] * 1000)
    ])

# One 7-value vector per OCR word
word_labels = []

for _, row in sample_df.iterrows():

    labels = []

    for field in target_fields:
        labels.append(row[field])

    word_labels.append(labels)

encoded = layout_tokenizer(
    words,
    boxes=boxes,
    is_split_into_words=True,
    truncation=True,
    max_length=512,
    stride=128,
    return_overflowing_tokens=True,
    return_attention_mask=True
)

print("OCR words:", len(words))
print("Chunks:", len(encoded["input_ids"]))

OCR words: 411
Chunks: 3


In [132]:
word_ids = encoded.word_ids(batch_index=0)

for i in range(min(30, len(word_ids))):

    word_id = word_ids[i]

    if word_id is None:
        continue

    token = layout_tokenizer.convert_ids_to_tokens(
        [encoded["input_ids"][0][i]]
    )[0]

    print(
        token,
        "→",
        word_labels[word_id]
    )

please → [0, 0, 0, 0, 0, 0, 0]
re → [0, 0, 0, 0, 0, 0, 0]
##mit → [0, 0, 0, 0, 0, 0, 0]
to → [0, 0, 0, 0, 0, 0, 0]
: → [0, 0, 0, 0, 0, 0, 0]
km → [1, 1, 0, 0, 0, 0, 0]
##oz → [1, 1, 0, 0, 0, 0, 0]
92 → [1, 1, 0, 0, 0, 0, 0]
. → [1, 1, 0, 0, 0, 0, 0]
3 → [1, 1, 0, 0, 0, 0, 0]
the → [1, 1, 0, 0, 0, 0, 0]
moose → [1, 1, 0, 0, 0, 0, 0]
136 → [0, 1, 0, 0, 0, 0, 0]
##0 → [0, 1, 0, 0, 0, 0, 0]
e → [0, 1, 0, 0, 0, 0, 0]
. → [0, 1, 0, 0, 0, 0, 0]
sherwood → [0, 1, 0, 0, 0, 0, 0]
drive → [0, 1, 0, 0, 0, 0, 0]
grand → [0, 1, 0, 0, 0, 0, 0]
junction → [0, 1, 0, 0, 0, 0, 0]
, → [0, 1, 0, 0, 0, 0, 0]
co → [0, 1, 0, 0, 0, 0, 0]
81 → [0, 1, 0, 0, 0, 0, 0]
##50 → [0, 1, 0, 0, 0, 0, 0]
##1 → [0, 1, 0, 0, 0, 0, 0]
97 → [0, 0, 0, 0, 0, 0, 0]
##0 → [0, 0, 0, 0, 0, 0, 0]
. → [0, 0, 0, 0, 0, 0, 0]
254 → [0, 0, 0, 0, 0, 0, 0]


In [148]:
def prepare_multilabel_layoutlm_examples(
    words_df,
    tokenizer,
    target_fields,
    max_length=256,
    stride=128
):
    words_df = words_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    ).reset_index(drop=True)

    words = words_df["text"].tolist()

    # Convert OCR coordinates from [0, 1] → [0, 1000]
    boxes = []

    for _, row in words_df.iterrows():
        boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    # One multi-label vector per OCR word
    word_labels = []

    for _, row in words_df.iterrows():

        labels = []

        for field in target_fields:
            labels.append(
                int(row[field])
            )

        word_labels.append(labels)

    encoded = tokenizer(
        words,
        boxes=boxes,
        is_split_into_words=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        return_overflowing_tokens=True,
        return_attention_mask=True
    )

    examples = []

    for chunk_idx in range(
        len(encoded["input_ids"])
    ):

        word_ids = encoded.word_ids(
            batch_index=chunk_idx
        )

        chunk_labels = []
        chunk_boxes = []

        for word_id in word_ids:

            # [CLS], [SEP], etc.
            if word_id is None:

                chunk_labels.append(
                    [-1] * len(target_fields)
                )

                chunk_boxes.append(
                    [0, 0, 0, 0]
                )

            else:

                chunk_labels.append(
                    word_labels[word_id]
                )

                chunk_boxes.append(
                    boxes[word_id]
                )

        examples.append({
            "input_ids": encoded["input_ids"][chunk_idx],
            "attention_mask": encoded["attention_mask"][chunk_idx],
            "bbox": chunk_boxes,
            "labels": chunk_labels
        })

    return examples

In [149]:
sample_doc_id = train_multi_df["document_id"].iloc[0]

sample_df = train_multi_df[
    train_multi_df["document_id"] == sample_doc_id
].copy()

sample_examples = prepare_multilabel_layoutlm_examples(
    sample_df,
    layout_tokenizer,
    target_fields
)

print("Document:", sample_doc_id)
print("Chunks:", len(sample_examples))
print("Tokens:", len(sample_examples[0]["input_ids"]))
print("Boxes:", len(sample_examples[0]["bbox"]))
print("Label vectors:", len(sample_examples[0]["labels"]))

print("\nExample label vector:")
print(sample_examples[0]["labels"][10])

Document: 00136a27c7774c1e8dc6b2f2
Chunks: 7
Tokens: 256
Boxes: 256
Label vectors: 256

Example label vector:
[1, 1, 0, 0, 0, 0, 0]


In [150]:
def generate_multilabel_examples(split_df):
    for doc_id, words_df in split_df.groupby(
        "document_id",
        sort=False
    ):

        examples = prepare_multilabel_layoutlm_examples(
            words_df,
            layout_tokenizer,
            target_fields
        )

        for example in examples:
            yield example


train_multi_hf = Dataset.from_generator(
    lambda: generate_multilabel_examples(train_multi_df)
)

val_multi_hf = Dataset.from_generator(
    lambda: generate_multilabel_examples(val_multi_df)
)

test_multi_hf = Dataset.from_generator(
    lambda: generate_multilabel_examples(test_multi_df)
)

print("Train:", train_multi_hf)
print("Validation:", val_multi_hf)
print("Test:", test_multi_hf)

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Train: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 10247
})
Validation: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 2169
})
Test: Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 2077
})


In [136]:
from pathlib import Path

multi_layout_dir = Path(
    "../processed/layoutlm_multilabel"
)

multi_layout_dir.mkdir(
    parents=True,
    exist_ok=True
)

train_multi_hf.save_to_disk(
    multi_layout_dir / "train"
)

val_multi_hf.save_to_disk(
    multi_layout_dir / "validation"
)

test_multi_hf.save_to_disk(
    multi_layout_dir / "test"
)

print("Multi-label LayoutLM datasets saved.")

Saving the dataset (0/1 shards):   0%|          | 0/4522 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/954 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/925 [00:00<?, ? examples/s]

Multi-label LayoutLM datasets saved.


In [137]:
from datasets import load_from_disk

check_train = load_from_disk(
    multi_layout_dir / "train"
)

print(check_train)

Dataset({
    features: ['input_ids', 'attention_mask', 'bbox', 'labels'],
    num_rows: 4522
})


In [138]:
import numpy as np

positive_counts = np.zeros(len(target_fields), dtype=np.int64)
negative_counts = np.zeros(len(target_fields), dtype=np.int64)

for example in train_multi_hf:

    labels_array = np.array(
        example["labels"],
        dtype=np.int8
    )

    for field_idx in range(len(target_fields)):

        field_labels = labels_array[:, field_idx]

        positive_counts[field_idx] += (
            field_labels == 1
        ).sum()

        negative_counts[field_idx] += (
            field_labels == 0
        ).sum()


# Smoothed positive weights
pos_weights = np.sqrt(
    negative_counts / positive_counts
)

weight_df = pd.DataFrame({
    "field": target_fields,
    "positive": positive_counts,
    "negative": negative_counts,
    "pos_weight": pos_weights
})

print(weight_df)

                      field  positive  negative  pos_weight
0               vendor_name     21648   1674390    8.794667
1            vendor_address     69699   1623257    4.825923
2     customer_billing_name     18586   1677673    9.500811
3  customer_billing_address     55353   1639719    5.442697
4                date_issue     14922   1681224   10.614493
5        amount_total_gross     14703   1681453   10.693980
6                amount_due     16127   1680029   10.206612


In [139]:
from dataclasses import dataclass
from typing import List, Dict, Any

import torch
from torch.nn.utils.rnn import pad_sequence


@dataclass
class MultiLabelLayoutLMDataCollator:
    tokenizer: Any
    label_pad_value: float = -1.0

    def __call__(self, features: List[Dict[str, Any]]):

        input_ids = [
            torch.tensor(
                feature["input_ids"],
                dtype=torch.long
            )
            for feature in features
        ]

        attention_masks = [
            torch.tensor(
                feature["attention_mask"],
                dtype=torch.long
            )
            for feature in features
        ]

        bboxes = [
            torch.tensor(
                feature["bbox"],
                dtype=torch.long
            )
            for feature in features
        ]

        labels = [
            torch.tensor(
                feature["labels"],
                dtype=torch.float
            )
            for feature in features
        ]

        input_ids = pad_sequence(
            input_ids,
            batch_first=True,
            padding_value=self.tokenizer.pad_token_id
        )

        attention_masks = pad_sequence(
            attention_masks,
            batch_first=True,
            padding_value=0
        )

        max_length = input_ids.size(1)

        padded_bboxes = []
        padded_labels = []

        for bbox, label in zip(bboxes, labels):

            padding_length = (
                max_length - bbox.size(0)
            )

            if padding_length > 0:

                bbox_padding = torch.zeros(
                    (padding_length, 4),
                    dtype=torch.long
                )

                label_padding = torch.full(
                    (padding_length, len(target_fields)),
                    self.label_pad_value,
                    dtype=torch.float
                )

                bbox = torch.cat(
                    [bbox, bbox_padding],
                    dim=0
                )

                label = torch.cat(
                    [label, label_padding],
                    dim=0
                )

            padded_bboxes.append(bbox)
            padded_labels.append(label)

        bboxes = torch.stack(padded_bboxes)
        labels = torch.stack(padded_labels)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_masks,
            "bbox": bboxes,
            "labels": labels
        }


multi_data_collator = MultiLabelLayoutLMDataCollator(
    tokenizer=layout_tokenizer
)

print("Multi-label LayoutLM collator ready")

Multi-label LayoutLM collator ready


In [140]:
test_batch = multi_data_collator([
    train_multi_hf[0],
    train_multi_hf[1]
])

for key, value in test_batch.items():
    print(key, value.shape)    

input_ids torch.Size([2, 512])
attention_mask torch.Size([2, 512])
bbox torch.Size([2, 512, 4])
labels torch.Size([2, 512, 7])


In [152]:
import torch
import torch.nn as nn

from transformers import LayoutLMModel
from transformers.modeling_outputs import TokenClassifierOutput


class MultiLabelLayoutLM(nn.Module):

    def __init__(
        self,
        model_name,
        num_labels,
        pos_weight
    ):
        super().__init__()

        self.num_labels = num_labels

        # Pretrained LayoutLM backbone
        self.layoutlm = LayoutLMModel.from_pretrained(
            model_name
        )

        hidden_size = (
            self.layoutlm.config.hidden_size
        )

        self.dropout = nn.Dropout(
            self.layoutlm.config.hidden_dropout_prob
        )

        # 7 independent predictions per token
        self.classifier = nn.Linear(
            hidden_size,
            num_labels
        )

        self.register_buffer(
            "pos_weight",
            torch.tensor(
                pos_weight,
                dtype=torch.float32
            )
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        bbox=None,
        labels=None
    ):

        outputs = self.layoutlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox
        )

        sequence_output = self.dropout(
            outputs.last_hidden_state
        )

        logits = self.classifier(
            sequence_output
        )

        loss = None

        if labels is not None:

            loss_function = nn.BCEWithLogitsLoss(
                pos_weight=self.pos_weight,
                reduction="none"
            )

            # Element-wise loss
            loss_values = loss_function(
                logits,
                labels.clamp(min=0)
            )

            # Ignore:
            # - special tokens
            # - uncertain annotations (-1)
            valid_mask = (
                labels != -1
            ).float()

            loss_values = (
                loss_values * valid_mask
            )

            loss = (
                loss_values.sum()
                / valid_mask.sum().clamp(min=1)
            )

        return TokenClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=outputs.hidden_states,
            attentions=outputs.attentions
        )


multi_layout_model = MultiLabelLayoutLM(
    model_name="microsoft/layoutlm-base-uncased",
    num_labels=len(target_fields),
    pos_weight=pos_weights
)

multi_layout_model.to(device)

print("Multi-label LayoutLM loaded")
print(
    "Output labels per token:",
    len(target_fields)
)
print(
    "Device:",
    next(multi_layout_model.parameters()).device
)

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Multi-label LayoutLM loaded
Output labels per token: 7
Device: cuda:0


In [142]:
# Take one small batch
test_batch = multi_data_collator([
    train_multi_hf[0],
    train_multi_hf[1]
])

# Move batch to GPU
test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

# Forward pass
with torch.no_grad():
    output = multi_layout_model(**test_batch)

print("Logits shape:", output.logits.shape)
print("Loss:", output.loss.item())
print("Loss is finite:", torch.isfinite(output.loss).item())

Logits shape: torch.Size([2, 512, 7])
Loss: 0.7475994229316711
Loss is finite: True


In [154]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

multi_trainer = Trainer(
    model=multi_layout_model,
    args=TrainingArguments(
        output_dir="../models/layoutlm_multilabel",

        num_train_epochs=2,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        gradient_accumulation_steps=1,

        learning_rate=3e-5,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        fp16=True,

        logging_steps=100,
        report_to="none"
    ),

    train_dataset=train_multi_hf,
    eval_dataset=val_multi_hf,
    data_collator=multi_data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("Trainer ready")

Trainer ready


In [144]:
train_result = multi_trainer.train()

print("Training complete.")
print("Best checkpoint:", multi_trainer.state.best_model_checkpoint)
print("Best eval loss:", multi_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [151]:
print("Train chunks:", len(train_multi_hf))
print("Validation chunks:", len(val_multi_hf))
print("Test chunks:", len(test_multi_hf))

Train chunks: 10247
Validation chunks: 2169
Test chunks: 2077


In [153]:
print(next(multi_layout_model.parameters()).device)

cuda:0


In [155]:
train_result = multi_trainer.train()

print("Training complete.")
print("Best checkpoint:", multi_trainer.state.best_model_checkpoint)
print("Best eval loss:", multi_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss
1,0.031638,0.031582
2,0.021469,0.029392


Training complete.
Best checkpoint: ../models/layoutlm_multilabel\checkpoint-2562
Best eval loss: 0.029392315074801445


In [156]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

print("Running test-set prediction...")

pred_output = multi_trainer.predict(test_multi_hf)

logits = pred_output.predictions
true_labels = pred_output.label_ids

# Convert logits to probabilities
probs = 1 / (1 + np.exp(-logits))

# Binary predictions
pred_labels = (probs >= 0.5).astype(int)

field_metrics = []

for i, field in enumerate(target_fields):
    y_true = true_labels[:, :, i]
    y_pred = pred_labels[:, :, i]

    # Ignore uncertain/padding tokens
    mask = y_true != -1

    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_valid,
        y_pred_valid,
        average="binary",
        zero_division=0
    )

    field_metrics.append({
        "field": field,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

metrics_df = pd.DataFrame(field_metrics)

print(metrics_df)

Running test-set prediction...


                      field  precision    recall        f1
0               vendor_name   0.687235  0.937852  0.793218
1            vendor_address   0.857284  0.953120  0.902665
2     customer_billing_name   0.759669  0.967369  0.851030
3  customer_billing_address   0.883725  0.978972  0.928913
4                date_issue   0.811619  0.989326  0.891705
5        amount_total_gross   0.464080  0.949140  0.623366
6                amount_due   0.678908  0.958089  0.794692


In [157]:
print("Generating validation predictions...")

val_output = multi_trainer.predict(val_multi_hf)

val_logits = val_output.predictions
val_true_labels = val_output.label_ids

val_probs = 1 / (1 + np.exp(-val_logits))

threshold_results = []

for i, field in enumerate(target_fields):
    y_true = val_true_labels[:, :, i]
    
    mask = y_true != -1
    y_true_valid = y_true[mask]
    prob_valid = val_probs[:, :, i][mask]

    best_threshold = 0.5
    best_f1 = 0.0
    best_precision = 0.0
    best_recall = 0.0

    for threshold in np.arange(0.30, 0.96, 0.05):
        y_pred_valid = (prob_valid >= threshold).astype(int)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true_valid,
            y_pred_valid,
            average="binary",
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    threshold_results.append({
        "field": field,
        "best_threshold": round(float(best_threshold), 2),
        "precision": round(float(best_precision), 4),
        "recall": round(float(best_recall), 4),
        "f1": round(float(best_f1), 4)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df)

Generating validation predictions...


C:\Users\sudee\AppData\Local\Temp\ipykernel_32468\686801018.py:8: RuntimeWarning: overflow encountered in exp
  val_probs = 1 / (1 + np.exp(-val_logits))


ValueError: Target is multiclass but average='binary'. Please choose another average setting, one of [None, 'micro', 'macro', 'weighted'].

In [158]:
for i, field in enumerate(target_fields):
    y = val_true_labels[:, :, i]
    print(field, "unique labels:", np.unique(y))

vendor_name unique labels: [-100.   -1.    0.    1.]
vendor_address unique labels: [-100.   -1.    0.    1.]
customer_billing_name unique labels: [-100.   -1.    0.    1.]
customer_billing_address unique labels: [-100.   -1.    0.    1.]
date_issue unique labels: [-100.   -1.    0.    1.]
amount_total_gross unique labels: [-100.   -1.    0.    1.]
amount_due unique labels: [-100.   -1.    0.    1.]


In [159]:
threshold_results = []

for i, field in enumerate(target_fields):
    y_true = val_true_labels[:, :, i]

    # Keep only genuine 0/1 labels
    mask = (y_true == 0) | (y_true == 1)

    y_true_valid = y_true[mask].astype(int)
    prob_valid = val_probs[:, :, i][mask]

    best_threshold = 0.5
    best_f1 = 0.0
    best_precision = 0.0
    best_recall = 0.0

    for threshold in np.arange(0.30, 0.96, 0.05):
        y_pred_valid = (prob_valid >= threshold).astype(int)

        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true_valid,
            y_pred_valid,
            average="binary",
            zero_division=0
        )

        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold
            best_precision = precision
            best_recall = recall

    threshold_results.append({
        "field": field,
        "best_threshold": round(float(best_threshold), 2),
        "precision": round(float(best_precision), 4),
        "recall": round(float(best_recall), 4),
        "f1": round(float(best_f1), 4)
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df)

                      field  best_threshold  precision  recall      f1
0               vendor_name            0.95     0.8597  0.8460  0.8528
1            vendor_address            0.90     0.9302  0.9231  0.9267
2     customer_billing_name            0.95     0.9086  0.9000  0.9043
3  customer_billing_address            0.90     0.9364  0.9538  0.9450
4                date_issue            0.95     0.9364  0.9694  0.9526
5        amount_total_gross            0.90     0.6524  0.7900  0.7146
6                amount_due            0.95     0.8312  0.8768  0.8534


In [160]:
# Apply validation-selected thresholds to the test set

test_probs = 1 / (1 + np.exp(-logits))

test_thresholds = {
    row["field"]: row["best_threshold"]
    for _, row in threshold_df.iterrows()
}

test_metrics = []

for i, field in enumerate(target_fields):
    y_true = true_labels[:, :, i]

    # Only evaluate genuine 0/1 labels
    mask = (y_true == 0) | (y_true == 1)

    y_true_valid = y_true[mask].astype(int)
    prob_valid = test_probs[:, :, i][mask]

    threshold = test_thresholds[field]
    y_pred_valid = (prob_valid >= threshold).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_valid,
        y_pred_valid,
        average="binary",
        zero_division=0
    )

    test_metrics.append({
        "field": field,
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })

test_metrics_df = pd.DataFrame(test_metrics)

print(test_metrics_df)
print("\nMacro F1:", test_metrics_df["f1"].mean())

                      field  threshold  precision    recall        f1
0               vendor_name       0.95   0.854791  0.835936  0.845258
1            vendor_address       0.90   0.917536  0.919358  0.918446
2     customer_billing_name       0.95   0.881298  0.935955  0.907804
3  customer_billing_address       0.90   0.922625  0.962887  0.942326
4                date_issue       0.95   0.893199  0.971101  0.930523
5        amount_total_gross       0.90   0.639472  0.808845  0.714255
6                amount_due       0.95   0.852625  0.874718  0.863530

Macro F1: 0.8745918150414511


In [161]:
import json
from pathlib import Path

model_save_path = Path("../models/layoutlm_multilabel/final")
model_save_path.mkdir(parents=True, exist_ok=True)

multi_trainer.save_model(str(model_save_path))

threshold_path = model_save_path / "field_thresholds.json"

with open(threshold_path, "w") as f:
    json.dump(test_thresholds, f, indent=2)

print("Model saved to:", model_save_path)
print("Thresholds saved to:", threshold_path)
print(test_thresholds)

Model saved to: ..\models\layoutlm_multilabel\final
Thresholds saved to: ..\models\layoutlm_multilabel\final\field_thresholds.json
{'vendor_name': 0.95, 'vendor_address': 0.9, 'customer_billing_name': 0.95, 'customer_billing_address': 0.9, 'date_issue': 0.95, 'amount_total_gross': 0.9, 'amount_due': 0.95}


In [162]:
def ocr_to_dataframe(ocr_data):
    rows = []

    for page_idx, page in enumerate(ocr_data["pages"]):
        for block_id, block in enumerate(page["blocks"]):
            for line_id, line in enumerate(block["lines"]):
                for word_id, word in enumerate(line["words"]):
                    geometry = word["geometry"]

                    rows.append({
                        "page": page_idx,
                        "block_id": block_id,
                        "line_id": line_id,
                        "word_id": word_id,
                        "text": word["value"],
                        "confidence": word["confidence"],
                        "x1": geometry[0][0],
                        "y1": geometry[0][1],
                        "x2": geometry[1][0],
                        "y2": geometry[1][1]
                    })

    return pd.DataFrame(rows)

In [163]:
print("OCR parser ready")

OCR parser ready


In [164]:
def predict_invoice_fields(ocr_data):
    # Convert OCR JSON to word-level dataframe
    words_df = ocr_to_dataframe(ocr_data).copy()

    words = words_df["text"].tolist()

    # DocILE geometry is normalized to 0-1.
    # LayoutLM expects bounding boxes in 0-1000.
    boxes = []

    for _, row in words_df.iterrows():
        boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    tokenizer = LayoutLMTokenizerFast.from_pretrained(
        "microsoft/layoutlm-base-uncased"
    )

    encoded = tokenizer(
        words,
        boxes=boxes,
        is_split_into_words=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=256,
        stride=128,
        padding=False,
        return_tensors="pt"
    )

    all_word_probs = np.zeros(
        (len(words), len(target_fields)),
        dtype=np.float32
    )

    all_word_counts = np.zeros(
        len(words),
        dtype=np.int32
    )

    multi_layout_model.eval()

    for chunk_idx in range(len(encoded["input_ids"])):

        input_ids = encoded["input_ids"][chunk_idx].unsqueeze(0).to(device)
        attention_mask = encoded["attention_mask"][chunk_idx].unsqueeze(0).to(device)
        bbox = encoded["bbox"][chunk_idx].unsqueeze(0).to(device)

        with torch.no_grad():
            output = multi_layout_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox
            )

        probs = torch.sigmoid(output.logits)[0].cpu().numpy()

        word_ids = encoded.word_ids(batch_index=chunk_idx)

        for token_idx, word_idx in enumerate(word_ids):

            if word_idx is None:
                continue

            all_word_probs[word_idx] = np.maximum(
                all_word_probs[word_idx],
                probs[token_idx]
            )

            all_word_counts[word_idx] += 1

    result_df = words_df.copy()

    for i, field in enumerate(target_fields):
        result_df[field + "_prob"] = all_word_probs[:, i]

    return result_df

In [165]:
print("Invoice prediction function ready")

Invoice prediction function ready


In [166]:
# Pick one invoice from the untouched test set
sample_doc_id = test_multi_df["document_id"].iloc[0]

sample_words_df = (
    test_multi_df[test_multi_df["document_id"] == sample_doc_id]
    .sort_values(["page", "block_id", "line_id", "word_id"])
    .reset_index(drop=True)
)

print("Document:", sample_doc_id)
print("OCR words:", len(sample_words_df))
print(sample_words_df[["text", "page", "x1", "y1", "x2", "y2"]].head(20))

Document: 00405d767fb1406180d05ef2
OCR words: 79
                 text  page        x1        y1        x2        y2
0   Rothstein-lauber,     0  0.250000  0.140625  0.386719  0.155273
1        Incorporated     0  0.389648  0.140625  0.495117  0.155273
2              Market     0  0.249023  0.154297  0.303711  0.166016
3            Research     0  0.303711  0.154297  0.375000  0.166016
4            Services     0  0.375000  0.154297  0.438477  0.166016
5                1351     0  0.250000  0.166992  0.282227  0.182617
6          Washington     0  0.283203  0.168945  0.367188  0.182617
7     Blvd.-Stamford,     0  0.368164  0.167969  0.478516  0.182617
8             CT06902     0  0.479492  0.168945  0.543945  0.180664
9               (203)     0  0.250000  0.179688  0.288086  0.195312
10       324-2420-FAX     0  0.288086  0.179688  0.387695  0.193359
11              (203)     0  0.384766  0.179688  0.424805  0.194336
12           964-8269     0  0.422852  0.180664  0.487305  0.192383

In [167]:
from collections import defaultdict

# Reconstruct DocILE-style OCR JSON from the test dataframe
ocr_data = {
    "pages": []
}

page_groups = sample_words_df.groupby("page", sort=True)

for page_num, page_df in page_groups:
    page_data = {"blocks": []}

    block_groups = page_df.groupby("block_id", sort=True)

    for block_num, block_df in block_groups:
        block_data = {"lines": []}

        line_groups = block_df.groupby("line_id", sort=True)

        for line_num, line_df in line_groups:
            line_data = {"words": []}

            for _, row in line_df.sort_values("word_id").iterrows():
                line_data["words"].append({
                    "value": row["text"],
                    "confidence": row["confidence"],
                    "geometry": [
                        [row["x1"], row["y1"]],
                        [row["x2"], row["y2"]]
                    ]
                })

            block_data["lines"].append(line_data)

        page_data["blocks"].append(block_data)

    ocr_data["pages"].append(page_data)


# Run the trained LayoutLM model
prediction_df = predict_invoice_fields(ocr_data)

print("Prediction complete.")
print("Words:", len(prediction_df))

KeyError: 'bbox'

In [168]:
def predict_invoice_fields(ocr_data):
    words_df = ocr_to_dataframe(ocr_data).copy()

    words = words_df["text"].tolist()

    # Convert normalized DocILE boxes (0-1) to LayoutLM boxes (0-1000)
    word_boxes = []

    for _, row in words_df.iterrows():
        word_boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    tokenizer = LayoutLMTokenizerFast.from_pretrained(
        "microsoft/layoutlm-base-uncased"
    )

    encoded = tokenizer(
        words,
        boxes=word_boxes,
        is_split_into_words=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=256,
        stride=128,
        padding=False,
        return_tensors="pt"
    )

    # Store probability for every original OCR word
    all_word_probs = np.zeros(
        (len(words), len(target_fields)),
        dtype=np.float32
    )

    multi_layout_model.eval()

    for chunk_idx in range(len(encoded["input_ids"])):

        input_ids = encoded["input_ids"][chunk_idx].unsqueeze(0).to(device)
        attention_mask = encoded["attention_mask"][chunk_idx].unsqueeze(0).to(device)

        # Build bbox manually from word_ids
        word_ids = encoded.word_ids(batch_index=chunk_idx)

        chunk_bboxes = []

        for word_idx in word_ids:
            if word_idx is None:
                chunk_bboxes.append([0, 0, 0, 0])
            else:
                chunk_bboxes.append(word_boxes[word_idx])

        bbox = torch.tensor(
            chunk_bboxes,
            dtype=torch.long
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            output = multi_layout_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox
            )

        probs = torch.sigmoid(
            output.logits
        )[0].cpu().numpy()

        # Assign token probabilities back to original OCR words
        for token_idx, word_idx in enumerate(word_ids):

            if word_idx is None:
                continue

            # Use maximum probability when the same word
            # appears in overlapping chunks.
            all_word_probs[word_idx] = np.maximum(
                all_word_probs[word_idx],
                probs[token_idx]
            )

    result_df = words_df.copy()

    for i, field in enumerate(target_fields):
        result_df[field + "_prob"] = all_word_probs[:, i]

    return result_df

In [169]:
print("Updated prediction function ready")

Updated prediction function ready


In [170]:
prediction_df = predict_invoice_fields(ocr_data)

print("Prediction complete.")
print("Words:", len(prediction_df))

Prediction complete.
Words: 79


In [171]:
for field in target_fields:
    threshold = test_thresholds[field]

    field_df = prediction_df[
        prediction_df[field + "_prob"] >= threshold
    ]

    print(f"\n===== {field} (threshold={threshold}) =====")

    if len(field_df) == 0:
        print("No prediction")
    else:
        print(" ".join(field_df["text"].tolist()))


===== vendor_name (threshold=0.95) =====
Rothstein-lauber, Incorporated Market Services

===== vendor_address (threshold=0.9) =====
Rothstein-lauber, Incorporated Market Research Services 1351 Washington Blvd.-Stamford, CT06902

===== customer_billing_name (threshold=0.95) =====
Lorillard Tobacco Company

===== customer_billing_address (threshold=0.9) =====
Lorillard Tobacco Company PO Box 10529 Greensboro, NC 27404-0529 Attention: Mr. Scott Benson

===== date_issue (threshold=0.95) =====
September 3, 1999

===== amount_total_gross (threshold=0.9) =====
$15,800.00

===== amount_due (threshold=0.95) =====
$15,800.00


In [172]:
sample_gt = invoice_gt_df[
    invoice_gt_df["document_id"] == sample_doc_id
]

display(sample_gt.T)

,7
document_id,00405d767fb1406180d05ef2
date_issue,"September 3, 1999"
vendor_name,"Rothstein-Tauber, Incorporated"
vendor_address,"Rothstein-Tauber, Incorporated\nMarket Researc..."
customer_billing_name,Lorillard Tobacco Company
customer_billing_address,Lorillard Tobacco Company\nPO Box 10529\nGreen...
amount_total_gross,"$15,800.00"
amount_due,"$15,800.00"
payment_terms,30 days


In [173]:
for field in target_fields:
    value = sample_gt.iloc[0][field]
    print(f"\n{field}:")
    print(repr(value))


vendor_name:
'Rothstein-Tauber, Incorporated'

vendor_address:
'Rothstein-Tauber, Incorporated\nMarket Research Services\n1351 Washington Blvd.•Stamford, CT 06902'

customer_billing_name:
'Lorillard Tobacco Company'

customer_billing_address:
'Lorillard Tobacco Company\nPO Box 10529\nGreensboro, NC 27404-0529\nAttention: Mr. Scott Benson'

date_issue:
'September 3, 1999'

amount_total_gross:
'$15,800.00'

amount_due:
'$15,800.00'


In [174]:
for field in target_fields:
    threshold = test_thresholds[field]

    selected = prediction_df[
        prediction_df[field + "_prob"] >= threshold
    ].copy()

    print(f"\n===== {field} =====")

    if len(selected) == 0:
        print("No prediction")
        continue

    grouped = selected.groupby(
        ["page", "block_id", "line_id"],
        sort=True
    )

    for (page, block, line), group in grouped:
        text = " ".join(group["text"].tolist())
        print(f"Page {page}, Block {block}, Line {line}: {text}")


===== vendor_name =====
Page 0, Block 0, Line 0: Rothstein-lauber, Incorporated
Page 0, Block 0, Line 1: Market Services

===== vendor_address =====
Page 0, Block 0, Line 0: Rothstein-lauber, Incorporated
Page 0, Block 0, Line 1: Market Research Services
Page 0, Block 0, Line 2: 1351 Washington Blvd.-Stamford, CT06902

===== customer_billing_name =====
Page 0, Block 3, Line 0: Lorillard Tobacco Company

===== customer_billing_address =====
Page 0, Block 3, Line 0: Lorillard Tobacco Company
Page 0, Block 3, Line 1: PO Box 10529
Page 0, Block 3, Line 2: Greensboro, NC 27404-0529
Page 0, Block 3, Line 3: Attention: Mr. Scott Benson

===== date_issue =====
Page 0, Block 1, Line 0: September 3, 1999

===== amount_total_gross =====
Page 0, Block 10, Line 2: $15,800.00

===== amount_due =====
Page 0, Block 10, Line 2: $15,800.00


In [175]:
for field in ["vendor_name", "vendor_address", "customer_billing_name"]:
    print(f"\n===== {field} =====")

    field_prob = field + "_prob"

    grouped = prediction_df.groupby(
        ["page", "block_id", "line_id"],
        sort=True
    )

    for (page, block, line), group in grouped:
        if group[field_prob].max() >= 0.5:
            text = " ".join(group["text"].tolist())

            print(
                f"Page {page}, Block {block}, Line {line}"
                f"\nText: {text}"
                f"\nMean probability: {group[field_prob].mean():.4f}"
                f"\nMax probability:  {group[field_prob].max():.4f}"
                f"\nMin probability:  {group[field_prob].min():.4f}"
            )


===== vendor_name =====
Page 0, Block 0, Line 0
Text: Rothstein-lauber, Incorporated
Mean probability: 0.9959
Max probability:  0.9962
Min probability:  0.9956
Page 0, Block 0, Line 1
Text: Market Research Services
Mean probability: 0.9539
Max probability:  0.9574
Min probability:  0.9483

===== vendor_address =====
Page 0, Block 0, Line 0
Text: Rothstein-lauber, Incorporated
Mean probability: 0.9885
Max probability:  0.9909
Min probability:  0.9862
Page 0, Block 0, Line 1
Text: Market Research Services
Mean probability: 0.9939
Max probability:  0.9941
Min probability:  0.9936
Page 0, Block 0, Line 2
Text: 1351 Washington Blvd.-Stamford, CT06902
Mean probability: 0.9967
Max probability:  0.9971
Min probability:  0.9963

===== customer_billing_name =====
Page 0, Block 3, Line 0
Text: Lorillard Tobacco Company
Mean probability: 0.9969
Max probability:  0.9972
Min probability:  0.9968


In [176]:
# Show predicted lines for all fields with their average probability.
# This helps us see whether boundary errors are common.

for field in target_fields:
    prob_col = field + "_prob"
    threshold = test_thresholds[field]

    print(f"\n{'=' * 70}")
    print(field)

    grouped = prediction_df.groupby(
        ["page", "block_id", "line_id"],
        sort=True
    )

    for (page, block, line), group in grouped:
        mean_prob = group[prob_col].mean()
        max_prob = group[prob_col].max()

        if max_prob >= 0.50:
            text = " ".join(group["text"].tolist())

            print(
                f"Line {line}: {text}\n"
                f"  mean={mean_prob:.3f}, max={max_prob:.3f}, "
                f"above_threshold={mean_prob >= threshold}"
            )


vendor_name
Line 0: Rothstein-lauber, Incorporated
  mean=0.996, max=0.996, above_threshold=True
Line 1: Market Research Services
  mean=0.954, max=0.957, above_threshold=True

vendor_address
Line 0: Rothstein-lauber, Incorporated
  mean=0.989, max=0.991, above_threshold=True
Line 1: Market Research Services
  mean=0.994, max=0.994, above_threshold=True
Line 2: 1351 Washington Blvd.-Stamford, CT06902
  mean=0.997, max=0.997, above_threshold=True

customer_billing_name
Line 0: Lorillard Tobacco Company
  mean=0.997, max=0.997, above_threshold=True

customer_billing_address
Line 0: Lorillard Tobacco Company
  mean=0.987, max=0.988, above_threshold=True
Line 1: PO Box 10529
  mean=0.996, max=0.996, above_threshold=True
Line 2: Greensboro, NC 27404-0529
  mean=0.996, max=0.997, above_threshold=True
Line 3: Attention: Mr. Scott Benson
  mean=0.987, max=0.991, above_threshold=True

date_issue
Line 0: September 3, 1999
  mean=0.998, max=0.998, above_threshold=True

amount_total_gross
Line 2:

In [177]:
print(
    "Current LayoutLM encoder:",
    multi_layout_model.layoutlm.config.hidden_size
)

print(
    "Current model device:",
    next(multi_layout_model.parameters()).device
)

print(
    "Target fields:",
    target_fields
)

Current LayoutLM encoder: 768
Current model device: cuda:0
Target fields: ['vendor_name', 'vendor_address', 'customer_billing_name', 'customer_billing_address', 'date_issue', 'amount_total_gross', 'amount_due']


In [178]:
print("invoice_extraction_df columns:")
print(invoice_extraction_df.columns.tolist())

print("\nShape:")
print(invoice_extraction_df.shape)

print("\nFirst 3 rows:")
display(invoice_extraction_df.head(3))

invoice_extraction_df columns:
['document_id', 'field', 'value', 'page', 'bbox']

Shape:
(30823, 5)

First 3 rows:


,document_id,field,value,page,bbox
0,00134dd365a24343b35b78c6,date_issue,03/25/99,0,"[0.7830815892017836, 0.36320179960633614, 0.87..."
1,00134dd365a24343b35b78c6,vendor_name,"BIO RELIANCE Testing & Development, Inc.",0,"[0.0726911179849757, 0.1801480926047427, 0.463..."
2,00134dd365a24343b35b78c6,vendor_address,"BIORELIANCE- Testing & Development, Inc.\n1492...",0,"[0.07385267708679748, 0.17891134682472695, 0.4..."


In [179]:
# Convert the existing multi-label 0/1/-1 annotations into
# independent BIO labels for each field.
#
# BIO encoding:
#   0  = O
#   1  = B
#   2  = I
#  -100 = ignore

bio_fields = target_fields

bio_df = multilabel_df.copy()

# Make sure OCR words are in their original document order
bio_df = bio_df.sort_values(
    ["document_id", "page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

for field in bio_fields:
    source_col = field + "_label"
    bio_col = field + "_bio"

    # Default: O
    bio_df[bio_col] = 0

    # Uncertain annotations → ignore
    bio_df.loc[
        bio_df[source_col] == -1,
        bio_col
    ] = -100

    # Positive token runs
    positive_mask = bio_df[source_col] == 1

    # A token is B if the previous OCR word in the same document
    # is not a positive token for this field.
    previous_positive = (
        bio_df.groupby("document_id")[source_col]
        .shift(1)
        .eq(1)
    )

    # B
    bio_df.loc[
        positive_mask & ~previous_positive.fillna(False),
        bio_col
    ] = 1

    # I
    bio_df.loc[
        positive_mask & previous_positive.fillna(False),
        bio_col
    ] = 2

print("BIO labels created.")

for field in bio_fields:
    col = field + "_bio"
    print(
        field,
        dict(
            bio_df[col].value_counts().sort_index()
        )
    )

KeyError: 'vendor_name_label'

In [180]:
bio_fields = target_fields

bio_df = multilabel_df.copy()

bio_df = bio_df.sort_values(
    ["document_id", "page", "block_id", "line_id", "word_id"]
).reset_index(drop=True)

for field in bio_fields:
    source_col = field
    bio_col = field + "_bio"

    # Default: O
    bio_df[bio_col] = 0

    # Uncertain → ignore
    bio_df.loc[
        bio_df[source_col] == -1,
        bio_col
    ] = -100

    # Positive tokens
    positive_mask = bio_df[source_col] == 1

    previous_positive = (
        bio_df.groupby("document_id")[source_col]
        .shift(1)
        .eq(1)
    )

    # Beginning of entity
    bio_df.loc[
        positive_mask & ~previous_positive.fillna(False),
        bio_col
    ] = 1

    # Inside entity
    bio_df.loc[
        positive_mask & previous_positive.fillna(False),
        bio_col
    ] = 2

print("BIO labels created.")

for field in bio_fields:
    col = field + "_bio"
    print(f"\n{field}:")
    print(bio_df[col].value_counts().sort_index().to_dict())

BIO labels created.

vendor_name:
{-100: 251, 0: 923627, 1: 5210, 2: 11260}

vendor_address:
{-100: 2269, 0: 889829, 1: 5653, 2: 42597}

customer_billing_name:
{-100: 158, 0: 923411, 1: 4134, 2: 12645}

customer_billing_address:
{-100: 964, 0: 893596, 1: 4043, 2: 41745}

date_issue:
{-100: 102, 0: 933568, 1: 4343, 2: 2335}

amount_total_gross:
{-100: 135, 0: 936360, 1: 3681, 2: 172}

amount_due:
{-100: 140, 0: 935985, 1: 4072, 2: 151}


In [181]:
sample_bio = bio_df[
    bio_df["document_id"] == sample_doc_id
].copy()

sample_bio = sample_bio.sort_values(
    ["page", "block_id", "line_id", "word_id"]
)

for field in ["vendor_name", "vendor_address"]:
    print(f"\n===== {field} =====")

    bio_col = field + "_bio"

    for _, row in sample_bio.iterrows():
        label = row[bio_col]

        if label == 1:
            tag = "B"
        elif label == 2:
            tag = "I"
        elif label == 0:
            tag = "O"
        else:
            tag = "IGNORE"

        if tag != "O":
            print(f"{tag:7} {row['text']}")


===== vendor_name =====
B       Rothstein-lauber,
I       Incorporated

===== vendor_address =====
B       Rothstein-lauber,
I       Incorporated
I       Market
I       Research
I       Services
I       1351
I       Washington
I       Blvd.-Stamford,
I       CT06902


In [182]:
# Reuse the existing document-level splits
train_bio_df = bio_df[
    bio_df["document_id"].isin(train_multi_df["document_id"].unique())
].copy()

val_bio_df = bio_df[
    bio_df["document_id"].isin(val_multi_df["document_id"].unique())
].copy()

test_bio_df = bio_df[
    bio_df["document_id"].isin(test_multi_df["document_id"].unique())
].copy()

print("Train documents:", train_bio_df["document_id"].nunique())
print("Validation documents:", val_bio_df["document_id"].nunique())
print("Test documents:", test_bio_df["document_id"].nunique())

print("\nRows:")
print("Train:", len(train_bio_df))
print("Validation:", len(val_bio_df))
print("Test:", len(test_bio_df))

Train documents: 2695
Validation documents: 577
Test documents: 578

Rows:
Train: 659937
Validation: 143347
Test: 137064


In [183]:
from datasets import Dataset

bio_label_columns = [field + "_bio" for field in bio_fields]


def prepare_bio_layoutlm_examples(df):
    examples = []

    doc_ids = df["document_id"].unique()

    for doc_id in doc_ids:
        doc_df = df[df["document_id"] == doc_id].sort_values(
            ["page", "block_id", "line_id", "word_id"]
        )

        words = doc_df["text"].tolist()

        boxes = []

        for _, row in doc_df.iterrows():
            boxes.append([
                int(row["x1"] * 1000),
                int(row["y1"] * 1000),
                int(row["x2"] * 1000),
                int(row["y2"] * 1000)
            ])

        # One BIO label vector per OCR word
        word_labels = []

        for _, row in doc_df.iterrows():
            word_labels.append([
                int(row[col]) for col in bio_label_columns
            ])

        encoded = layout_tokenizer(
            words,
            boxes=boxes,
            is_split_into_words=True,
            return_overflowing_tokens=True,
            truncation=True,
            max_length=256,
            stride=128,
            padding=False
        )

        for chunk_idx in range(len(encoded["input_ids"])):

            word_ids = encoded.word_ids(batch_index=chunk_idx)

            token_labels = []

            for token_idx, word_idx in enumerate(word_ids):

                if word_idx is None:
                    token_labels.append([-100] * len(bio_fields))
                    continue

                original_labels = word_labels[word_idx]
                new_labels = []

                for label in original_labels:

                    if label == -100:
                        new_labels.append(-100)

                    elif label == 0:
                        new_labels.append(0)

                    elif label == 1:
                        # B on first subtoken, I on subsequent subtokens
                        first_occurrence = (
                            token_idx == 0
                            or word_ids[token_idx - 1] != word_idx
                        )
                        new_labels.append(1 if first_occurrence else 2)

                    elif label == 2:
                        new_labels.append(2)

                    else:
                        new_labels.append(-100)

                token_labels.append(new_labels)

            examples.append({
                "input_ids": encoded["input_ids"][chunk_idx],
                "attention_mask": encoded["attention_mask"][chunk_idx],
                "bbox": encoded["bbox"][chunk_idx],
                "labels": token_labels
            })

    return Dataset.from_list(examples)


train_bio_hf = prepare_bio_layoutlm_examples(train_bio_df)
val_bio_hf = prepare_bio_layoutlm_examples(val_bio_df)
test_bio_hf = prepare_bio_layoutlm_examples(test_bio_df)

print("Train chunks:", len(train_bio_hf))
print("Validation chunks:", len(val_bio_hf))
print("Test chunks:", len(test_bio_hf))

KeyError: 'bbox'

In [184]:
from datasets import Dataset


bio_label_columns = [field + "_bio" for field in bio_fields]


def prepare_bio_layoutlm_examples(df):
    examples = []

    doc_ids = df["document_id"].unique()

    for doc_id in doc_ids:
        doc_df = df[
            df["document_id"] == doc_id
        ].sort_values(
            ["page", "block_id", "line_id", "word_id"]
        ).reset_index(drop=True)

        words = doc_df["text"].tolist()

        word_boxes = []

        for _, row in doc_df.iterrows():
            word_boxes.append([
                int(row["x1"] * 1000),
                int(row["y1"] * 1000),
                int(row["x2"] * 1000),
                int(row["y2"] * 1000)
            ])

        word_labels = []

        for _, row in doc_df.iterrows():
            word_labels.append([
                int(row[col]) for col in bio_label_columns
            ])

        encoded = layout_tokenizer(
            words,
            boxes=word_boxes,
            is_split_into_words=True,
            return_overflowing_tokens=True,
            truncation=True,
            max_length=256,
            stride=128,
            padding=False
        )

        for chunk_idx in range(len(encoded["input_ids"])):

            word_ids = encoded.word_ids(
                batch_index=chunk_idx
            )

            chunk_bboxes = []
            token_labels = []

            for token_idx, word_idx in enumerate(word_ids):

                if word_idx is None:
                    chunk_bboxes.append([0, 0, 0, 0])
                    token_labels.append(
                        [-100] * len(bio_fields)
                    )
                    continue

                # Copy original OCR-word bbox
                chunk_bboxes.append(
                    word_boxes[word_idx]
                )

                original_labels = word_labels[word_idx]
                new_labels = []

                # Check whether this is the first subtoken
                is_first_subtoken = (
                    token_idx == 0
                    or word_ids[token_idx - 1] != word_idx
                )

                for label in original_labels:

                    if label == -100:
                        new_labels.append(-100)

                    elif label == 0:
                        new_labels.append(0)

                    elif label == 1:
                        new_labels.append(
                            1 if is_first_subtoken else 2
                        )

                    elif label == 2:
                        new_labels.append(2)

                    else:
                        new_labels.append(-100)

                token_labels.append(new_labels)

            examples.append({
                "input_ids": encoded["input_ids"][chunk_idx],
                "attention_mask": encoded["attention_mask"][chunk_idx],
                "bbox": chunk_bboxes,
                "labels": token_labels
            })

    return Dataset.from_list(examples)

In [185]:
train_bio_hf = prepare_bio_layoutlm_examples(train_bio_df)
val_bio_hf = prepare_bio_layoutlm_examples(val_bio_df)
test_bio_hf = prepare_bio_layoutlm_examples(test_bio_df)

print("Train chunks:", len(train_bio_hf))
print("Validation chunks:", len(val_bio_hf))
print("Test chunks:", len(test_bio_hf))

Train chunks: 10247
Validation chunks: 2169
Test chunks: 2077


In [186]:
import torch
import torch.nn as nn
from transformers import LayoutLMModel
from transformers.modeling_outputs import TokenClassifierOutput


class BIOLayoutLM(nn.Module):
    def __init__(self, encoder, num_fields):
        super().__init__()

        # Reuse the already-trained LayoutLM encoder
        self.layoutlm = encoder

        self.dropout = nn.Dropout(
            self.layoutlm.config.hidden_dropout_prob
        )

        # 3 BIO classes for each of 7 fields
        self.classifier = nn.Linear(
            self.layoutlm.config.hidden_size,
            num_fields * 3
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        bbox,
        labels=None
    ):
        outputs = self.layoutlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox
        )

        sequence_output = self.dropout(
            outputs.last_hidden_state
        )

        logits = self.classifier(
            sequence_output
        )

        # [batch, seq, 21] → [batch, seq, 7, 3]
        logits = logits.view(
            logits.size(0),
            logits.size(1),
            len(bio_fields),
            3
        )

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(
                ignore_index=-100
            )

            total_loss = 0.0
            valid_fields = 0

            for field_idx in range(len(bio_fields)):

                field_logits = logits[:, :, field_idx, :]

                field_labels = labels[:, :, field_idx]

                loss_value = loss_fn(
                    field_logits.reshape(-1, 3),
                    field_labels.reshape(-1)
                )

                total_loss += loss_value
                valid_fields += 1

            loss = total_loss / valid_fields

        # Flatten back to [batch, seq, 21] for compatibility
        flat_logits = logits.view(
            logits.size(0),
            logits.size(1),
            -1
        )

        return TokenClassifierOutput(
            loss=loss,
            logits=flat_logits
        )


bio_layout_model = BIOLayoutLM(
    encoder=multi_layout_model.layoutlm,
    num_fields=len(bio_fields)
)

bio_layout_model = bio_layout_model.to(device)

print("BIO model ready")
print("Output classes:", len(bio_fields) * 3)
print("Device:", next(bio_layout_model.parameters()).device)

BIO model ready
Output classes: 21
Device: cuda:0


In [187]:
test_batch = bio_data_collator([
    train_bio_hf[0],
    train_bio_hf[1]
])

test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

with torch.no_grad():
    bio_output = bio_layout_model(**test_batch)

print("Logits shape:", bio_output.logits.shape)
print("Loss:", bio_output.loss.item())
print("Loss is finite:", torch.isfinite(bio_output.loss).item())

NameError: name 'bio_data_collator' is not defined

In [188]:
from transformers import default_data_collator

bio_data_collator = default_data_collator

print("BIO data collator ready")

BIO data collator ready


In [189]:
test_batch = bio_data_collator([
    train_bio_hf[0],
    train_bio_hf[1]
])

test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

with torch.no_grad():
    bio_output = bio_layout_model(**test_batch)

print("Logits shape:", bio_output.logits.shape)
print("Loss:", bio_output.loss.item())
print("Loss is finite:", torch.isfinite(bio_output.loss).item())

Logits shape: torch.Size([2, 256, 21])
Loss: 1.0987169742584229
Loss is finite: True


In [190]:
bio_class_weights = torch.tensor(
    [1.0, 10.0, 5.0],
    dtype=torch.float32
).to(device)

print("BIO class weights:", bio_class_weights)

BIO class weights: tensor([ 1., 10.,  5.], device='cuda:0')


In [191]:
class BIOLayoutLM(nn.Module):
    def __init__(self, encoder, num_fields, class_weights):
        super().__init__()

        self.layoutlm = encoder

        self.dropout = nn.Dropout(
            self.layoutlm.config.hidden_dropout_prob
        )

        # 3 BIO classes × 7 fields = 21 logits
        self.classifier = nn.Linear(
            self.layoutlm.config.hidden_size,
            num_fields * 3
        )

        self.register_buffer(
            "class_weights",
            class_weights
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        bbox,
        labels=None
    ):
        outputs = self.layoutlm(
            input_ids=input_ids,
            attention_mask=attention_mask,
            bbox=bbox
        )

        sequence_output = self.dropout(
            outputs.last_hidden_state
        )

        logits = self.classifier(sequence_output)

        logits = logits.view(
            logits.size(0),
            logits.size(1),
            len(bio_fields),
            3
        )

        loss = None

        if labels is not None:
            total_loss = 0.0

            for field_idx in range(len(bio_fields)):

                field_logits = logits[:, :, field_idx, :]
                field_labels = labels[:, :, field_idx]

                loss_fn = nn.CrossEntropyLoss(
                    weight=self.class_weights,
                    ignore_index=-100
                )

                field_loss = loss_fn(
                    field_logits.reshape(-1, 3),
                    field_labels.reshape(-1)
                )

                total_loss += field_loss

            loss = total_loss / len(bio_fields)

        flat_logits = logits.view(
            logits.size(0),
            logits.size(1),
            -1
        )

        return TokenClassifierOutput(
            loss=loss,
            logits=flat_logits
        )


bio_layout_model = BIOLayoutLM(
    encoder=multi_layout_model.layoutlm,
    num_fields=len(bio_fields),
    class_weights=bio_class_weights
).to(device)

print("Weighted BIO model ready")
print("Output classes:", len(bio_fields) * 3)
print("Device:", next(bio_layout_model.parameters()).device)

Weighted BIO model ready
Output classes: 21
Device: cuda:0


In [192]:
test_batch = bio_data_collator([
    train_bio_hf[0],
    train_bio_hf[1]
])

test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

with torch.no_grad():
    bio_output = bio_layout_model(**test_batch)

print("Logits shape:", bio_output.logits.shape)
print("Loss:", bio_output.loss.item())
print("Loss is finite:", torch.isfinite(bio_output.loss).item())

Logits shape: torch.Size([2, 256, 21])
Loss: 0.9317963123321533
Loss is finite: True


In [193]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

bio_trainer = Trainer(
    model=bio_layout_model,

    args=TrainingArguments(
        output_dir="../models/layoutlm_bio",

        num_train_epochs=2,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        gradient_accumulation_steps=1,

        learning_rate=2e-5,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        fp16=True,

        logging_steps=100,
        report_to="none"
    ),

    train_dataset=train_bio_hf,
    eval_dataset=val_bio_hf,

    data_collator=bio_data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("BIO Trainer ready")

BIO Trainer ready


In [194]:
bio_train_result = bio_trainer.train()

print("Training complete.")
print("Best checkpoint:", bio_trainer.state.best_model_checkpoint)
print("Best eval loss:", bio_trainer.state.best_metric)

ValueError: expected sequence of length 256 at dim 1 (got 175)

In [195]:
from transformers import DataCollatorForTokenClassification

bio_data_collator = DataCollatorForTokenClassification(
    tokenizer=layout_tokenizer,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("BIO padding collator ready")

BIO padding collator ready


In [196]:
class BIOMultiFieldCollator:
    def __init__(self, tokenizer, num_fields):
        self.tokenizer = tokenizer
        self.num_fields = num_fields

    def __call__(self, features):
        batch_size = len(features)
        max_len = max(len(f["input_ids"]) for f in features)

        input_ids = []
        attention_masks = []
        bboxes = []
        labels = []

        pad_token_id = self.tokenizer.pad_token_id

        for f in features:
            seq_len = len(f["input_ids"])
            pad_len = max_len - seq_len

            input_ids.append(
                f["input_ids"] + [pad_token_id] * pad_len
            )

            attention_masks.append(
                f["attention_mask"] + [0] * pad_len
            )

            bboxes.append(
                f["bbox"] + [[0, 0, 0, 0]] * pad_len
            )

            labels.append(
                f["labels"] + [
                    [-100] * self.num_fields
                ] * pad_len
            )

        return {
            "input_ids": torch.tensor(
                input_ids,
                dtype=torch.long
            ),
            "attention_mask": torch.tensor(
                attention_masks,
                dtype=torch.long
            ),
            "bbox": torch.tensor(
                bboxes,
                dtype=torch.long
            ),
            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )
        }


bio_data_collator = BIOMultiFieldCollator(
    layout_tokenizer,
    len(bio_fields)
)

print("BIO custom collator ready")

BIO custom collator ready


In [197]:
test_batch = bio_data_collator([
    train_bio_hf[0],
    train_bio_hf[1]
])

test_batch = {
    key: value.to(device)
    for key, value in test_batch.items()
}

with torch.no_grad():
    bio_output = bio_layout_model(**test_batch)

print("Input shape:", test_batch["input_ids"].shape)
print("Labels shape:", test_batch["labels"].shape)
print("Logits shape:", bio_output.logits.shape)
print("Loss:", bio_output.loss.item())
print("Loss is finite:", torch.isfinite(bio_output.loss).item())

Input shape: torch.Size([2, 256])
Labels shape: torch.Size([2, 256, 7])
Logits shape: torch.Size([2, 256, 21])
Loss: 0.9305597543716431
Loss is finite: True


In [199]:
bio_trainer = Trainer(
    model=bio_layout_model,

    args=TrainingArguments(
        output_dir="../models/layoutlm_bio",

        num_train_epochs=2,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        gradient_accumulation_steps=1,

        learning_rate=2e-5,
        weight_decay=0.01,

        eval_strategy="epoch",
        save_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        fp16=True,

        logging_steps=100,
        report_to="none"
    ),

    train_dataset=train_bio_hf,
    eval_dataset=val_bio_hf,

    data_collator=bio_data_collator,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=1
        )
    ]
)

print("BIO Trainer recreated with custom collator")

BIO Trainer recreated with custom collator


In [200]:
bio_train_result = bio_trainer.train()

print("Training complete.")
print("Best checkpoint:", bio_trainer.state.best_model_checkpoint)
print("Best eval loss:", bio_trainer.state.best_metric)

Epoch,Training Loss,Validation Loss
1,0.018337,0.027373
2,0.012452,0.027980


Training complete.
Best checkpoint: ../models/layoutlm_bio\checkpoint-1281
Best eval loss: 0.027372946962714195


In [201]:
bio_test_output = bio_trainer.predict(test_bio_hf)

print("Test prediction complete.")
print("Logits shape:", bio_test_output.predictions.shape)

Test prediction complete.
Logits shape: (2077, 256, 21)


In [202]:
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import pandas as pd

# Reshape:
# [chunks, 256, 21] -> [chunks, 256, 7 fields, 3 BIO classes]
bio_logits = bio_test_output.predictions.reshape(
    bio_test_output.predictions.shape[0],
    bio_test_output.predictions.shape[1],
    len(bio_fields),
    3
)

# Predicted BIO class: 0=O, 1=B, 2=I
bio_preds = np.argmax(bio_logits, axis=-1)

bio_metrics = []

for field_idx, field in enumerate(bio_fields):

    y_true = bio_test_output.label_ids[:, :, field_idx]
    y_pred = bio_preds[:, :, field_idx]

    # Ignore padding/special/uncertain positions
    mask = (
        (y_true == 0) |
        (y_true == 1) |
        (y_true == 2)
    )

    y_true_valid = y_true[mask]
    y_pred_valid = y_pred[mask]

    # Treat B and I as "inside this field"
    y_true_entity = (y_true_valid > 0).astype(int)
    y_pred_entity = (y_pred_valid > 0).astype(int)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_entity,
        y_pred_entity,
        average="binary",
        zero_division=0
    )

    # Exact BIO token accuracy
    bio_accuracy = (
        y_true_valid == y_pred_valid
    ).mean()

    bio_metrics.append({
        "field": field,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "bio_token_accuracy": bio_accuracy
    })

bio_metrics_df = pd.DataFrame(bio_metrics)

print(bio_metrics_df)

print(
    "\nMacro F1:",
    bio_metrics_df["f1"].mean()
)

                      field  precision    recall        f1  bio_token_accuracy
0               vendor_name   0.745189  0.904041  0.816965            0.994806
1            vendor_address   0.862126  0.946744  0.902456            0.991677
2     customer_billing_name   0.813751  0.964329  0.882664            0.997357
3  customer_billing_address   0.879034  0.976730  0.925311            0.994845
4                date_issue   0.829429  0.986202  0.901047            0.998283
5        amount_total_gross   0.535714  0.895577  0.670406            0.992663
6                amount_due   0.770938  0.931275  0.843555            0.996850

Macro F1: 0.8489148804128341


In [203]:
def predict_invoice_bio_fields(ocr_data):
    words_df = ocr_to_dataframe(ocr_data).copy()

    words = words_df["text"].tolist()

    word_boxes = []

    for _, row in words_df.iterrows():
        word_boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    tokenizer = LayoutLMTokenizerFast.from_pretrained(
        "microsoft/layoutlm-base-uncased"
    )

    encoded = tokenizer(
        words,
        boxes=word_boxes,
        is_split_into_words=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=256,
        stride=128,
        padding=False
    )

    num_words = len(words)
    num_fields = len(bio_fields)

    # [word, field, BIO-class]
    word_probs = np.zeros(
        (num_words, num_fields, 3),
        dtype=np.float32
    )

    for chunk_idx in range(len(encoded["input_ids"])):

        input_ids = encoded["input_ids"][chunk_idx].unsqueeze(0).to(device)
        attention_mask = encoded["attention_mask"][chunk_idx].unsqueeze(0).to(device)

        word_ids = encoded.word_ids(batch_index=chunk_idx)

        chunk_bboxes = []

        for word_idx in word_ids:
            if word_idx is None:
                chunk_bboxes.append([0, 0, 0, 0])
            else:
                chunk_bboxes.append(word_boxes[word_idx])

        bbox = torch.tensor(
            chunk_bboxes,
            dtype=torch.long
        ).unsqueeze(0).to(device)

        with torch.no_grad():
            output = bio_layout_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox
            )

        logits = output.logits[0].view(
            -1,
            num_fields,
            3
        )

        probs = torch.softmax(
            logits,
            dim=-1
        ).cpu().numpy()

        for token_idx, word_idx in enumerate(word_ids):

            if word_idx is None:
                continue

            # Keep the strongest evidence across overlapping chunks
            word_probs[word_idx] = np.maximum(
                word_probs[word_idx],
                probs[token_idx]
            )

    result_df = words_df.copy()

    for field_idx, field in enumerate(bio_fields):

        result_df[field + "_O_prob"] = word_probs[:, field_idx, 0]
        result_df[field + "_B_prob"] = word_probs[:, field_idx, 1]
        result_df[field + "_I_prob"] = word_probs[:, field_idx, 2]

        result_df[field + "_bio_pred"] = np.argmax(
            word_probs[:, field_idx, :],
            axis=1
        )

    return result_df


bio_prediction_df = predict_invoice_bio_fields(ocr_data)

print("BIO document inference complete.")
print("Words:", len(bio_prediction_df))

AttributeError: 'list' object has no attribute 'unsqueeze'

In [204]:
def predict_invoice_bio_fields(ocr_data):
    words_df = ocr_to_dataframe(ocr_data).copy()

    words = words_df["text"].tolist()

    # ---------------------------------------------------------
    # Convert DocILE normalized boxes (0-1)
    # to LayoutLM boxes (0-1000)
    # ---------------------------------------------------------
    word_boxes = []

    for _, row in words_df.iterrows():
        word_boxes.append([
            int(row["x1"] * 1000),
            int(row["y1"] * 1000),
            int(row["x2"] * 1000),
            int(row["y2"] * 1000)
        ])

    # ---------------------------------------------------------
    # Tokenizer
    # ---------------------------------------------------------
    tokenizer = LayoutLMTokenizerFast.from_pretrained(
        "microsoft/layoutlm-base-uncased"
    )

    encoded = tokenizer(
        words,
        boxes=word_boxes,
        is_split_into_words=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=256,
        stride=128,
        padding=False
    )

    num_words = len(words)
    num_fields = len(bio_fields)

    # ---------------------------------------------------------
    # Store probabilities for every OCR word
    #
    # Shape:
    # [number_of_words, number_of_fields, 3]
    #
    # 3 classes:
    #   0 = O
    #   1 = B
    #   2 = I
    # ---------------------------------------------------------
    word_probs = np.zeros(
        (num_words, num_fields, 3),
        dtype=np.float32
    )

    bio_layout_model.eval()

    # ---------------------------------------------------------
    # Process every 256-token chunk
    # ---------------------------------------------------------
    for chunk_idx in range(len(encoded["input_ids"])):

        # Tokenizer returns Python lists,
        # so explicitly convert them to tensors.
        input_ids = torch.tensor(
            encoded["input_ids"][chunk_idx],
            dtype=torch.long
        ).unsqueeze(0).to(device)

        attention_mask = torch.tensor(
            encoded["attention_mask"][chunk_idx],
            dtype=torch.long
        ).unsqueeze(0).to(device)

        # Which original OCR word does each token belong to?
        word_ids = encoded.word_ids(
            batch_index=chunk_idx
        )

        # -----------------------------------------------------
        # Build token-level bounding boxes
        # -----------------------------------------------------
        chunk_bboxes = []

        for word_idx in word_ids:

            if word_idx is None:
                chunk_bboxes.append(
                    [0, 0, 0, 0]
                )
            else:
                chunk_bboxes.append(
                    word_boxes[word_idx]
                )

        bbox = torch.tensor(
            chunk_bboxes,
            dtype=torch.long
        ).unsqueeze(0).to(device)

        # -----------------------------------------------------
        # Model inference
        # -----------------------------------------------------
        with torch.no_grad():

            output = bio_layout_model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                bbox=bbox
            )

        # [1, seq_len, 21]
        # -> [seq_len, 7 fields, 3 BIO classes]
        logits = output.logits[0].view(
            -1,
            num_fields,
            3
        )

        probs = torch.softmax(
            logits,
            dim=-1
        ).cpu().numpy()

        # -----------------------------------------------------
        # Map token probabilities back to original OCR words
        # -----------------------------------------------------
        for token_idx, word_idx in enumerate(word_ids):

            if word_idx is None:
                continue

            # Because chunks overlap, a word can appear
            # in multiple chunks. Keep the strongest prediction.
            word_probs[word_idx] = np.maximum(
                word_probs[word_idx],
                probs[token_idx]
            )

    # ---------------------------------------------------------
    # Create result dataframe
    # ---------------------------------------------------------
    result_df = words_df.copy()

    for field_idx, field in enumerate(bio_fields):

        result_df[field + "_O_prob"] = (
            word_probs[:, field_idx, 0]
        )

        result_df[field + "_B_prob"] = (
            word_probs[:, field_idx, 1]
        )

        result_df[field + "_I_prob"] = (
            word_probs[:, field_idx, 2]
        )

        result_df[field + "_bio_pred"] = np.argmax(
            word_probs[:, field_idx, :],
            axis=1
        )

    return result_df

In [205]:
bio_prediction_df = predict_invoice_bio_fields(ocr_data)

print("BIO document inference complete.")
print("Words:", len(bio_prediction_df))

BIO document inference complete.
Words: 79


In [206]:
for field in bio_fields:
    pred_col = field + "_bio_pred"

    print(f"\n===== {field} =====")

    current_span = []

    for _, row in bio_prediction_df.iterrows():
        label = int(row[pred_col])

        if label == 1:  # B
            if current_span:
                print(" ".join(current_span))

            current_span = [row["text"]]

        elif label == 2:  # I
            if current_span:
                current_span.append(row["text"])

        else:  # O
            if current_span:
                print(" ".join(current_span))
                current_span = []

    if current_span:
        print(" ".join(current_span))


===== vendor_name =====
Rothstein-lauber, Incorporated Market Research Services

===== vendor_address =====

===== customer_billing_name =====
Lorillard Tobacco Company

===== customer_billing_address =====
Lorillard Tobacco Company PO Box 10529 Greensboro, NC 27404-0529 Attention: Mr. Scott Benson

===== date_issue =====
September 3, 1999

===== amount_total_gross =====
$15,800.00

===== amount_due =====
$15,800.00


In [207]:
for field in ["vendor_name", "vendor_address"]:
    print(f"\n{'=' * 70}")
    print(field)

    for _, row in bio_prediction_df.iterrows():

        pred = int(row[field + "_bio_pred"])

        if pred != 0:
            print(
                f"{row['text']:25s} "
                f"B={row[field + '_B_prob']:.3f} "
                f"I={row[field + '_I_prob']:.3f} "
                f"O={row[field + '_O_prob']:.3f} "
                f"PRED={pred}"
            )


vendor_name
Rothstein-lauber,         B=0.995 I=0.991 O=0.010 PRED=1
Incorporated              B=0.001 I=0.991 O=0.007 PRED=2
Market                    B=0.031 I=0.679 O=0.290 PRED=2
Research                  B=0.004 I=0.882 O=0.113 PRED=2
Services                  B=0.004 I=0.899 O=0.097 PRED=2

vendor_address
Rothstein-lauber,         B=0.990 I=0.995 O=0.005 PRED=2
Incorporated              B=0.001 I=0.994 O=0.005 PRED=2
Market                    B=0.019 I=0.979 O=0.002 PRED=2
Research                  B=0.001 I=0.997 O=0.002 PRED=2
Services                  B=0.001 I=0.996 O=0.002 PRED=2
1351                      B=0.135 I=0.997 O=0.004 PRED=2
Washington                B=0.001 I=0.998 O=0.002 PRED=2
Blvd.-Stamford,           B=0.001 I=0.998 O=0.002 PRED=2
CT06902                   B=0.001 I=0.998 O=0.002 PRED=2


In [208]:
def decode_field_spans(prediction_df, field):
    """
    Decode field text using BIO probabilities + OCR line structure.

    For name fields:
        choose the strongest line containing B evidence.

    For address fields:
        choose the strongest B/I region and keep consecutive lines
        within the same OCR block.
    """

    result = []

    grouped = list(
        prediction_df.groupby(
            ["page", "block_id", "line_id"],
            sort=True
        )
    )

    line_info = []

    for (page, block, line), group in grouped:

        group = group.sort_values("word_id").copy()

        b_col = field + "_B_prob"
        i_col = field + "_I_prob"

        line_info.append({
            "page": page,
            "block": block,
            "line": line,
            "text": " ".join(group["text"].tolist()),
            "mean_b": group[b_col].mean(),
            "max_b": group[b_col].max(),
            "mean_i": group[i_col].mean(),
            "group": group
        })

    if not line_info:
        return ""

    # Strongest line containing the beginning of the entity
    start_idx = max(
        range(len(line_info)),
        key=lambda i: line_info[i]["max_b"]
    )

    start = line_info[start_idx]

    # Name fields: normally take the strongest line only
    if field in [
        "vendor_name",
        "customer_billing_name"
    ]:
        return start["text"]

    # Address fields:
    # keep lines in the same OCR block after the starting line
    selected = [
        start["text"]
    ]

    start_block = start["block"]

    for i in range(start_idx + 1, len(line_info)):

        current = line_info[i]

        if current["block"] != start_block:
            break

        # Keep continuation lines with strong I evidence
        if current["mean_i"] >= 0.50:
            selected.append(current["text"])
        else:
            break

    return " ".join(selected)

In [209]:
for field in [
    "vendor_name",
    "vendor_address",
    "customer_billing_name",
    "customer_billing_address",
    "date_issue",
    "amount_total_gross",
    "amount_due"
]:
    value = decode_field_spans(
        bio_prediction_df,
        field
    )

    print(f"{field}: {value}")

vendor_name: Rothstein-lauber, Incorporated
vendor_address: Rothstein-lauber, Incorporated Market Research Services 1351 Washington Blvd.-Stamford, CT06902
customer_billing_name: Lorillard Tobacco Company
customer_billing_address: Lorillard Tobacco Company PO Box 10529 Greensboro, NC 27404-0529 Attention: Mr. Scott Benson
date_issue: September 3, 1999
amount_total_gross: $15,800.00
amount_due: $15,800.00


In [210]:
from difflib import SequenceMatcher
from tqdm.auto import tqdm
import re
import pandas as pd


def normalize_value(value):
    if pd.isna(value):
        return ""

    value = str(value).lower()

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    # Normalize common punctuation differences
    value = value.replace(",", "")
    value = value.replace(" ", "")

    return value.strip()


def similarity_score(predicted, actual):
    predicted = normalize_value(predicted)
    actual = normalize_value(actual)

    if not predicted or not actual:
        return 0.0

    return SequenceMatcher(
        None,
        predicted,
        actual
    ).ratio()


def dataframe_to_ocr_json(doc_df):
    ocr_data = {"pages": []}

    for page_num, page_df in doc_df.groupby(
        "page", sort=True
    ):
        page_data = {"blocks": []}

        for block_num, block_df in page_df.groupby(
            "block_id", sort=True
        ):
            block_data = {"lines": []}

            for line_num, line_df in block_df.groupby(
                "line_id", sort=True
            ):
                line_data = {"words": []}

                for _, row in line_df.sort_values(
                    "word_id"
                ).iterrows():

                    line_data["words"].append({
                        "value": row["text"],
                        "confidence": row["confidence"],
                        "geometry": [
                            [row["x1"], row["y1"]],
                            [row["x2"], row["y2"]]
                        ]
                    })

                block_data["lines"].append(line_data)

            page_data["blocks"].append(block_data)

        ocr_data["pages"].append(page_data)

    return ocr_data


test_doc_ids = (
    test_bio_df["document_id"]
    .drop_duplicates()
    .head(20)
    .tolist()
)

evaluation_rows = []

for doc_id in tqdm(
    test_doc_ids,
    desc="Evaluating 20 invoices"
):

    doc_df = test_bio_df[
        test_bio_df["document_id"] == doc_id
    ].copy()

    doc_df = doc_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    )

    doc_ocr = dataframe_to_ocr_json(doc_df)

    pred_df = predict_invoice_bio_fields(
        doc_ocr
    )

    row_gt = invoice_gt_df[
        invoice_gt_df["document_id"] == doc_id
    ]

    if len(row_gt) == 0:
        continue

    row_gt = row_gt.iloc[0]

    for field in bio_fields:

        predicted = decode_field_spans(
            pred_df,
            field
        )

        actual = row_gt[field]

        evaluation_rows.append({
            "document_id": doc_id,
            "field": field,
            "predicted": predicted,
            "actual": actual,
            "similarity": similarity_score(
                predicted,
                actual
            )
        })


small_eval_df = pd.DataFrame(
    evaluation_rows
)

print(
    "Evaluated documents:",
    small_eval_df["document_id"].nunique()
)

display(
    small_eval_df.head(20)
)

print(
    "\nMean similarity:",
    small_eval_df["similarity"].mean()
)

print(
    "Similarity >= 0.80:",
    (
        small_eval_df["similarity"] >= 0.80
    ).mean()
)

Evaluating 20 invoices:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluated documents: 20


,document_id,field,predicted,actual,similarity
0,00405d767fb1406180d05ef2,vendor_name,"Rothstein-lauber, Incorporated","Rothstein-Tauber, Incorporated",0.964286
1,00405d767fb1406180d05ef2,vendor_address,"Rothstein-lauber, Incorporated Market Research...","Rothstein-Tauber, Incorporated\nMarket Researc...",0.976471
2,00405d767fb1406180d05ef2,customer_billing_name,Lorillard Tobacco Company,Lorillard Tobacco Company,1.000000
3,00405d767fb1406180d05ef2,customer_billing_address,Lorillard Tobacco Company PO Box 10529 Greensb...,Lorillard Tobacco Company\nPO Box 10529\nGreen...,1.000000
4,00405d767fb1406180d05ef2,date_issue,"September 3, 1999","September 3, 1999",1.000000
5,00405d767fb1406180d05ef2,amount_total_gross,"$15,800.00","$15,800.00",1.000000
6,00405d767fb1406180d05ef2,amount_due,"$15,800.00","$15,800.00",1.000000
7,00e7f330e0344e9abdef3073,vendor_name,WLKS-FM,WLKS-FM,1.000000
8,00e7f330e0344e9abdef3073,vendor_address,"WLKS-FM Lexington, KY","WLKS-FM\nLexington, KY",1.000000
9,00e7f330e0344e9abdef3073,customer_billing_name,Regional Reps,Regional Reps,1.000000



Mean similarity: 0.8211608349058336
Similarity >= 0.80: 0.6928571428571428


In [211]:
import re


def clean_extracted_value(field, text):
    text = str(text).strip()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # ---------------------------------------------------------
    # NAME FIELDS
    # ---------------------------------------------------------
    if field in [
        "vendor_name",
        "customer_billing_name"
    ]:
        # Remove common OCR labels if they appear
        text = re.sub(
            r"^(vendor|supplier|customer|bill\s*to|sold\s*to)\s*[:\-]?\s*",
            "",
            text,
            flags=re.IGNORECASE
        )

        return text.strip()

    # ---------------------------------------------------------
    # DATE
    # ---------------------------------------------------------
    if field == "date_issue":

        date_patterns = [
            r"\b(?:January|February|March|April|May|June|July|August|"
            r"September|October|November|December)\s+\d{1,2},?\s+\d{2,4}\b",

            r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",

            r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b"
        ]

        for pattern in date_patterns:
            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE
            )

            if match:
                return match.group(0).strip()

        return text

    # ---------------------------------------------------------
    # MONEY FIELDS
    # ---------------------------------------------------------
    if field in [
        "amount_total_gross",
        "amount_due"
    ]:

        money_patterns = [
            r"[$₹€£]\s?\d[\d,]*(?:\.\d{2})?",
            r"\d[\d,]*(?:\.\d{2})?"
        ]

        # Prefer currency-prefixed amount
        for pattern in money_patterns:
            matches = re.findall(
                pattern,
                text
            )

            if matches:
                return matches[-1].strip()

        return text

    return text


def decode_field_spans_v2(prediction_df, field):

    grouped = list(
        prediction_df.groupby(
            ["page", "block_id", "line_id"],
            sort=True
        )
    )

    if not grouped:
        return ""

    line_info = []

    for (page, block, line), group in grouped:

        group = group.sort_values("word_id")

        b_col = field + "_B_prob"
        i_col = field + "_I_prob"

        line_info.append({
            "page": page,
            "block": block,
            "line": line,
            "text": " ".join(group["text"].tolist()),
            "max_b": group[b_col].max(),
            "mean_i": group[i_col].mean(),
        })

    # ---------------------------------------------------------
    # NAME FIELDS
    # ---------------------------------------------------------
    if field in [
        "vendor_name",
        "customer_billing_name"
    ]:

        start_idx = max(
            range(len(line_info)),
            key=lambda i: line_info[i]["max_b"]
        )

        # Names normally occupy one OCR line.
        value = line_info[start_idx]["text"]

        return clean_extracted_value(
            field,
            value
        )

    # ---------------------------------------------------------
    # ADDRESS FIELDS
    # ---------------------------------------------------------
    if field in [
        "vendor_address",
        "customer_billing_address"
    ]:

        start_idx = max(
            range(len(line_info)),
            key=lambda i: line_info[i]["max_b"]
        )

        start = line_info[start_idx]

        selected_lines = [
            start["text"]
        ]

        for i in range(
            start_idx + 1,
            len(line_info)
        ):

            current = line_info[i]

            if current["block"] != start["block"]:
                break

            if current["mean_i"] >= 0.50:
                selected_lines.append(
                    current["text"]
                )
            else:
                break

        value = " ".join(selected_lines)

        return clean_extracted_value(
            field,
            value
        )

    # ---------------------------------------------------------
    # DATE AND MONEY
    # ---------------------------------------------------------
    start_idx = max(
        range(len(line_info)),
        key=lambda i: line_info[i]["max_b"]
    )

    value = line_info[start_idx]["text"]

    return clean_extracted_value(
        field,
        value
    )

In [212]:
import re
import pandas as pd
from difflib import SequenceMatcher
from tqdm.auto import tqdm


# =========================================================
# 1. VALUE NORMALIZATION
# =========================================================

def normalize_value(value):
    if pd.isna(value):
        return ""

    value = str(value).lower()

    # Normalize whitespace
    value = re.sub(r"\s+", " ", value)

    # Ignore commas and spaces for similarity comparison
    value = value.replace(",", "")
    value = value.replace(" ", "")

    return value.strip()


def similarity_score(predicted, actual):
    predicted = normalize_value(predicted)
    actual = normalize_value(actual)

    if not predicted or not actual:
        return 0.0

    return SequenceMatcher(
        None,
        predicted,
        actual
    ).ratio()


# =========================================================
# 2. FIELD-SPECIFIC CLEANING
# =========================================================

def clean_extracted_value(field, text):

    text = str(text).strip()

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    # -----------------------------------------------------
    # NAME FIELDS
    # -----------------------------------------------------

    if field in [
        "vendor_name",
        "customer_billing_name"
    ]:

        text = re.sub(
            r"^(vendor|supplier|customer|bill\s*to|sold\s*to)"
            r"\s*[:\-]?\s*",
            "",
            text,
            flags=re.IGNORECASE
        )

        return text.strip()


    # -----------------------------------------------------
    # DATE
    # -----------------------------------------------------

    if field == "date_issue":

        date_patterns = [

            # September 3, 1999
            r"\b(?:January|February|March|April|May|June|July|August|"
            r"September|October|November|December)"
            r"\s+\d{1,2},?\s+\d{2,4}\b",

            # 03/25/99 or 03-25-1999
            r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",

            # 1999/03/25
            r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b"
        ]

        for pattern in date_patterns:

            match = re.search(
                pattern,
                text,
                flags=re.IGNORECASE
            )

            if match:
                return match.group(0).strip()

        return text


    # -----------------------------------------------------
    # MONEY
    # -----------------------------------------------------

    if field in [
        "amount_total_gross",
        "amount_due"
    ]:

        # Prefer values containing currency symbols
        currency_pattern = (
            r"[$₹€£]\s?"
            r"\d[\d,]*(?:\.\d{2})?"
        )

        matches = re.findall(
            currency_pattern,
            text
        )

        if matches:
            return matches[-1].strip()


        # Fallback: numbers without currency symbol
        number_pattern = (
            r"\b\d[\d,]*(?:\.\d{2})?\b"
        )

        matches = re.findall(
            number_pattern,
            text
        )

        if matches:
            return matches[-1].strip()

        return text


    # -----------------------------------------------------
    # OTHER
    # -----------------------------------------------------

    return text


# =========================================================
# 3. IMPROVED DOCUMENT-LEVEL DECODER
# =========================================================

def decode_field_spans_v2(prediction_df, field):

    grouped = list(
        prediction_df.groupby(
            ["page", "block_id", "line_id"],
            sort=True
        )
    )

    if not grouped:
        return ""


    # -----------------------------------------------------
    # Build line information
    # -----------------------------------------------------

    line_info = []

    for (page, block, line), group in grouped:

        group = group.sort_values(
            "word_id"
        )

        b_col = field + "_B_prob"
        i_col = field + "_I_prob"

        line_info.append({

            "page": page,
            "block": block,
            "line": line,

            "text": " ".join(
                group["text"].tolist()
            ),

            "max_b": group[b_col].max(),

            "mean_b": group[b_col].mean(),

            "mean_i": group[i_col].mean(),

            "max_i": group[i_col].max()
        })


    # =====================================================
    # NAME FIELDS
    # =====================================================

    if field in [
        "vendor_name",
        "customer_billing_name"
    ]:

        # Strongest B line
        start_idx = max(
            range(len(line_info)),
            key=lambda i: line_info[i]["max_b"]
        )

        value = line_info[start_idx]["text"]

        return clean_extracted_value(
            field,
            value
        )


    # =====================================================
    # ADDRESS FIELDS
    # =====================================================

    if field in [
        "vendor_address",
        "customer_billing_address"
    ]:

        start_idx = max(
            range(len(line_info)),
            key=lambda i: line_info[i]["max_b"]
        )

        start = line_info[start_idx]

        selected_lines = [
            start["text"]
        ]

        start_block = start["block"]


        # Add following lines from the same block
        for i in range(
            start_idx + 1,
            len(line_info)
        ):

            current = line_info[i]

            # Stop at another block
            if current["block"] != start_block:
                break

            # Keep strong continuation lines
            if current["mean_i"] >= 0.50:

                selected_lines.append(
                    current["text"]
                )

            else:
                break


        value = " ".join(
            selected_lines
        )

        return clean_extracted_value(
            field,
            value
        )


    # =====================================================
    # DATE / MONEY FIELDS
    # =====================================================

    start_idx = max(
        range(len(line_info)),
        key=lambda i: line_info[i]["max_b"]
    )

    value = line_info[start_idx]["text"]

    return clean_extracted_value(
        field,
        value
    )


# =========================================================
# 4. CONVERT DATAFRAME → OCR JSON
# =========================================================

def dataframe_to_ocr_json(doc_df):

    ocr_data = {
        "pages": []
    }

    for page_num, page_df in doc_df.groupby(
        "page",
        sort=True
    ):

        page_data = {
            "blocks": []
        }

        for block_num, block_df in page_df.groupby(
            "block_id",
            sort=True
        ):

            block_data = {
                "lines": []
            }

            for line_num, line_df in block_df.groupby(
                "line_id",
                sort=True
            ):

                line_data = {
                    "words": []
                }

                for _, row in line_df.sort_values(
                    "word_id"
                ).iterrows():

                    line_data["words"].append({

                        "value": row["text"],

                        "confidence": row["confidence"],

                        "geometry": [
                            [row["x1"], row["y1"]],
                            [row["x2"], row["y2"]]
                        ]
                    })

                block_data["lines"].append(
                    line_data
                )

            page_data["blocks"].append(
                block_data
            )

        ocr_data["pages"].append(
            page_data
        )

    return ocr_data


# =========================================================
# 5. EVALUATE 20 TEST INVOICES
# =========================================================

test_doc_ids = (
    test_bio_df["document_id"]
    .drop_duplicates()
    .head(20)
    .tolist()
)

evaluation_rows = []


for doc_id in tqdm(
    test_doc_ids,
    desc="Evaluating 20 invoices"
):

    # -----------------------------------------------------
    # Get OCR words for document
    # -----------------------------------------------------

    doc_df = test_bio_df[
        test_bio_df["document_id"] == doc_id
    ].copy()

    doc_df = doc_df.sort_values(
        [
            "page",
            "block_id",
            "line_id",
            "word_id"
        ]
    )


    # -----------------------------------------------------
    # Convert to OCR JSON
    # -----------------------------------------------------

    doc_ocr = dataframe_to_ocr_json(
        doc_df
    )


    # -----------------------------------------------------
    # Run BIO model
    # -----------------------------------------------------

    pred_df = predict_invoice_bio_fields(
        doc_ocr
    )


    # -----------------------------------------------------
    # Ground truth
    # -----------------------------------------------------

    row_gt = invoice_gt_df[
        invoice_gt_df["document_id"] == doc_id
    ]


    if len(row_gt) == 0:
        continue


    row_gt = row_gt.iloc[0]


    # -----------------------------------------------------
    # Decode each field
    # -----------------------------------------------------

    for field in bio_fields:

        predicted = decode_field_spans_v2(
            pred_df,
            field
        )

        actual = row_gt[field]

        evaluation_rows.append({

            "document_id": doc_id,

            "field": field,

            "predicted": predicted,

            "actual": actual,

            "similarity": similarity_score(
                predicted,
                actual
            )
        })


# =========================================================
# 6. RESULTS
# =========================================================

small_eval_df = pd.DataFrame(
    evaluation_rows
)


print(
    "Evaluated documents:",
    small_eval_df["document_id"].nunique()
)

print(
    "Evaluated fields:",
    len(small_eval_df)
)


# Detailed results
display(
    small_eval_df
)


# Overall metrics
print(
    "\nMean similarity:",
    round(
        small_eval_df["similarity"].mean(),
        4
    )
)

print(
    "Similarity >= 0.80:",
    round(
        (
            small_eval_df["similarity"] >= 0.80
        ).mean(),
        4
    )
)


# =========================================================
# 7. FIELD-WISE RESULTS
# =========================================================

field_summary = (
    small_eval_df
    .groupby("field")["similarity"]
    .agg(
        mean_similarity="mean",
        success_rate_80=lambda x: (
            x >= 0.80
        ).mean()
    )
    .reset_index()
)


print("\nField-wise results:")

display(
    field_summary
)

Evaluating 20 invoices:   0%|          | 0/20 [00:00<?, ?it/s]

Evaluated documents: 20
Evaluated fields: 140


,document_id,field,predicted,actual,similarity
0,00405d767fb1406180d05ef2,vendor_name,"Rothstein-lauber, Incorporated","Rothstein-Tauber, Incorporated",0.964286
1,00405d767fb1406180d05ef2,vendor_address,"Rothstein-lauber, Incorporated Market Research...","Rothstein-Tauber, Incorporated\nMarket Researc...",0.976471
2,00405d767fb1406180d05ef2,customer_billing_name,Lorillard Tobacco Company,Lorillard Tobacco Company,1.000000
3,00405d767fb1406180d05ef2,customer_billing_address,Lorillard Tobacco Company PO Box 10529 Greensb...,Lorillard Tobacco Company\nPO Box 10529\nGreen...,1.000000
4,00405d767fb1406180d05ef2,date_issue,"September 3, 1999","September 3, 1999",1.000000
...,...,...,...,...,...
135,0998d3bad867466d950533ab,customer_billing_name,"EROOK""S MT VIEW MINI 1AR BB",BROOK''S MT VIEW MINI MART BB,0.826087
136,0998d3bad867466d950533ab,customer_billing_address,"EROOK""S MT VIEW MINI 1AR BB 4455 HEW' 1. BROOK...","14455 HIWY 16 BROOKS, CA 95606",0.550725
137,0998d3bad867466d950533ab,date_issue,5/31/00,5/31/00,1.000000
138,0998d3bad867466d950533ab,amount_total_gross,12.85,"2,527.08",0.500000



Mean similarity: 0.8476
Similarity >= 0.80: 0.7571

Field-wise results:


,field,mean_similarity,success_rate_80
0,amount_due,0.867133,0.75
1,amount_total_gross,0.871830,0.75
2,customer_billing_address,0.846833,0.80
3,customer_billing_name,0.816150,0.70
4,date_issue,0.981750,0.95
5,vendor_address,0.733980,0.55
6,vendor_name,0.815845,0.80


In [213]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", 120)

display(
    field_summary.sort_values(
        "mean_similarity",
        ascending=False
    )
)

,field,mean_similarity,success_rate_80
4,date_issue,0.981750,0.95
1,amount_total_gross,0.871830,0.75
0,amount_due,0.867133,0.75
2,customer_billing_address,0.846833,0.80
3,customer_billing_name,0.816150,0.70
6,vendor_name,0.815845,0.80
5,vendor_address,0.733980,0.55


In [214]:
from tqdm.auto import tqdm

all_test_doc_ids = (
    test_bio_df["document_id"]
    .drop_duplicates()
    .tolist()
)

print("Test invoices:", len(all_test_doc_ids))

final_test_rows = []

for doc_id in tqdm(
    all_test_doc_ids,
    desc="Evaluating full test set"
):

    doc_df = test_bio_df[
        test_bio_df["document_id"] == doc_id
    ].copy()

    doc_df = doc_df.sort_values(
        ["page", "block_id", "line_id", "word_id"]
    )

    doc_ocr = dataframe_to_ocr_json(doc_df)

    pred_df = predict_invoice_bio_fields(doc_ocr)

    gt_rows = invoice_gt_df[
        invoice_gt_df["document_id"] == doc_id
    ]

    if len(gt_rows) == 0:
        continue

    gt = gt_rows.iloc[0]

    for field in bio_fields:

        predicted = decode_field_spans_v2(
            pred_df,
            field
        )

        actual = gt[field]

        final_test_rows.append({
            "document_id": doc_id,
            "field": field,
            "predicted": predicted,
            "actual": actual,
            "similarity": similarity_score(
                predicted,
                actual
            )
        })


full_test_eval_df = pd.DataFrame(final_test_rows)

print(
    "Evaluated documents:",
    full_test_eval_df["document_id"].nunique()
)

print(
    "Mean similarity:",
    round(
        full_test_eval_df["similarity"].mean(),
        4
    )
)

print(
    "Similarity >= 0.80:",
    round(
        (
            full_test_eval_df["similarity"] >= 0.80
        ).mean(),
        4
    )
)

Test invoices: 578


Evaluating full test set:   0%|          | 0/578 [00:00<?, ?it/s]

Evaluated documents: 578
Mean similarity: 0.8087
Similarity >= 0.80: 0.7108


In [215]:
final_field_summary = (
    full_test_eval_df
    .groupby("field")["similarity"]
    .agg(
        mean_similarity="mean",
        success_rate_80=lambda x: (x >= 0.80).mean()
    )
    .reset_index()
)

final_field_summary["mean_similarity"] *= 100
final_field_summary["success_rate_80"] *= 100

final_field_summary = final_field_summary.round(2)

display(final_field_summary)

,field,mean_similarity,success_rate_80
0,amount_due,88.25,82.87
1,amount_total_gross,83.51,74.91
2,customer_billing_address,79.97,73.88
3,customer_billing_name,79.64,62.28
4,date_issue,91.95,87.20
5,vendor_address,68.58,52.25
6,vendor_name,74.20,64.19
